# Step31 — GSE326201 Independent Single-Cell Validation

## Step31A — Data Asset Audit

### Primary objective

Validate the frozen findings from Steps 23–30 in the independent GSE326201 dataset.

### Frozen validation targets

1. MS3 Conserved Core50
2. MS3 proliferative / G2M / mitotic-CIN biology
3. MS6 hepatic-differentiation / metabolic biology
4. Reciprocal MS3-like ↔ MS6-like malignant-state axis
5. Step29 frozen regulator architecture
6. Step30 communication/mechanistic routes only if the dataset supports them

### Locked boundaries

- Do NOT redefine MS1–MS8.
- Do NOT reselect MS3 Core50.
- Do NOT reconstruct Original RiskScore in single cells.
- Do NOT optimize signatures using GSE326201.
- Do NOT reselect Step29 regulators.
- Do NOT rerun Step30 communication discovery in the validation cohort.

GSE326201 is used as an independent validation dataset.

In [ ]:
from hcc_release_paths import project_path as _hcc_path
# =============================================================================
# Step31A Cell 2
# Environment and workspace audit
# =============================================================================

from pathlib import Path
import sys
import platform
import importlib.metadata as md

import numpy as np
import pandas as pd
import scipy
import sklearn
import scanpy as sc
import anndata as ad

ROOT = Path(_hcc_path(''))

STEP31_DIR = ROOT / "LIHC_scRNA_Step31_Independent_Validation"
DATA_DIR = STEP31_DIR / "Data"
TABLE_DIR = STEP31_DIR / "Tables"
FIGURE_DIR = STEP31_DIR / "Figures"
AUDIT_DIR = STEP31_DIR / "Audit"
RESOURCE_DIR = STEP31_DIR / "Resources"

for d in [
    STEP31_DIR,
    DATA_DIR,
    TABLE_DIR,
    FIGURE_DIR,
    AUDIT_DIR,
    RESOURCE_DIR
]:
    d.mkdir(parents=True, exist_ok=True)

print("Python executable:")
print(sys.executable)

print("\nPython:", sys.version.split()[0])
print("Platform:", platform.platform())

print("\nPackage versions:")
print("scanpy:", sc.__version__)
print("anndata:", ad.__version__)
print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("scipy:", scipy.__version__)
print("sklearn:", sklearn.__version__)

print("\nROOT:", ROOT)
print("STEP31_DIR:", STEP31_DIR)

print("\nSTEP31 ENVIRONMENT: PASS")

In [ ]:
# =============================================================================
# Step31A Cell 3
# Locate GSE326201-related assets
# =============================================================================

records = []

for p in ROOT.rglob("*"):

    if "GSE326201" not in str(p).upper():
        continue

    if p.is_file():
        try:
            size_mb = p.stat().st_size / 1024**2
        except Exception:
            size_mb = np.nan

        records.append({
            "type": "file",
            "name": p.name,
            "size_mb": size_mb,
            "path": str(p)
        })

    elif p.is_dir():

        records.append({
            "type": "directory",
            "name": p.name,
            "size_mb": np.nan,
            "path": str(p)
        })


gse_assets = pd.DataFrame(records)

if len(gse_assets):
    gse_assets = (
        gse_assets
        .drop_duplicates("path")
        .sort_values(
            ["type", "path"]
        )
        .reset_index(drop=True)
    )

print(
    "GSE326201-related assets:",
    len(gse_assets)
)

display(gse_assets)

In [ ]:
# =============================================================================
# Step31A Cell 4
# Inventory files inside GSE326201-related directories
# =============================================================================

gse_dirs = []

if len(gse_assets):

    gse_dirs = [
        Path(x)
        for x in gse_assets.loc[
            gse_assets["type"] == "directory",
            "path"
        ].tolist()
    ]


file_records = []


for d in gse_dirs:

    for p in d.rglob("*"):

        if not p.is_file():
            continue

        try:
            size_mb = p.stat().st_size / 1024**2
        except Exception:
            size_mb = np.nan

        file_records.append({
            "parent_dir": str(d),
            "name": p.name,
            "suffix": "".join(p.suffixes),
            "size_mb": size_mb,
            "path": str(p)
        })


gse_files = pd.DataFrame(file_records)


# Also include GSE326201-named files that may sit outside a dedicated folder
if len(gse_assets):

    standalone = (
        gse_assets.loc[
            gse_assets["type"] == "file",
            ["name", "size_mb", "path"]
        ]
        .copy()
    )

    if len(standalone):

        standalone["parent_dir"] = (
            standalone["path"]
            .apply(
                lambda x: str(
                    Path(x).parent
                )
            )
        )

        standalone["suffix"] = (
            standalone["name"]
            .apply(
                lambda x: "".join(
                    Path(x).suffixes
                )
            )
        )

        standalone = standalone[
            [
                "parent_dir",
                "name",
                "suffix",
                "size_mb",
                "path"
            ]
        ]

        gse_files = pd.concat(
            [
                gse_files,
                standalone
            ],
            ignore_index=True
        )


if len(gse_files):

    gse_files = (
        gse_files
        .drop_duplicates("path")
        .sort_values(
            "size_mb",
            ascending=False
        )
        .reset_index(drop=True)
    )


print(
    "Files collected:",
    len(gse_files)
)

display(gse_files)

In [ ]:
# =============================================================================
# Step31A Cell 5
# File-format classification
# =============================================================================

def classify_file(name):

    n = name.lower()

    if n.endswith(".h5ad"):
        return "AnnData_h5ad"

    if n.endswith(".loom"):
        return "Loom"

    if n.endswith(".h5"):
        return "HDF5_h5"

    if n.endswith(".mtx") or n.endswith(".mtx.gz"):
        return "MatrixMarket"

    if n.endswith(".rds"):
        return "RDS"

    if n.endswith(".rdata"):
        return "RData"

    if n.endswith(".csv") or n.endswith(".csv.gz"):
        return "CSV"

    if (
        n.endswith(".tsv")
        or n.endswith(".tsv.gz")
        or n.endswith(".txt")
        or n.endswith(".txt.gz")
    ):
        return "TSV_TXT"

    if n.endswith(".gz"):
        return "GZIP_other"

    return "Other"


if len(gse_files):

    gse_files["file_class"] = (
        gse_files["name"]
        .apply(classify_file)
    )

    print("File classes:")

    display(
        gse_files["file_class"]
        .value_counts()
        .rename_axis("file_class")
        .reset_index(name="n_files")
    )

    print("\nLargest files:")

    display(
        gse_files[
            [
                "name",
                "file_class",
                "size_mb",
                "path"
            ]
        ]
        .head(30)
    )

In [ ]:
# =============================================================================
# Step31A Cell 6
# Identify likely expression and annotation assets
# =============================================================================

if len(gse_files):

    names = (
        gse_files["name"]
        .str.lower()
    )


    expression_mask = names.str.contains(
        "count|matrix|expr|expression|rna|raw|filtered",
        regex=True
    )


    annotation_mask = names.str.contains(
        "meta|annot|celltype|cell_type|cluster|sample|patient|barcode|clinical",
        regex=True
    )


    print("Potential expression files:")

    display(
        gse_files.loc[
            expression_mask,
            [
                "name",
                "file_class",
                "size_mb",
                "path"
            ]
        ]
    )


    print("\nPotential annotation / metadata files:")

    display(
        gse_files.loc[
            annotation_mask,
            [
                "name",
                "file_class",
                "size_mb",
                "path"
            ]
        ]
    )

In [ ]:
# =============================================================================
# Step31A Cell 7
# Lightweight header inspection
# =============================================================================

preview_rows = []


if len(gse_files):

    text_files = gse_files.loc[
        gse_files["file_class"].isin(
            [
                "CSV",
                "TSV_TXT"
            ]
        )
    ].copy()


    for _, row in text_files.iterrows():

        path = Path(row["path"])

        try:

            if (
                path.name.lower().endswith(".csv")
                or
                path.name.lower().endswith(".csv.gz")
            ):
                tmp = pd.read_csv(
                    path,
                    nrows=5
                )

            else:
                tmp = pd.read_csv(
                    path,
                    sep="\t",
                    nrows=5
                )


            preview_rows.append({
                "name":
                    path.name,

                "path":
                    str(path),

                "n_preview_columns":
                    tmp.shape[1],

                "columns":
                    " | ".join(
                        map(
                            str,
                            tmp.columns[:30]
                        )
                    )
            })


        except Exception as e:

            preview_rows.append({
                "name":
                    path.name,

                "path":
                    str(path),

                "n_preview_columns":
                    np.nan,

                "columns":
                    f"READ_FAILED: {repr(e)}"
            })


text_preview = pd.DataFrame(
    preview_rows
)

display(text_preview)

In [ ]:
# =============================================================================
# Step31A Cell 8
# Lightweight h5ad metadata audit
# =============================================================================

h5ad_rows = []


if len(gse_files):

    h5ad_files = gse_files.loc[
        gse_files[
            "file_class"
        ] == "AnnData_h5ad"
    ]


    for _, row in h5ad_files.iterrows():

        path = Path(
            row["path"]
        )

        try:

            a = sc.read_h5ad(
                path,
                backed="r"
            )

            h5ad_rows.append({
                "name":
                    path.name,

                "path":
                    str(path),

                "n_obs":
                    a.n_obs,

                "n_vars":
                    a.n_vars,

                "obs_columns":
                    " | ".join(
                        map(
                            str,
                            a.obs.columns[:50]
                        )
                    ),

                "var_columns":
                    " | ".join(
                        map(
                            str,
                            a.var.columns[:30]
                        )
                    ),

                "obsm_keys":
                    " | ".join(
                        map(
                            str,
                            a.obsm.keys()
                        )
                    ),

                "layers":
                    " | ".join(
                        map(
                            str,
                            a.layers.keys()
                        )
                    )
            })

            a.file.close()

        except Exception as e:

            h5ad_rows.append({
                "name":
                    path.name,

                "path":
                    str(path),

                "n_obs":
                    np.nan,

                "n_vars":
                    np.nan,

                "obs_columns":
                    f"READ_FAILED: {repr(e)}",

                "var_columns":
                    "",

                "obsm_keys":
                    "",

                "layers":
                    ""
            })


h5ad_audit = pd.DataFrame(
    h5ad_rows
)

display(h5ad_audit)

In [ ]:
# =============================================================================
# Step31A Cell 9
# Save Step31A audit
# =============================================================================

gse_assets.to_csv(
    AUDIT_DIR /
    "Step31A_GSE326201_AssetLocations.csv",
    index=False
)


gse_files.to_csv(
    AUDIT_DIR /
    "Step31A_GSE326201_FileInventory.csv",
    index=False
)


text_preview.to_csv(
    AUDIT_DIR /
    "Step31A_TextFile_HeaderAudit.csv",
    index=False
)


h5ad_audit.to_csv(
    AUDIT_DIR /
    "Step31A_H5AD_MetadataAudit.csv",
    index=False
)


print(
    "STEP31A AUDIT FILES SAVED"
)

In [ ]:
# =============================================================================
# Step31A Cell 10
# REPORT — data asset audit
# =============================================================================

print("=" * 110)

print(
    "STEP31A — GSE326201 INDEPENDENT VALIDATION DATA AUDIT"
)

print("=" * 110)


print("\n[1] ENVIRONMENT")

print(
    "Python:",
    sys.executable
)

print(
    "Scanpy:",
    sc.__version__
)

print(
    "AnnData:",
    ad.__version__
)


print("\n[2] GSE326201 ASSETS")

print(
    "Asset records:",
    len(gse_assets)
)

display(
    gse_assets
)


print("\n[3] FILE INVENTORY")

print(
    "Total files:",
    len(gse_files)
)

if len(gse_files):

    display(
        gse_files[
            [
                "name",
                "file_class",
                "size_mb",
                "path"
            ]
        ]
        .sort_values(
            "size_mb",
            ascending=False
        )
    )


print("\n[4] FILE CLASSES")

if len(gse_files):

    display(
        gse_files[
            "file_class"
        ]
        .value_counts()
        .rename_axis(
            "file_class"
        )
        .reset_index(
            name="n_files"
        )
    )


print("\n[5] TEXT / METADATA HEADER AUDIT")

display(
    text_preview
)


print("\n[6] H5AD AUDIT")

display(
    h5ad_audit
)


print("\n[7] FROZEN VALIDATION TARGETS")

print(
"""
Primary:
  MS3 Conserved Core50

Biological validation:
  MS3 proliferative / G2M / spindle / CIN
  MS6 hepatic differentiation / bile-acid / metabolic identity
  reciprocal MS3-like <-> MS6-like axis

Secondary:
  Step29 frozen regulator architecture

Conditional:
  Step30 communication/mechanistic routes
  only if suitable TME + malignant cells are available.
"""
)


print("\n[8] LOCKED RESTRICTIONS")

print(
"""
No MS1-MS8 redefinition.
No Core50 reselection.
No Original RiskScore reconstruction.
No signature refitting on GSE326201.
No Step29 regulator reselection.
No Step30 rediscovery.
"""
)


print("\nSTEP31A: COMPLETE")

print(
    "NEXT: choose the exact loading strategy based on the real GSE326201 assets."
)

print("=" * 110)

In [ ]:
# =============================================================================
# Step31B1 Cell 11
# Locate frozen GSE326201 RAW archive
# =============================================================================

from pathlib import Path

raw_tar_candidates = list(
    ROOT.rglob("GSE326201_RAW.tar")
)

print(
    "RAW archive candidates:",
    len(raw_tar_candidates)
)

for p in raw_tar_candidates:
    print(
        p,
        f"{p.stat().st_size / 1024**2:.2f} MB"
    )


assert len(raw_tar_candidates) == 1, (
    "Expected exactly one GSE326201_RAW.tar"
)

RAW_TAR = raw_tar_candidates[0]

print(
    "\nFrozen RAW archive:"
)

print(
    RAW_TAR
)

In [ ]:
# =============================================================================
# Step31B1 Cell 12
# Audit archive members WITHOUT extraction
# =============================================================================

import tarfile
import pandas as pd

member_rows = []


with tarfile.open(
    RAW_TAR,
    mode="r"
) as tar:

    members = tar.getmembers()

    for m in members:

        member_rows.append({
            "name":
                m.name,

            "is_file":
                m.isfile(),

            "size_mb":
                (
                    m.size / 1024**2
                    if m.isfile()
                    else 0
                )
        })


tar_inventory = pd.DataFrame(
    member_rows
)


print(
    "Archive members:",
    len(tar_inventory)
)

print(
    "Files:",
    int(
        tar_inventory[
            "is_file"
        ].sum()
    )
)

print(
    "Archive payload size:",
    round(
        tar_inventory[
            "size_mb"
        ].sum(),
        2
    ),
    "MB"
)


display(
    tar_inventory.head(
        100
    )
)

In [ ]:
# =============================================================================
# Step31B1 Cell 13
# Classify files inside RAW archive
# =============================================================================

def classify_archive_member(name):

    n = name.lower()

    if (
        n.endswith(
            "matrix.mtx.gz"
        )
        or
        n.endswith(
            "matrix.mtx"
        )
    ):
        return "10X_matrix"

    if "barcode" in n:
        return "barcodes"

    if (
        "feature" in n
        or
        "gene" in n
    ):
        return "features_genes"

    if (
        n.endswith(".csv.gz")
        or
        n.endswith(".csv")
    ):
        return "CSV"

    if (
        n.endswith(".tsv.gz")
        or
        n.endswith(".tsv")
    ):
        return "TSV"

    if (
        n.endswith(".txt.gz")
        or
        n.endswith(".txt")
    ):
        return "TXT"

    if n.endswith(".h5"):
        return "H5"

    if n.endswith(".h5ad"):
        return "H5AD"

    if n.endswith(".mtx.gz"):
        return "MatrixMarket"

    if n.endswith(".gz"):
        return "GZIP_other"

    return "Other"


tar_files = (
    tar_inventory.loc[
        tar_inventory[
            "is_file"
        ]
    ]
    .copy()
)


tar_files[
    "file_class"
] = (
    tar_files[
        "name"
    ]
    .apply(
        classify_archive_member
    )
)


print(
    "File classes:"
)

display(
    tar_files[
        "file_class"
    ]
    .value_counts()
    .rename_axis(
        "file_class"
    )
    .reset_index(
        name="n_files"
    )
)


print(
    "\nComplete archive file list:"
)

display(
    tar_files[
        [
            "name",
            "file_class",
            "size_mb"
        ]
    ]
)

In [ ]:
# =============================================================================
# Step31B1 Cell 14
# Infer sample / GSM structure from filenames
# =============================================================================

import re

sample_rows = []


for name in tar_files[
    "name"
]:

    gsm_match = re.search(
        r"(GSM\d+)",
        name,
        flags=re.I
    )

    sample_rows.append({
        "name":
            name,

        "GSM":
            (
                gsm_match.group(
                    1
                ).upper()
                if gsm_match
                else None
            )
    })


sample_audit = pd.DataFrame(
    sample_rows
)


print(
    "Unique GSM IDs detected:",
    sample_audit[
        "GSM"
    ].nunique(
        dropna=True
    )
)


print(
    "\nFiles per GSM:"
)

display(
    sample_audit[
        "GSM"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "GSM"
    )
    .reset_index(
        name="n_files"
    )
)

In [ ]:
# =============================================================================
# Step31B1 Cell 15
# Save RAW archive manifest
# =============================================================================

tar_files.to_csv(
    AUDIT_DIR /
    "Step31B1_GSE326201_RAW_ArchiveManifest.csv",
    index=False
)


sample_audit.to_csv(
    AUDIT_DIR /
    "Step31B1_GSE326201_RAW_SampleAudit.csv",
    index=False
)


print(
    "STEP31B1 archive manifests saved."
)

In [ ]:
# =============================================================================
# Step31B1 Cell 16
# REPORT — GEO RAW archive audit
# =============================================================================

print(
    "=" * 110
)

print(
    "STEP31B1 — GSE326201 RAW ARCHIVE AUDIT"
)

print(
    "=" * 110
)


print(
    "\n[1] RAW ARCHIVE"
)

print(
    RAW_TAR
)

print(
    "Archive size:",
    round(
        RAW_TAR.stat().st_size
        /
        1024**2,
        2
    ),
    "MB"
)


print(
    "\n[2] ARCHIVE CONTENT"
)

print(
    "Total files:",
    len(
        tar_files
    )
)

print(
    "Payload size:",
    round(
        tar_files[
            "size_mb"
        ].sum(),
        2
    ),
    "MB"
)


print(
    "\n[3] FILE CLASSES"
)

display(
    tar_files[
        "file_class"
    ]
    .value_counts()
    .rename_axis(
        "file_class"
    )
    .reset_index(
        name="n_files"
    )
)


print(
    "\n[4] UNIQUE GSM SAMPLES"
)

print(
    sample_audit[
        "GSM"
    ].nunique(
        dropna=True
    )
)


display(
    sample_audit[
        "GSM"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "GSM"
    )
    .reset_index(
        name="n_files"
    )
)


print(
    "\n[5] COMPLETE FILE LIST"
)

display(
    tar_files[
        [
            "name",
            "file_class",
            "size_mb"
        ]
    ]
)


print(
    "\n[6] CURRENT DECISION"
)

print(
"""
GSE326201 is currently available locally as a GEO RAW archive.

No processed AnnData object or standalone metadata table was identified
during Step31A.

The archive has NOT yet been extracted or transformed.

NEXT:
Determine whether the archive contains:
  - 10X matrix/barcodes/features,
  - per-sample count matrices,
  - or another raw format.

Only after confirming the internal structure will the data be extracted
and loaded reproducibly.
"""
)


print(
    "\nSTEP31B1 RAW ARCHIVE AUDIT: COMPLETE"
)

print(
    "=" * 110
)

## Step31B2 — 10x HDF5 extraction and sample-level audit

### Objective

Extract and audit the 18 frozen 10x Genomics filtered feature-barcode matrices
from the original GEO RAW archive.

This step will:

1. safely extract the HDF5 matrices;
2. reconstruct sample/patient/tissue metadata;
3. read each matrix independently;
4. audit cell counts, gene universes and feature types;
5. preserve raw count matrices without filtering.

No QC thresholding, normalization, integration, clustering or signature scoring
is performed in Step31B2.

In [ ]:
# =============================================================================
# Step31B2 Cell 18
# Safely extract 18 filtered_feature_bc_matrix.h5 files
# =============================================================================

import tarfile
import shutil
from pathlib import Path

EXTRACT_DIR = (
    STEP31_DIR /
    "Data" /
    "Raw" /
    "GSE326201_10xH5"
)

EXTRACT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


with tarfile.open(
    RAW_TAR,
    mode="r"
) as tar:

    h5_members = [
        m
        for m in tar.getmembers()
        if (
            m.isfile()
            and
            m.name.lower().endswith(".h5")
        )
    ]

    print(
        "H5 files in archive:",
        len(h5_members)
    )

    assert len(h5_members) == 18


    for m in h5_members:

        out_path = (
            EXTRACT_DIR /
            Path(m.name).name
        )

        # Do not overwrite an already extracted file
        if out_path.exists():

            print(
                "SKIP existing:",
                out_path.name
            )

            continue


        src = tar.extractfile(m)

        assert src is not None

        with open(
            out_path,
            "wb"
        ) as dst:

            shutil.copyfileobj(
                src,
                dst
            )

        print(
            "Extracted:",
            out_path.name
        )


print(
    "\nExtraction directory:"
)

print(
    EXTRACT_DIR
)

print(
    "\nExtracted H5 count:",
    len(
        list(
            EXTRACT_DIR.glob("*.h5")
        )
    )
)

In [ ]:
# =============================================================================
# Step31B2 Cell 19
# Build frozen sample metadata
# =============================================================================

import re
import pandas as pd


ETIOLOGY_MAP = {
    "001": "HCV",
    "002": "NBNC",
    "004": "HBV",
    "005": "NBNC",
    "006": "NBNC",
    "008": "NBNC",
    "009": "HBV",
    "010": "NBNC",
    "011": "HCV",
    "012": "HBV"
}


sample_rows = []


for path in sorted(
    EXTRACT_DIR.glob(
        "*.h5"
    )
):

    name = path.name


    match = re.match(
        r"(GSM\d+)_(\d+)([NT])_filtered_feature_bc_matrix\.h5$",
        name
    )

    assert match is not None, (
        f"Unexpected filename: {name}"
    )


    gsm = match.group(1)

    patient_short = match.group(2)

    tissue_code = match.group(3)


    patient_id = (
        f"{int(patient_short):03d}"
    )


    sample_id = (
        f"{int(patient_short)}{tissue_code}"
    )


    tissue = (
        "Tumor"
        if tissue_code == "T"
        else "NonTumor"
    )


    sample_rows.append({

        "GSM":
            gsm,

        "sample_id":
            sample_id,

        "patient_id":
            patient_id,

        "tissue_code":
            tissue_code,

        "tissue":
            tissue,

        "etiology":
            ETIOLOGY_MAP[
                patient_id
            ],

        "file_name":
            name,

        "file_path":
            str(path),

        "file_size_mb":
            path.stat().st_size /
            1024**2
    })


sample_metadata = (
    pd.DataFrame(
        sample_rows
    )
    .sort_values(
        [
            "patient_id",
            "tissue_code"
        ]
    )
    .reset_index(
        drop=True
    )
)


display(
    sample_metadata
)


print(
    "\nSamples:",
    len(sample_metadata)
)

print(
    "Patients:",
    sample_metadata[
        "patient_id"
    ].nunique()
)

print(
    "\nTissue:"
)

print(
    sample_metadata[
        "tissue"
    ].value_counts()
)

print(
    "\nEtiology:"
)

print(
    sample_metadata[
        "etiology"
    ].value_counts()
)

In [ ]:
# =============================================================================
# Step31B2 Cell 20
# Patient pairing audit
# =============================================================================

pairing = (
    sample_metadata
    .pivot_table(
        index=[
            "patient_id",
            "etiology"
        ],
        columns="tissue",
        values="sample_id",
        aggfunc="first"
    )
    .reset_index()
)


pairing[
    "paired"
] = (
    pairing[
        "Tumor"
    ].notna()
    &
    pairing[
        "NonTumor"
    ].notna()
)


display(
    pairing
)


print(
    "Paired patients:",
    int(
        pairing[
            "paired"
        ].sum()
    )
)

print(
    "Tumor-only patients:",
    int(
        (
            ~pairing[
                "paired"
            ]
        ).sum()
    )
)

In [ ]:
# =============================================================================
# Step31B2 Cell 21
# Per-sample 10x H5 audit
# =============================================================================

import scanpy as sc
import numpy as np
import scipy.sparse as sp


matrix_rows = []


for _, meta in (
    sample_metadata.iterrows()
):

    path = Path(
        meta[
            "file_path"
        ]
    )


    print(
        "\nReading:",
        meta[
            "sample_id"
        ],
        path.name
    )


    adata_tmp = (
        sc.read_10x_h5(
            path,
            gex_only=False
        )
    )


    # ---------------------------------------------------------
    # Feature types
    # ---------------------------------------------------------

    if (
        "feature_types"
        in adata_tmp.var.columns
    ):

        feature_counts = (
            adata_tmp.var[
                "feature_types"
            ]
            .value_counts()
            .to_dict()
        )

    else:

        feature_counts = {}


    # ---------------------------------------------------------
    # Raw matrix audit
    # ---------------------------------------------------------

    X = adata_tmp.X


    if sp.issparse(X):

        nnz = X.nnz

        data_values = X.data

    else:

        nnz = np.count_nonzero(
            X
        )

        data_values = X[
            X != 0
        ]


    integer_like = (
        bool(
            np.allclose(
                data_values,
                np.round(
                    data_values
                )
            )
        )
        if len(
            data_values
        )
        else True
    )


    matrix_rows.append({

        "GSM":
            meta["GSM"],

        "sample_id":
            meta["sample_id"],

        "patient_id":
            meta["patient_id"],

        "tissue":
            meta["tissue"],

        "etiology":
            meta["etiology"],

        "n_cells":
            adata_tmp.n_obs,

        "n_features":
            adata_tmp.n_vars,

        "nnz":
            int(nnz),

        "integer_like_counts":
            integer_like,

        "duplicated_var_names":
            int(
                adata_tmp.var_names
                .duplicated()
                .sum()
            ),

        "feature_types":
            str(
                feature_counts
            )
    })


    print(
        "shape:",
        adata_tmp.shape
    )

    print(
        "feature types:",
        feature_counts
    )

    print(
        "integer-like:",
        integer_like
    )


    del adata_tmp


matrix_audit = (
    pd.DataFrame(
        matrix_rows
    )
)


display(
    matrix_audit
)

In [ ]:
# =============================================================================
# Step31B2 Cell 22
# Gene-universe consistency audit
# =============================================================================

gene_sets = {}
gene_orders = {}


for _, meta in (
    sample_metadata.iterrows()
):

    sample_id = meta[
        "sample_id"
    ]

    path = Path(
        meta[
            "file_path"
        ]
    )


    a = sc.read_10x_h5(
        path,
        gex_only=True
    )


    genes = list(
        map(
            str,
            a.var_names
        )
    )


    gene_sets[
        sample_id
    ] = set(
        genes
    )

    gene_orders[
        sample_id
    ] = genes


    del a


reference_sample = (
    sample_metadata.iloc[
        0
    ][
        "sample_id"
    ]
)


reference_set = (
    gene_sets[
        reference_sample
    ]
)


reference_order = (
    gene_orders[
        reference_sample
    ]
)


gene_universe_rows = []


for sample_id in (
    sample_metadata[
        "sample_id"
    ]
):

    current_set = (
        gene_sets[
            sample_id
        ]
    )

    current_order = (
        gene_orders[
            sample_id
        ]
    )


    gene_universe_rows.append({

        "sample_id":
            sample_id,

        "n_genes":
            len(
                current_set
            ),

        "same_gene_set_as_reference":
            current_set
            ==
            reference_set,

        "same_gene_order_as_reference":
            current_order
            ==
            reference_order,

        "missing_vs_reference":
            len(
                reference_set
                -
                current_set
            ),

        "extra_vs_reference":
            len(
                current_set
                -
                reference_set
            )
    })


gene_universe_audit = (
    pd.DataFrame(
        gene_universe_rows
    )
)


display(
    gene_universe_audit
)


print(
    "\nAll same gene set:",
    bool(
        gene_universe_audit[
            "same_gene_set_as_reference"
        ].all()
    )
)


print(
    "All same gene order:",
    bool(
        gene_universe_audit[
            "same_gene_order_as_reference"
        ].all()
    )
)

In [ ]:
# =============================================================================
# Step31B2 Cell 23
# Save frozen sample and matrix audit
# =============================================================================

sample_metadata.to_csv(
    AUDIT_DIR /
    "Step31B2_GSE326201_SampleMetadata.csv",
    index=False
)


pairing.to_csv(
    AUDIT_DIR /
    "Step31B2_GSE326201_PatientPairing.csv",
    index=False
)


matrix_audit.to_csv(
    AUDIT_DIR /
    "Step31B2_GSE326201_MatrixAudit.csv",
    index=False
)


gene_universe_audit.to_csv(
    AUDIT_DIR /
    "Step31B2_GSE326201_GeneUniverseAudit.csv",
    index=False
)


print(
    "STEP31B2 AUDIT OUTPUTS SAVED"
)

In [ ]:
# =============================================================================
# Step31B2 Cell 24
# REPORT — extracted 10x H5 audit
# =============================================================================

print(
    "=" * 110
)

print(
    "STEP31B2 — GSE326201 10X HDF5 SAMPLE AUDIT"
)

print(
    "=" * 110
)


print(
    "\n[1] DATA FORMAT"
)

print(
    "18 sample-specific 10x Genomics filtered_feature_bc_matrix.h5 files"
)


print(
    "\n[2] SAMPLE DESIGN"
)

print(
    "Samples:",
    len(
        sample_metadata
    )
)

print(
    "Patients:",
    sample_metadata[
        "patient_id"
    ].nunique()
)

print(
    "Tumor:",
    int(
        (
            sample_metadata[
                "tissue"
            ]
            ==
            "Tumor"
        ).sum()
    )
)

print(
    "NonTumor:",
    int(
        (
            sample_metadata[
                "tissue"
            ]
            ==
            "NonTumor"
        ).sum()
    )
)

print(
    "Paired patients:",
    int(
        pairing[
            "paired"
        ].sum()
    )
)


print(
    "\nSample metadata:"
)

display(
    sample_metadata[
        [
            "GSM",
            "sample_id",
            "patient_id",
            "tissue",
            "etiology",
            "file_size_mb"
        ]
    ]
)


print(
    "\n[3] MATRIX AUDIT"
)

display(
    matrix_audit
)


print(
    "\nTotal filtered cells:",
    int(
        matrix_audit[
            "n_cells"
        ].sum()
    )
)


print(
    "\nCells by tissue:"
)

display(
    matrix_audit
    .groupby(
        "tissue"
    )[
        "n_cells"
    ]
    .agg(
        [
            "count",
            "sum",
            "median",
            "min",
            "max"
        ]
    )
)


print(
    "\n[4] RAW COUNT CHECK"
)

print(
    "All matrices integer-like:",
    bool(
        matrix_audit[
            "integer_like_counts"
        ].all()
    )
)


print(
    "\n[5] FEATURE TYPES"
)

display(
    matrix_audit[
        [
            "sample_id",
            "n_features",
            "feature_types",
            "duplicated_var_names"
        ]
    ]
)


print(
    "\n[6] GENE UNIVERSE"
)

display(
    gene_universe_audit
)


print(
    "\nAll same gene set:",
    bool(
        gene_universe_audit[
            "same_gene_set_as_reference"
        ].all()
    )
)

print(
    "All same gene order:",
    bool(
        gene_universe_audit[
            "same_gene_order_as_reference"
        ].all()
    )
)


print(
    "\n[7] CURRENT INTERPRETATION"
)

print(
"""
GSE326201 consists of independent raw 10x filtered count matrices.

No author-derived clustering or malignant-state labels have been used.

The next step will construct a reproducible combined raw-count object,
followed by sample-aware QC.

QC thresholds will NOT be chosen before inspecting the observed
per-cell distributions.
"""
)


print(
    "\nSTEP31B2: COMPLETE"
)

print(
    "NEXT: Step31B3 — combined raw object + QC audit."
)

print(
    "=" * 110
)

## Step31B3 — Gene-reference harmonization and QC audit

### Objectives

1. Audit Ensembl gene-ID compatibility across all 18 libraries.
2. Separate gene identity from duplicated gene symbols.
3. Quantify the common gene universe across all samples.
4. Check frozen-signature gene availability before any validation analysis.
5. Compute per-cell QC metrics independently within each sample.
6. Inspect QC distributions before defining filtering thresholds.

### Important

No cells are filtered in this step.

No normalization, integration, clustering, malignant-cell identification,
signature refitting, or biological validation is performed here.

In [ ]:
# =============================================================================
# Step31B3 Cell 26
# Gene ID / gene-symbol reference audit
# =============================================================================

gene_id_sets = {}
gene_id_orders = {}
gene_symbol_maps = {}

reference_rows = []


for _, meta in sample_metadata.iterrows():

    sample_id = meta["sample_id"]
    path = Path(meta["file_path"])

    a = sc.read_10x_h5(
        path,
        gex_only=True
    )

    print(
        sample_id,
        a.shape,
        list(a.var.columns)
    )

    assert "gene_ids" in a.var.columns, (
        f"{sample_id}: gene_ids missing"
    )

    gene_ids = (
        a.var["gene_ids"]
        .astype(str)
        .tolist()
    )

    gene_symbols = (
        a.var_names
        .astype(str)
        .tolist()
    )

    gene_id_sets[sample_id] = set(
        gene_ids
    )

    gene_id_orders[sample_id] = gene_ids

    gene_symbol_maps[sample_id] = pd.DataFrame({
        "gene_id": gene_ids,
        "gene_symbol": gene_symbols
    })

    reference_rows.append({
        "sample_id":
            sample_id,

        "n_features":
            a.n_vars,

        "n_unique_gene_ids":
            len(
                set(gene_ids)
            ),

        "duplicated_gene_ids":
            pd.Index(
                gene_ids
            ).duplicated().sum(),

        "n_unique_gene_symbols":
            len(
                set(gene_symbols)
            ),

        "duplicated_gene_symbols":
            pd.Index(
                gene_symbols
            ).duplicated().sum()
    })

    del a


gene_reference_audit = pd.DataFrame(
    reference_rows
)

display(
    gene_reference_audit
)

In [ ]:
# =============================================================================
# Step31B3 Cell 27
# Common and union Ensembl gene universes
# =============================================================================

all_gene_sets = list(
    gene_id_sets.values()
)


COMMON_GENE_IDS = set.intersection(
    *all_gene_sets
)

UNION_GENE_IDS = set.union(
    *all_gene_sets
)


print(
    "Common Ensembl genes across all 18 samples:",
    len(COMMON_GENE_IDS)
)

print(
    "Union Ensembl genes across all 18 samples:",
    len(UNION_GENE_IDS)
)

print(
    "Union - common:",
    len(
        UNION_GENE_IDS
        -
        COMMON_GENE_IDS
    )
)


gene_id_overlap_rows = []


reference_sample = (
    sample_metadata.iloc[0][
        "sample_id"
    ]
)

reference_ids = (
    gene_id_sets[
        reference_sample
    ]
)


for sample_id in (
    sample_metadata[
        "sample_id"
    ]
):

    current = (
        gene_id_sets[
            sample_id
        ]
    )

    gene_id_overlap_rows.append({

        "sample_id":
            sample_id,

        "n_gene_ids":
            len(current),

        "n_common_all18":
            len(
                current
                &
                COMMON_GENE_IDS
            ),

        "missing_vs_reference":
            len(
                reference_ids
                -
                current
            ),

        "extra_vs_reference":
            len(
                current
                -
                reference_ids
            )
    })


gene_id_overlap_audit = (
    pd.DataFrame(
        gene_id_overlap_rows
    )
)

display(
    gene_id_overlap_audit
)

In [ ]:
# =============================================================================
# Step31B3 Cell 28
# Ensembl-to-symbol mapping consistency
# =============================================================================

mapping_all = []


for sample_id, df in (
    gene_symbol_maps.items()
):

    tmp = df.copy()

    tmp[
        "sample_id"
    ] = sample_id

    mapping_all.append(
        tmp
    )


mapping_all = pd.concat(
    mapping_all,
    ignore_index=True
)


mapping_consistency = (
    mapping_all
    .groupby(
        "gene_id"
    )[
        "gene_symbol"
    ]
    .nunique()
)


mapping_conflicts = (
    mapping_consistency[
        mapping_consistency > 1
    ]
)


print(
    "Gene IDs with >1 gene-symbol mapping:",
    len(
        mapping_conflicts
    )
)


if len(
    mapping_conflicts
):

    conflict_table = (
        mapping_all.loc[
            mapping_all[
                "gene_id"
            ].isin(
                mapping_conflicts.index
            )
        ]
        .sort_values(
            [
                "gene_id",
                "sample_id"
            ]
        )
    )

    display(
        conflict_table.head(
            100
        )
    )

else:

    print(
        "Ensembl-to-symbol mapping is consistent."
    )

In [ ]:
# =============================================================================
# Step31B3 Cell 29
# Locate frozen MS3 Core50 resources
# =============================================================================

core50_candidates = []


for p in ROOT.rglob("*"):

    if not p.is_file():
        continue

    name_upper = p.name.upper()

    if (
        "CORE50"
        in name_upper
        and
        p.suffix.lower()
        in {
            ".csv",
            ".tsv",
            ".txt",
            ".json"
        }
    ):

        try:
            size_kb = (
                p.stat().st_size
                /
                1024
            )
        except Exception:
            size_kb = np.nan

        core50_candidates.append({
            "name":
                p.name,

            "size_kb":
                size_kb,

            "path":
                str(p)
        })


core50_candidates = (
    pd.DataFrame(
        core50_candidates
    )
)


print(
    "Core50 candidate files:",
    len(
        core50_candidates
    )
)

display(
    core50_candidates
)

In [ ]:
# =============================================================================
# Step31B3 Cell 30
# Per-sample QC metric helper
# =============================================================================

import scipy.sparse as sp


def compute_qc_for_sample(
    path,
    meta
):

    a = sc.read_10x_h5(
        path,
        gex_only=True
    )

    # Preserve symbols explicitly
    a.var[
        "gene_symbol"
    ] = (
        a.var_names.astype(str)
    )

    # Stable Ensembl gene ID axis
    a.var_names = (
        a.var[
            "gene_ids"
        ].astype(str)
    )

    assert (
        ~a.var_names.duplicated()
    ).all()


    symbols = (
        a.var[
            "gene_symbol"
        ].astype(str)
    )


    a.var[
        "mt"
    ] = (
        symbols.str.upper()
        .str.startswith(
            "MT-"
        )
    )


    a.var[
        "ribo"
    ] = (
        symbols.str.upper()
        .str.startswith(
            (
                "RPS",
                "RPL"
            )
        )
    )


    sc.pp.calculate_qc_metrics(
        a,
        qc_vars=[
            "mt",
            "ribo"
        ],
        inplace=True,
        log1p=False,
        percent_top=None
    )


    obs = a.obs[
        [
            "total_counts",
            "n_genes_by_counts",
            "pct_counts_mt",
            "pct_counts_ribo"
        ]
    ].copy()


    obs[
        "sample_id"
    ] = meta[
        "sample_id"
    ]

    obs[
        "patient_id"
    ] = meta[
        "patient_id"
    ]

    obs[
        "tissue"
    ] = meta[
        "tissue"
    ]

    obs[
        "etiology"
    ] = meta[
        "etiology"
    ]


    return obs

In [ ]:
# =============================================================================
# Step31B3 Cell 31
# Compute per-cell QC metrics sample by sample
# =============================================================================

qc_tables = []


for _, meta in (
    sample_metadata.iterrows()
):

    print(
        "QC:",
        meta[
            "sample_id"
        ]
    )

    qc_tmp = (
        compute_qc_for_sample(
            Path(
                meta[
                    "file_path"
                ]
            ),
            meta
        )
    )

    qc_tables.append(
        qc_tmp
    )


qc_cells = pd.concat(
    qc_tables,
    axis=0,
    ignore_index=False
)


print(
    "\nQC cells:",
    len(
        qc_cells
    )
)

assert (
    len(
        qc_cells
    )
    ==
    matrix_audit[
        "n_cells"
    ].sum()
)

display(
    qc_cells.head()
)

In [ ]:
# =============================================================================
# Step31B3 Cell 32
# Sample-aware QC distribution summary
# =============================================================================

QC_METRICS = [
    "total_counts",
    "n_genes_by_counts",
    "pct_counts_mt",
    "pct_counts_ribo"
]


quantiles = [
    0.01,
    0.05,
    0.10,
    0.25,
    0.50,
    0.75,
    0.90,
    0.95,
    0.99
]


qc_summary_rows = []


for sample_id, sub in (
    qc_cells.groupby(
        "sample_id"
    )
):

    row = {
        "sample_id":
            sample_id,

        "n_cells":
            len(sub),

        "patient_id":
            sub[
                "patient_id"
            ].iloc[0],

        "tissue":
            sub[
                "tissue"
            ].iloc[0]
    }


    for metric in QC_METRICS:

        q = (
            sub[
                metric
            ]
            .quantile(
                quantiles
            )
        )

        for qq, value in q.items():

            row[
                f"{metric}_q{int(qq*100):02d}"
            ] = float(
                value
            )


    qc_summary_rows.append(
        row
    )


qc_sample_summary = (
    pd.DataFrame(
        qc_summary_rows
    )
    .sort_values(
        [
            "patient_id",
            "tissue"
        ]
    )
    .reset_index(
        drop=True
    )
)


display(
    qc_sample_summary
)

In [ ]:
# =============================================================================
# Step31B3 Cell 33
# Compact QC summary
# =============================================================================

qc_compact = (
    qc_cells
    .groupby(
        [
            "sample_id",
            "patient_id",
            "tissue"
        ]
    )
    .agg(
        n_cells=(
            "total_counts",
            "size"
        ),

        median_counts=(
            "total_counts",
            "median"
        ),

        median_genes=(
            "n_genes_by_counts",
            "median"
        ),

        median_mt=(
            "pct_counts_mt",
            "median"
        ),

        p95_mt=(
            "pct_counts_mt",
            lambda x:
                x.quantile(
                    0.95
                )
        ),

        p99_mt=(
            "pct_counts_mt",
            lambda x:
                x.quantile(
                    0.99
                )
        )
    )
    .reset_index()
)


display(
    qc_compact
)

In [ ]:
# =============================================================================
# Step31B3 Cell 34
# QC-threshold sensitivity audit
# NO filtering is applied here
# =============================================================================

QC_SCENARIOS = {

    "Loose": {
        "min_genes":
            200,

        "max_mt":
            30
    },

    "Moderate": {
        "min_genes":
            300,

        "max_mt":
            25
    },

    "Strict": {
        "min_genes":
            500,

        "max_mt":
            20
    }
}


qc_sensitivity_rows = []


for scenario, params in (
    QC_SCENARIOS.items()
):

    pass_mask = (
        (
            qc_cells[
                "n_genes_by_counts"
            ]
            >=
            params[
                "min_genes"
            ]
        )
        &
        (
            qc_cells[
                "pct_counts_mt"
            ]
            <=
            params[
                "max_mt"
            ]
        )
    )


    tmp = (
        qc_cells.assign(
            qc_pass=
                pass_mask
        )
        .groupby(
            [
                "sample_id",
                "patient_id",
                "tissue"
            ]
        )[
            "qc_pass"
        ]
        .agg(
            [
                "count",
                "sum"
            ]
        )
        .reset_index()
    )


    tmp[
        "scenario"
    ] = scenario

    tmp[
        "min_genes"
    ] = params[
        "min_genes"
    ]

    tmp[
        "max_mt"
    ] = params[
        "max_mt"
    ]

    tmp[
        "retained_fraction"
    ] = (
        tmp[
            "sum"
        ]
        /
        tmp[
            "count"
        ]
    )


    qc_sensitivity_rows.append(
        tmp
    )


qc_threshold_sensitivity = (
    pd.concat(
        qc_sensitivity_rows,
        ignore_index=True
    )
)


display(
    qc_threshold_sensitivity
)

In [ ]:
# =============================================================================
# Step31B3 Cell 35
# Save reference and QC audits
# =============================================================================

gene_reference_audit.to_csv(
    AUDIT_DIR /
    "Step31B3_GeneReferenceAudit.csv",
    index=False
)


gene_id_overlap_audit.to_csv(
    AUDIT_DIR /
    "Step31B3_GeneIDOverlapAudit.csv",
    index=False
)


qc_cells.to_csv(
    AUDIT_DIR /
    "Step31B3_PerCellQC.csv",
    index=True
)


qc_sample_summary.to_csv(
    AUDIT_DIR /
    "Step31B3_QCQuantiles_BySample.csv",
    index=False
)


qc_compact.to_csv(
    AUDIT_DIR /
    "Step31B3_QCCompact_BySample.csv",
    index=False
)


qc_threshold_sensitivity.to_csv(
    AUDIT_DIR /
    "Step31B3_QCThresholdSensitivity.csv",
    index=False
)


core50_candidates.to_csv(
    AUDIT_DIR /
    "Step31B3_Core50CandidateFiles.csv",
    index=False
)


print(
    "STEP31B3 AUDIT OUTPUTS SAVED"
)

In [ ]:
# =============================================================================
# Step31B3 Cell 36
# REPORT — reference harmonization + QC
# =============================================================================

print(
    "=" * 110
)

print(
    "STEP31B3 — REFERENCE HARMONIZATION AND QC AUDIT"
)

print(
    "=" * 110
)


print(
    "\n[1] DATASET"
)

print(
    "Samples:",
    len(
        sample_metadata
    )
)

print(
    "Patients:",
    sample_metadata[
        "patient_id"
    ].nunique()
)

print(
    "Total filtered cells:",
    len(
        qc_cells
    )
)


print(
    "\n[2] GENE REFERENCE AUDIT"
)

display(
    gene_reference_audit
)


print(
    "\nCommon Ensembl genes across all 18 libraries:",
    len(
        COMMON_GENE_IDS
    )
)

print(
    "Union Ensembl genes:",
    len(
        UNION_GENE_IDS
    )
)

print(
    "Union minus common:",
    len(
        UNION_GENE_IDS
        -
        COMMON_GENE_IDS
    )
)


print(
    "\nGene ID overlap:"
)

display(
    gene_id_overlap_audit
)


print(
    "\nEnsembl IDs with conflicting symbols:",
    len(
        mapping_conflicts
    )
)


print(
    "\n[3] FROZEN CORE50 RESOURCE SEARCH"
)

print(
    "Candidate files:",
    len(
        core50_candidates
    )
)

display(
    core50_candidates
)


print(
    "\n[4] SAMPLE-AWARE QC"
)

display(
    qc_compact
)


print(
    "\n[5] QC THRESHOLD SENSITIVITY"
)

display(
    qc_threshold_sensitivity[
        [
            "sample_id",
            "tissue",
            "scenario",
            "count",
            "sum",
            "retained_fraction"
        ]
    ]
)


print(
    "\nOverall retention by scenario:"
)

display(
    qc_threshold_sensitivity
    .groupby(
        "scenario"
    )
    .agg(
        cells_before=(
            "count",
            "sum"
        ),
        cells_after=(
            "sum",
            "sum"
        )
    )
    .assign(
        retained_fraction=
            lambda x:
                x[
                    "cells_after"
                ]
                /
                x[
                    "cells_before"
                ]
    )
)


print(
    "\n[6] CURRENT DECISION BOUNDARY"
)

print(
"""
No QC threshold has been frozen yet.

No combined expression object has been generated yet.

Because the 18 libraries contain at least two gene-reference universes,
cross-sample harmonization must be based on stable gene identity rather
than naïve gene-symbol concatenation.

The frozen Core50 resource must be identified and its coverage checked
before independent biological validation begins.
"""
)


print(
    "\nSTEP31B3: COMPLETE"
)

print(
    "NEXT: freeze gene universe + QC criteria, then build the combined raw object."
)

print(
    "=" * 110
)

In [ ]:
# =============================================================================
# Step31B4 Cell 37
# Load canonical frozen MS3 Core50
# =============================================================================

from pathlib import Path

core50_row = core50_candidates.loc[
    core50_candidates["name"]
    ==
    "Step25_FROZEN_MS3_Core50_Genes.txt"
]

assert len(core50_row) == 1, (
    "Canonical Core50 file not uniquely identified."
)

CORE50_PATH = Path(
    core50_row.iloc[0]["path"]
)

print("Frozen Core50 file:")
print(CORE50_PATH)


raw_lines = [
    x.strip()
    for x in CORE50_PATH.read_text(
        encoding="utf-8"
    ).splitlines()
    if x.strip()
]


# Remove header only if explicitly present
if raw_lines[0].lower() in {
    "gene",
    "genes",
    "gene_symbol",
    "symbol"
}:
    raw_lines = raw_lines[1:]


CORE50 = list(
    dict.fromkeys(
        raw_lines
    )
)


print("\nCore50 genes:")
print(CORE50)

print("\nN genes:", len(CORE50))
print("N unique:", len(set(CORE50)))

assert len(CORE50) == 50
assert len(set(CORE50)) == 50

print("\nFROZEN CORE50 LOAD: PASS")

In [ ]:
# =============================================================================
# Step31B4 Cell 38
# Frozen Core50 coverage across all 18 libraries
# =============================================================================

core50_coverage_rows = []


for sample_id, mapping in gene_symbol_maps.items():

    symbols = set(
        mapping["gene_symbol"]
        .astype(str)
    )

    present = [
        g
        for g in CORE50
        if g in symbols
    ]

    missing = [
        g
        for g in CORE50
        if g not in symbols
    ]

    core50_coverage_rows.append({
        "sample_id":
            sample_id,

        "n_present":
            len(present),

        "coverage_fraction":
            len(present) / 50,

        "missing_genes":
            " | ".join(missing)
    })


core50_coverage = pd.DataFrame(
    core50_coverage_rows
)


display(
    core50_coverage
)


print(
    "\nMinimum Core50 coverage:",
    core50_coverage[
        "n_present"
    ].min(),
    "/ 50"
)

In [ ]:
# =============================================================================
# Step31B4 Cell 39
# Map Core50 symbols to common Ensembl universe
# =============================================================================

common_mapping = (
    mapping_all.loc[
        mapping_all["gene_id"]
        .isin(COMMON_GENE_IDS)
    ]
    [
        [
            "gene_id",
            "gene_symbol"
        ]
    ]
    .drop_duplicates()
)


core50_common_rows = []


for gene in CORE50:

    tmp = common_mapping.loc[
        common_mapping[
            "gene_symbol"
        ] == gene
    ]

    ids = sorted(
        tmp[
            "gene_id"
        ].unique()
        .tolist()
    )

    core50_common_rows.append({
        "gene_symbol":
            gene,

        "n_common_ensembl_ids":
            len(ids),

        "common_ensembl_ids":
            " | ".join(ids)
    })


core50_common_map = pd.DataFrame(
    core50_common_rows
)


display(
    core50_common_map
)


print(
    "\nCore50 symbols represented in common universe:",
    int(
        (
            core50_common_map[
                "n_common_ensembl_ids"
            ] > 0
        ).sum()
    ),
    "/ 50"
)

In [ ]:
# =============================================================================
# Step31B4 Cell 40
# Frozen Step29 regulator coverage
# =============================================================================

FROZEN_REGULATORS = {

    "MS3": [
        "FOXM1",
        "MYBL2",
        "E2F8",
        "TFDP1",
        "E2F1",
        "BRCA1"
    ],

    "MS5": [
        "KLF4",
        "FOS",
        "FOSB",
        "ELF3",
        "SOX4"
    ],

    "MS6": [
        "HNF4A",
        "NR1H4",
        "NR1I3",
        "CEBPA",
        "FOXA3"
    ]
}


regulator_rows = []


for state, genes in (
    FROZEN_REGULATORS.items()
):

    for gene in genes:

        tmp = common_mapping.loc[
            common_mapping[
                "gene_symbol"
            ] == gene
        ]

        regulator_rows.append({
            "state":
                state,

            "gene":
                gene,

            "present_common_universe":
                len(tmp) > 0,

            "ensembl_ids":
                " | ".join(
                    sorted(
                        tmp[
                            "gene_id"
                        ].unique()
                        .tolist()
                    )
                )
        })


regulator_coverage = pd.DataFrame(
    regulator_rows
)


display(
    regulator_coverage
)


print(
    "\nFrozen regulators present:",
    int(
        regulator_coverage[
            "present_common_universe"
        ].sum()
    ),
    "/",
    len(
        regulator_coverage
    )
)

In [ ]:
# =============================================================================
# Step31B4 Cell 41
# Freeze Step31 QC criteria
# =============================================================================

STEP31_QC = {
    "min_genes_by_counts":
        300,

    "max_pct_counts_mt":
        25.0,

    "upper_gene_cutoff":
        None,

    "upper_count_cutoff":
        None,

    "doublet_handling":
        "separate downstream procedure"
}


print(
    "Frozen Step31 QC:"
)

print(
    STEP31_QC
)


qc_pass = (
    (
        qc_cells[
            "n_genes_by_counts"
        ]
        >=
        STEP31_QC[
            "min_genes_by_counts"
        ]
    )
    &
    (
        qc_cells[
            "pct_counts_mt"
        ]
        <=
        STEP31_QC[
            "max_pct_counts_mt"
        ]
    )
)


print(
    "\nCells before QC:",
    len(qc_cells)
)

print(
    "Cells after QC:",
    int(
        qc_pass.sum()
    )
)

print(
    "Retention:",
    float(
        qc_pass.mean()
    )
)

In [ ]:
# =============================================================================
# Step31B4 Cell 42
# Build harmonized combined raw-count object
# =============================================================================

import scanpy as sc
import anndata as ad
import scipy.sparse as sp


combined_parts = []


# deterministic common-gene order:
# use first sample order restricted to common IDs
REFERENCE_SAMPLE = (
    sample_metadata.iloc[0][
        "sample_id"
    ]
)


COMMON_GENE_ORDER = [
    g
    for g in gene_id_orders[
        REFERENCE_SAMPLE
    ]
    if g in COMMON_GENE_IDS
]


assert (
    len(
        COMMON_GENE_ORDER
    )
    ==
    len(
        COMMON_GENE_IDS
    )
)


for _, meta in sample_metadata.iterrows():

    sample_id = meta["sample_id"]

    print(
        "Building:",
        sample_id
    )


    a = sc.read_10x_h5(
        meta["file_path"],
        gex_only=True
    )


    # Preserve gene symbols
    a.var[
        "gene_symbol"
    ] = (
        a.var_names
        .astype(str)
    )


    # Set Ensembl IDs as stable var index
    a.var_names = (
        a.var[
            "gene_ids"
        ]
        .astype(str)
    )


    assert (
        ~a.var_names
        .duplicated()
    ).all()


    # Common universe, deterministic order
    a = a[
        :,
        COMMON_GENE_ORDER
    ].copy()


    # Recompute QC on this sample's original GEX data
    symbols = (
        a.var[
            "gene_symbol"
        ].astype(str)
    )

    a.var[
        "mt"
    ] = (
        symbols.str.upper()
        .str.startswith(
            "MT-"
        )
    )


    sc.pp.calculate_qc_metrics(
        a,
        qc_vars=[
            "mt"
        ],
        inplace=True,
        log1p=False,
        percent_top=None
    )


    keep = (
        (
            a.obs[
                "n_genes_by_counts"
            ]
            >= 300
        )
        &
        (
            a.obs[
                "pct_counts_mt"
            ]
            <= 25
        )
    )


    a = a[
        keep
    ].copy()


    # Unique cell IDs across libraries
    a.obs_names = [
        f"{sample_id}_{x}"
        for x in a.obs_names
    ]


    a.obs[
        "sample_id"
    ] = sample_id

    a.obs[
        "patient_id"
    ] = meta[
        "patient_id"
    ]

    a.obs[
        "tissue"
    ] = meta[
        "tissue"
    ]

    a.obs[
        "etiology"
    ] = meta[
        "etiology"
    ]

    a.obs[
        "GSM"
    ] = meta[
        "GSM"
    ]


    combined_parts.append(
        a
    )


adata_raw = ad.concat(
    combined_parts,
    axis=0,
    join="inner",
    merge="same",
    index_unique=None
)


print(
    "\nCombined raw object:"
)

print(
    adata_raw
)


print(
    "\nShape:",
    adata_raw.shape
)


assert (
    adata_raw.n_vars
    ==
    32732
)


assert (
    adata_raw.obs_names.is_unique
)


print(
    "\nCOMBINED RAW OBJECT: PASS"
)

In [ ]:
# =============================================================================
# Step31B4 Cell 43
# Save frozen harmonized raw-count object
# =============================================================================

FINAL_RAW_PATH = (
    DATA_DIR /
    "GSE326201_STEP31B4_HARMONIZED_QC_RAW.h5ad"
)


adata_raw.write_h5ad(
    FINAL_RAW_PATH,
    compression="gzip"
)


print(
    "Saved:"
)

print(
    FINAL_RAW_PATH
)


print(
    "\nShape:",
    adata_raw.shape
)

In [ ]:
# =============================================================================
# Step31B4 Cell 44
# REPORT — harmonization + QC lock
# =============================================================================

print(
    "=" * 110
)

print(
    "STEP31B4 — HARMONIZATION AND QC LOCK REPORT"
)

print(
    "=" * 110
)


print(
    "\n[1] GENE UNIVERSE"
)

print(
    "Common Ensembl genes:",
    len(
        COMMON_GENE_ORDER
    )
)

print(
    "Original union:",
    len(
        UNION_GENE_IDS
    )
)


print(
    "\n[2] CORE50"
)

print(
    "Frozen Core50:",
    len(
        CORE50
    )
)

print(
    "Minimum per-sample coverage:",
    int(
        core50_coverage[
            "n_present"
        ].min()
    ),
    "/ 50"
)

print(
    "Core50 represented in common universe:",
    int(
        (
            core50_common_map[
                "n_common_ensembl_ids"
            ] > 0
        ).sum()
    ),
    "/ 50"
)


missing_core50 = (
    core50_common_map.loc[
        core50_common_map[
            "n_common_ensembl_ids"
        ] == 0,
        "gene_symbol"
    ].tolist()
)


print(
    "Missing Core50 genes:",
    missing_core50
)


print(
    "\n[3] STEP29 REGULATORS"
)

print(
    "Present:",
    int(
        regulator_coverage[
            "present_common_universe"
        ].sum()
    ),
    "/",
    len(
        regulator_coverage
    )
)


display(
    regulator_coverage
)


print(
    "\n[4] QC LOCK"
)

print(
    STEP31_QC
)


print(
    "\n[5] FINAL RAW OBJECT"
)

print(
    "Shape:",
    adata_raw.shape
)

print(
    "Patients:",
    adata_raw.obs[
        "patient_id"
    ].nunique()
)

print(
    "Samples:",
    adata_raw.obs[
        "sample_id"
    ].nunique()
)


print(
    "\nCells by tissue:"
)

display(
    adata_raw.obs[
        "tissue"
    ].value_counts()
)


print(
    "\nCells by sample:"
)

display(
    adata_raw.obs[
        "sample_id"
    ].value_counts()
)


print(
    "\n[6] FILE"
)

print(
    FINAL_RAW_PATH
)

print(
    "Exists:",
    FINAL_RAW_PATH.exists()
)


print(
    "\nSTEP31B4: FROZEN"
)

print(
    "NEXT: Step31C — doublet handling + preprocessing + independent cell-type/malignant-cell identification."
)

print(
    "=" * 110
)

In [ ]:
# =============================================================================
# Step31B4.1 Cell 45
# Audit SGOL1/SGOL2 symbol aliases across the two 10x references
# =============================================================================

TARGET_SYMBOLS = [
    "SGOL1",
    "SGO1",
    "SGOL2",
    "SGO2"
]


alias_rows = []

for sample_id, mapping in gene_symbol_maps.items():

    hit = mapping.loc[
        mapping["gene_symbol"].isin(
            TARGET_SYMBOLS
        )
    ].copy()

    if len(hit) == 0:

        alias_rows.append({
            "sample_id": sample_id,
            "gene_symbol": None,
            "gene_id": None
        })

    else:

        for _, r in hit.iterrows():

            alias_rows.append({
                "sample_id": sample_id,
                "gene_symbol": r["gene_symbol"],
                "gene_id": r["gene_id"]
            })


alias_audit = pd.DataFrame(
    alias_rows
)


display(
    alias_audit
)


print("\nSymbol counts across libraries:")

display(
    alias_audit[
        "gene_symbol"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "gene_symbol"
    )
    .reset_index(
        name="n_samples"
    )
)


print("\nCommon-universe hits:")

display(
    common_mapping.loc[
        common_mapping[
            "gene_symbol"
        ].isin(
            TARGET_SYMBOLS
        )
    ]
    .drop_duplicates()
)


print(
    "\nAlias audit complete."
)

In [ ]:
# =============================================================================
# Step31C1 Cell 46
# Doublet-detection capability audit
# =============================================================================

import scanpy as sc

print(
    "scanpy.pp.scrublet available:",
    hasattr(
        sc.pp,
        "scrublet"
    )
)

print(
    "scanpy.pp.scrublet_simulate_doublets available:",
    hasattr(
        sc.pp,
        "scrublet_simulate_doublets"
    )
)

In [ ]:
# =============================================================================
# Step31C1 Cell 47
# Reload frozen harmonized QC raw object
# =============================================================================

import scanpy as sc

RAW_QC_PATH = (
    DATA_DIR /
    "GSE326201_STEP31B4_HARMONIZED_QC_RAW.h5ad"
)


adata = sc.read_h5ad(
    RAW_QC_PATH
)


print(
    adata
)

print(
    "\nShape:",
    adata.shape
)


assert adata.n_obs == 90991
assert adata.n_vars == 32732

assert adata.obs_names.is_unique


print(
    "\nSTEP31C INPUT: PASS"
)

In [ ]:
# =============================================================================
# Step31C1 Cell 48A
# Install missing dependency for Scrublet automatic thresholding
# =============================================================================

%pip install scikit-image

In [ ]:
# =============================================================================
# Step31C1 Cell 48B
# Verify Scrublet dependency
# =============================================================================

import importlib.util
import skimage

print(
    "scikit-image version:",
    skimage.__version__
)

print(
    "skimage available:",
    importlib.util.find_spec("skimage") is not None
)

assert importlib.util.find_spec("skimage") is not None

print(
    "\nSCRUBLET DEPENDENCY: PASS"
)

In [ ]:
# =============================================================================
# Step31C1 Cell 48
# Sample-wise Scrublet
# =============================================================================

import pandas as pd
import numpy as np


doublet_tables = []


for sample_id in (
    adata.obs[
        "sample_id"
    ]
    .astype(str)
    .unique()
):

    print(
        "\n" + "=" * 80
    )

    print(
        "Scrublet:",
        sample_id
    )


    sub = adata[
        adata.obs[
            "sample_id"
        ].astype(str)
        ==
        sample_id
    ].copy()


    print(
        "Cells:",
        sub.n_obs
    )


    sc.pp.scrublet(
        sub,

        batch_key=None,

        sim_doublet_ratio=2.0,

        expected_doublet_rate=0.06,

        stdev_doublet_rate=0.02,

        synthetic_doublet_umi_subsampling=1.0,

        knn_dist_metric="euclidean",

        normalize_variance=True,

        log_transform=False,

        mean_center=True,

        n_prin_comps=30,

        use_approx_neighbors=True,

        get_doublet_neighbor_parents=False,

        n_neighbors=None,

        threshold=None,

        verbose=False,

        random_state=42
    )


    tmp = sub.obs[
        [
            "doublet_score",
            "predicted_doublet"
        ]
    ].copy()


    tmp[
        "sample_id"
    ] = sample_id


    doublet_tables.append(
        tmp
    )


    print(
        "Predicted doublets:",
        int(
            tmp[
                "predicted_doublet"
            ].sum()
        ),
        "/",
        len(tmp),
        "=",
        round(
            tmp[
                "predicted_doublet"
            ].mean(),
            4
        )
    )


doublet_calls = pd.concat(
    doublet_tables,
    axis=0
)


assert doublet_calls.index.is_unique

assert set(
    doublet_calls.index
) == set(
    adata.obs_names
)


print(
    "\nDOUBLETS COMPLETE"
)

In [ ]:
# =============================================================================
# Step31C1 Cell 49
# Attach doublet calls
# =============================================================================

adata.obs[
    "doublet_score"
] = (
    doublet_calls.loc[
        adata.obs_names,
        "doublet_score"
    ]
    .astype(float)
)


adata.obs[
    "predicted_doublet"
] = (
    doublet_calls.loc[
        adata.obs_names,
        "predicted_doublet"
    ]
    .astype(bool)
)


print(
    "Total cells:",
    adata.n_obs
)

print(
    "Predicted doublets:",
    int(
        adata.obs[
            "predicted_doublet"
        ].sum()
    )
)

print(
    "Predicted doublet fraction:",
    float(
        adata.obs[
            "predicted_doublet"
        ].mean()
    )
)

In [ ]:
# =============================================================================
# Step31C1 Cell 50
# Sample-wise doublet audit
# =============================================================================

doublet_summary = (
    adata.obs
    .groupby(
        [
            "sample_id",
            "patient_id",
            "tissue"
        ],
        observed=True
    )
    .agg(
        n_cells=(
            "predicted_doublet",
            "size"
        ),

        n_doublets=(
            "predicted_doublet",
            "sum"
        ),

        doublet_fraction=(
            "predicted_doublet",
            "mean"
        ),

        median_doublet_score=(
            "doublet_score",
            "median"
        )
    )
    .reset_index()
)


display(
    doublet_summary
)


print(
    "\nOverall doublet fraction:",
    adata.obs[
        "predicted_doublet"
    ].mean()
)

In [ ]:
# =============================================================================
# Step31C1 Cell 51
# Freeze singlet universe
# =============================================================================

adata_singlet = (
    adata[
        ~adata.obs[
            "predicted_doublet"
        ]
    ]
    .copy()
)


print(
    "Before doublet removal:",
    adata.n_obs
)

print(
    "After doublet removal:",
    adata_singlet.n_obs
)

print(
    "Removed:",
    adata.n_obs
    -
    adata_singlet.n_obs
)


print(
    "\nCells by tissue:"
)

display(
    adata_singlet.obs[
        "tissue"
    ].value_counts()
)

In [ ]:
# =============================================================================
# Step31C1 Cell 52
# Save singlet raw-count object
# =============================================================================

DOUBLETS_PATH = (
    AUDIT_DIR /
    "Step31C1_DoubletCalls.csv"
)

SINGLET_PATH = (
    DATA_DIR /
    "GSE326201_STEP31C1_SINGLET_RAW.h5ad"
)


doublet_summary.to_csv(
    AUDIT_DIR /
    "Step31C1_DoubletSummary_BySample.csv",
    index=False
)


adata.obs[
    [
        "sample_id",
        "patient_id",
        "tissue",
        "doublet_score",
        "predicted_doublet"
    ]
].to_csv(
    DOUBLETS_PATH
)


adata_singlet.write_h5ad(
    SINGLET_PATH,
    compression="gzip"
)


print(
    SINGLET_PATH
)

In [ ]:
# =============================================================================
# Step31C1 Cell 53
# REPORT — doublet handling
# =============================================================================

print(
    "=" * 110
)

print(
    "STEP31C1 — SAMPLE-WISE DOUBLET HANDLING REPORT"
)

print(
    "=" * 110
)


print(
    "\n[1] INPUT"
)

print(
    "Cells:",
    adata.n_obs
)

print(
    "Genes:",
    adata.n_vars
)

print(
    "Samples:",
    adata.obs[
        "sample_id"
    ].nunique()
)


print(
    "\n[2] DOUBLET SUMMARY"
)

display(
    doublet_summary
)


print(
    "\nTotal predicted doublets:",
    int(
        adata.obs[
            "predicted_doublet"
        ].sum()
    )
)

print(
    "Overall doublet fraction:",
    float(
        adata.obs[
            "predicted_doublet"
        ].mean()
    )
)


print(
    "\n[3] FINAL SINGLET UNIVERSE"
)

print(
    "Cells:",
    adata_singlet.n_obs
)

print(
    "Genes:",
    adata_singlet.n_vars
)


print(
    "\nCells by tissue:"
)

display(
    adata_singlet.obs[
        "tissue"
    ].value_counts()
)


print(
    "\nCells by sample:"
)

display(
    adata_singlet.obs[
        "sample_id"
    ].value_counts()
)


print(
    "\n[4] FILE"
)

print(
    SINGLET_PATH
)

print(
    "Exists:",
    SINGLET_PATH.exists()
)


print(
    "\nSTEP31C1: COMPLETE"
)

print(
    "NEXT: normalization + HVG + PCA + sample-aware integration + major-cell annotation."
)

print(
    "=" * 110
)

In [ ]:
# =============================================================================
# Step31C1.1 Cell 54
# Save QC-pass all-cell object with Scrublet metadata
# Doublets are flagged but NOT removed yet
# =============================================================================

SCRUBLET_FLAGGED_PATH = (
    DATA_DIR /
    "GSE326201_STEP31C1_QC_SCRUBLET_FLAGGED_RAW.h5ad"
)

adata.write_h5ad(
    SCRUBLET_FLAGGED_PATH,
    compression="gzip"
)

print("Saved:")
print(SCRUBLET_FLAGGED_PATH)

print("\nShape:")
print(adata.shape)

print("\nPredicted doublets:")
print(
    int(
        adata.obs[
            "predicted_doublet"
        ].sum()
    ),
    "/",
    adata.n_obs
)

print(
    "\nIMPORTANT:"
)

print(
    "Scrublet calls are provisional flags; "
    "cells have NOT been removed from the Step31 main analysis universe."
)

In [ ]:
# =============================================================================
# Step31C1.1 Cell 55
# Doublet-score distribution audit
# =============================================================================

doublet_score_audit = (
    adata.obs
    .groupby(
        [
            "sample_id",
            "patient_id",
            "tissue"
        ],
        observed=True
    )
    .agg(
        n_cells=(
            "doublet_score",
            "size"
        ),

        predicted_doublets=(
            "predicted_doublet",
            "sum"
        ),

        q50_score=(
            "doublet_score",
            lambda x: x.quantile(0.50)
        ),

        q90_score=(
            "doublet_score",
            lambda x: x.quantile(0.90)
        ),

        q95_score=(
            "doublet_score",
            lambda x: x.quantile(0.95)
        ),

        q99_score=(
            "doublet_score",
            lambda x: x.quantile(0.99)
        ),

        max_score=(
            "doublet_score",
            "max"
        )
    )
    .reset_index()
)

doublet_score_audit[
    "predicted_fraction"
] = (
    doublet_score_audit[
        "predicted_doublets"
    ]
    /
    doublet_score_audit[
        "n_cells"
    ]
)

display(
    doublet_score_audit
)

doublet_score_audit.to_csv(
    AUDIT_DIR /
    "Step31C1_DoubletScoreDistribution_BySample.csv",
    index=False
)

In [ ]:
# =============================================================================
# Step31C1.1 Cell 56
# Revised doublet-handling lock
# =============================================================================

print("=" * 100)

print(
    "STEP31C1 REVISED DOUBLET-HANDLING LOCK"
)

print("=" * 100)

print(
    "\nQC-pass cells:",
    adata.n_obs
)

print(
    "Scrublet-predicted doublets:",
    int(
        adata.obs[
            "predicted_doublet"
        ].sum()
    )
)

print(
    "Predicted fraction:",
    float(
        adata.obs[
            "predicted_doublet"
        ].mean()
    )
)

print(
"""
DECISION:

1. Scrublet was run independently within each of the 18 libraries.
2. Automatic thresholding produced a low overall predicted-doublet fraction.
3. Scrublet scores and predicted calls are therefore retained as provisional
   doublet evidence.
4. Predicted doublets are NOT removed before the initial major-cell landscape.
5. Final doublet exclusion will be performed after PCA / neighborhood /
   clustering audit using:
      - Scrublet score,
      - predicted-doublet enrichment,
      - mixed / incompatible lineage marker expression,
      - cluster context.
6. The provisional 90,787-cell singlet object is NOT the frozen Step31
   analysis universe.

Frozen current universe:
    90,991 QC-pass cells × 32,732 common Ensembl genes.
"""
)

print(
    "\nSTEP31C1.1: LOCKED"
)

print(
    "NEXT: Step31C2 — normalization, HVG, PCA and preliminary major-cell landscape."
)

print("=" * 100)

In [ ]:
# =============================================================================
# Step31B4.1 — Core50 SGOL1/SGOL2 alias audit
# Safe to rerun
# =============================================================================

import pandas as pd

TARGET_SYMBOLS = [
    "SGOL1",
    "SGO1",
    "SGOL2",
    "SGO2"
]

alias_rows = []

for sample_id, mapping in gene_symbol_maps.items():

    hit = mapping.loc[
        mapping["gene_symbol"]
        .astype(str)
        .isin(TARGET_SYMBOLS)
    ].copy()

    if len(hit) == 0:

        alias_rows.append({
            "sample_id": sample_id,
            "gene_symbol": None,
            "gene_id": None
        })

    else:

        for _, r in hit.iterrows():

            alias_rows.append({
                "sample_id": sample_id,
                "gene_symbol": str(r["gene_symbol"]),
                "gene_id": str(r["gene_id"])
            })


alias_audit = pd.DataFrame(alias_rows)


print("=" * 100)
print("CORE50 SGOL1 / SGOL2 ALIAS AUDIT")
print("=" * 100)


print("\n[1] Per-sample symbol / Ensembl mapping")

display(
    alias_audit
)


print("\n[2] Symbol occurrence across 18 libraries")

display(
    alias_audit[
        "gene_symbol"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "gene_symbol"
    )
    .reset_index(
        name="n_libraries"
    )
)


print("\n[3] Hits in the frozen 32,732-gene common universe")

common_alias_hits = (
    common_mapping.loc[
        common_mapping[
            "gene_symbol"
        ]
        .astype(str)
        .isin(TARGET_SYMBOLS)
    ]
    .drop_duplicates()
    .sort_values(
        [
            "gene_symbol",
            "gene_id"
        ]
    )
)

display(
    common_alias_hits
)


print("\n[4] Interpretation helper")

for old_symbol, candidate_alias in [
    ("SGOL1", "SGO1"),
    ("SGOL2", "SGO2")
]:

    old_present = (
        common_mapping[
            "gene_symbol"
        ]
        .astype(str)
        .eq(old_symbol)
        .any()
    )

    alias_present = (
        common_mapping[
            "gene_symbol"
        ]
        .astype(str)
        .eq(candidate_alias)
        .any()
    )

    print(
        f"{old_symbol}: "
        f"original_symbol_present={old_present}; "
        f"candidate_alias_{candidate_alias}_present={alias_present}"
    )


print("\nIMPORTANT:")
print(
    "Do NOT modify frozen Core50 yet. "
    "This cell only audits whether the two apparently missing genes "
    "are explained by nomenclature differences."
)

print("=" * 100)

In [ ]:
# =============================================================================
# Step31B4.1 Cell 57
# Freeze Core50 nomenclature harmonization
# =============================================================================

CORE50_ALIAS_MAP = {
    "SGOL1": "SGO1",
    "SGOL2": "SGO2"
}


core50_validation_rows = []


for canonical_gene in CORE50:

    # Keep original frozen Core50 identity,
    # but use the updated symbol only for GSE326201 expression retrieval.
    validation_symbol = (
        CORE50_ALIAS_MAP.get(
            canonical_gene,
            canonical_gene
        )
    )

    hit = (
        common_mapping.loc[
            common_mapping[
                "gene_symbol"
            ]
            .astype(str)
            ==
            validation_symbol
        ]
        .drop_duplicates()
    )

    ensembl_ids = sorted(
        hit[
            "gene_id"
        ]
        .astype(str)
        .unique()
        .tolist()
    )


    core50_validation_rows.append({

        "canonical_symbol":
            canonical_gene,

        "validation_symbol":
            validation_symbol,

        "alias_used":
            canonical_gene
            !=
            validation_symbol,

        "n_common_ensembl_ids":
            len(
                ensembl_ids
            ),

        "ensembl_ids":
            " | ".join(
                ensembl_ids
            )
    })


CORE50_VALIDATION_MAP = (
    pd.DataFrame(
        core50_validation_rows
    )
)


display(
    CORE50_VALIDATION_MAP
)


print(
    "\nCanonical frozen genes:",
    len(
        CORE50_VALIDATION_MAP
    )
)


print(
    "Genes recoverable in common universe:",
    int(
        (
            CORE50_VALIDATION_MAP[
                "n_common_ensembl_ids"
            ] > 0
        ).sum()
    ),
    "/ 50"
)


print(
    "Genes with unique one-to-one Ensembl mapping:",
    int(
        (
            CORE50_VALIDATION_MAP[
                "n_common_ensembl_ids"
            ] == 1
        ).sum()
    ),
    "/ 50"
)


print(
    "Alias substitutions used:",
    int(
        CORE50_VALIDATION_MAP[
            "alias_used"
        ].sum()
    )
)

In [ ]:
# =============================================================================
# Step31B4.1 Cell 58
# Validate and save frozen Core50 validation mapping
# =============================================================================

# -------------------------------------------------------------
# Strict checks
# -------------------------------------------------------------

assert len(
    CORE50_VALIDATION_MAP
) == 50


assert (
    CORE50_VALIDATION_MAP[
        "canonical_symbol"
    ].nunique()
    ==
    50
)


assert (
    CORE50_VALIDATION_MAP[
        "n_common_ensembl_ids"
    ] > 0
).all(), (
    "At least one frozen Core50 gene is not recoverable."
)


assert (
    CORE50_VALIDATION_MAP[
        "n_common_ensembl_ids"
    ] == 1
).all(), (
    "At least one Core50 gene does not have a unique Ensembl mapping."
)


# -------------------------------------------------------------
# Explicit SGOL / SGO checks
# -------------------------------------------------------------

sgol1_row = (
    CORE50_VALIDATION_MAP.loc[
        CORE50_VALIDATION_MAP[
            "canonical_symbol"
        ] == "SGOL1"
    ]
    .iloc[0]
)

sgol2_row = (
    CORE50_VALIDATION_MAP.loc[
        CORE50_VALIDATION_MAP[
            "canonical_symbol"
        ] == "SGOL2"
    ]
    .iloc[0]
)


assert (
    sgol1_row[
        "validation_symbol"
    ]
    ==
    "SGO1"
)

assert (
    sgol1_row[
        "ensembl_ids"
    ]
    ==
    "ENSG00000129810"
)


assert (
    sgol2_row[
        "validation_symbol"
    ]
    ==
    "SGO2"
)

assert (
    sgol2_row[
        "ensembl_ids"
    ]
    ==
    "ENSG00000163535"
)


# -------------------------------------------------------------
# Save mapping
# -------------------------------------------------------------

CORE50_MAP_PATH = (
    AUDIT_DIR /
    "Step31_Core50_FrozenValidationMapping.csv"
)


CORE50_VALIDATION_MAP.to_csv(
    CORE50_MAP_PATH,
    index=False
)


print(
    "Saved:"
)

print(
    CORE50_MAP_PATH
)


print(
    "\nCORE50 VALIDATION MAPPING: PASS"
)

In [ ]:
# =============================================================================
# Step31B4.1 Cell 59
# Freeze Ensembl-based Core50 validation feature set
# =============================================================================

CORE50_ENSEMBL_MAP = dict(
    zip(
        CORE50_VALIDATION_MAP[
            "canonical_symbol"
        ],
        CORE50_VALIDATION_MAP[
            "ensembl_ids"
        ]
    )
)


CORE50_ENSEMBL_IDS = (
    CORE50_VALIDATION_MAP[
        "ensembl_ids"
    ]
    .tolist()
)


print(
    "Frozen Core50 canonical genes:",
    len(
        CORE50_ENSEMBL_MAP
    )
)

print(
    "Frozen Core50 Ensembl IDs:",
    len(
        CORE50_ENSEMBL_IDS
    )
)

print(
    "Unique Ensembl IDs:",
    len(
        set(
            CORE50_ENSEMBL_IDS
        )
    )
)


assert (
    len(
        CORE50_ENSEMBL_IDS
    )
    ==
    50
)

assert (
    len(
        set(
            CORE50_ENSEMBL_IDS
        )
    )
    ==
    50
)


print(
    "\nAlias examples:"
)

print(
    "SGOL1 ->",
    CORE50_ALIAS_MAP[
        "SGOL1"
    ],
    "->",
    CORE50_ENSEMBL_MAP[
        "SGOL1"
    ]
)

print(
    "SGOL2 ->",
    CORE50_ALIAS_MAP[
        "SGOL2"
    ],
    "->",
    CORE50_ENSEMBL_MAP[
        "SGOL2"
    ]
)


print(
    "\nCORE50 ENSEMBL FEATURE SET: FROZEN"
)

In [ ]:
# =============================================================================
# Step31B4.1 Cell 60
# REPORT — Core50 nomenclature harmonization lock
# =============================================================================

print(
    "=" * 105
)

print(
    "STEP31 — CORE50 NOMENCLATURE HARMONIZATION LOCK"
)

print(
    "=" * 105
)


print(
    "\n[1] ORIGINAL SIGNATURE"
)

print(
    "Frozen Step25 Core50 genes:",
    len(
        CORE50
    )
)

print(
    "Signature membership changed:",
    False
)

print(
    "Genes reselected:",
    False
)


print(
    "\n[2] EXACT SYMBOL COVERAGE"
)

exact_symbol_coverage = sum(
    gene
    in
    set(
        common_mapping[
            "gene_symbol"
        ].astype(str)
    )
    for gene in CORE50
)

print(
    "Exact frozen-symbol coverage:",
    exact_symbol_coverage,
    "/ 50"
)


print(
    "\n[3] NOMENCLATURE HARMONIZATION"
)

alias_table = (
    CORE50_VALIDATION_MAP.loc[
        CORE50_VALIDATION_MAP[
            "alias_used"
        ],
        [
            "canonical_symbol",
            "validation_symbol",
            "ensembl_ids"
        ]
    ]
)

display(
    alias_table
)


print(
    "\n[4] FINAL BIOLOGICAL COVERAGE"
)

print(
    "Recoverable frozen genes:",
    int(
        (
            CORE50_VALIDATION_MAP[
                "n_common_ensembl_ids"
            ] > 0
        ).sum()
    ),
    "/ 50"
)

print(
    "Unique Ensembl mappings:",
    int(
        (
            CORE50_VALIDATION_MAP[
                "n_common_ensembl_ids"
            ] == 1
        ).sum()
    ),
    "/ 50"
)


print(
    "\n[5] LOCKED ALIAS MAP"
)

print(
    CORE50_ALIAS_MAP
)


print(
    "\n[6] INTERPRETATION"
)

print(
"""
The original Step25 MS3 Core50 signature remains unchanged.

Exact symbol matching recovered 48/50 genes.

The two apparently missing genes were explained entirely by gene-symbol
nomenclature differences in the GSE326201 reference:

    SGOL1 -> SGO1 -> ENSG00000129810
    SGOL2 -> SGO2 -> ENSG00000163535

Both updated symbols are present in all 18 libraries and belong to the
32,732-gene common Ensembl universe.

Therefore:

    biological Core50 coverage = 50/50

Future Step31 Core50 expression retrieval will use frozen Ensembl IDs,
while results and figures will continue to report the original canonical
Core50 gene identities.

No signature genes were added, removed, or reselected.
"""
)


print(
    "\n[7] SAVED MAPPING"
)

print(
    CORE50_MAP_PATH
)

print(
    "Exists:",
    CORE50_MAP_PATH.exists()
)


print(
    "\nCORE50 VALIDATION MAPPING: FROZEN"
)

print(
    "NEXT: Step31C2 — preprocessing and preliminary major-cell landscape."
)

print(
    "=" * 105
)

## Step31C2 — Preliminary major-cell landscape

### Principles

- Input: 90,991 QC-pass cells.
- Scrublet calls remain provisional and no cells are removed here.
- Raw counts remain preserved in the frozen Step31C1 raw-count H5AD.
- Expression is library-size normalized and log1p transformed.
- HVGs are selected in a sample-aware manner.
- PCA is calculated from shared HVGs.
- Harmony uses patient_id only, not sample_id or tissue.
- Harmony-corrected PCs are used only for neighborhood topology, UMAP and clustering.
- Biological program scores are calculated from uncorrected log-normalized expression.
- Broad lineage assignments remain preliminary until cluster and doublet audits are reviewed.

In [ ]:
# =============================================================================
# Step31C2 Cell 62
# Reload QC-pass Scrublet-flagged raw object
# =============================================================================

import scanpy as sc
import pandas as pd
import numpy as np
from pathlib import Path

SCRUBLET_FLAGGED_PATH = (
    DATA_DIR /
    "GSE326201_STEP31C1_QC_SCRUBLET_FLAGGED_RAW.h5ad"
)

adata_c2 = sc.read_h5ad(
    SCRUBLET_FLAGGED_PATH
)

print(adata_c2)

print("\nShape:", adata_c2.shape)

assert adata_c2.n_obs == 90991
assert adata_c2.n_vars == 32732
assert adata_c2.obs_names.is_unique

assert "doublet_score" in adata_c2.obs
assert "predicted_doublet" in adata_c2.obs

print("\nSTEP31C2 INPUT: PASS")

In [ ]:
# =============================================================================
# Step31C2 Cell 63
# Freeze canonical display symbol for common Ensembl genes
# =============================================================================

symbol_support = (
    mapping_all.loc[
        mapping_all["gene_id"].isin(
            adata_c2.var_names
        )
    ]
    .groupby(
        ["gene_id", "gene_symbol"]
    )
    .size()
    .reset_index(
        name="n_library_support"
    )
)

symbol_support = (
    symbol_support
    .sort_values(
        [
            "gene_id",
            "n_library_support",
            "gene_symbol"
        ],
        ascending=[
            True,
            False,
            True
        ]
    )
)

canonical_symbol_map = (
    symbol_support
    .drop_duplicates(
        subset="gene_id",
        keep="first"
    )
    .copy()
)

n_variants = (
    symbol_support
    .groupby("gene_id")
    ["gene_symbol"]
    .nunique()
    .rename("n_symbol_variants")
)

canonical_symbol_map = (
    canonical_symbol_map
    .merge(
        n_variants,
        on="gene_id",
        how="left"
    )
)

assert canonical_symbol_map["gene_id"].nunique() == 32732


symbol_dict = dict(
    zip(
        canonical_symbol_map["gene_id"],
        canonical_symbol_map["gene_symbol"]
    )
)


adata_c2.var["gene_symbol"] = [
    symbol_dict[x]
    for x in adata_c2.var_names
]


display(
    canonical_symbol_map.head()
)

print(
    "\nGenes with >1 historical symbol:",
    int(
        (
            canonical_symbol_map[
                "n_symbol_variants"
            ] > 1
        ).sum()
    )
)

In [ ]:
# =============================================================================
# Step31C2 Cell 64
# Save canonical gene-symbol mapping
# =============================================================================

CANONICAL_SYMBOL_PATH = (
    AUDIT_DIR /
    "Step31C2_CommonGene_CanonicalSymbolMap.csv"
)

canonical_symbol_map.to_csv(
    CANONICAL_SYMBOL_PATH,
    index=False
)

print(CANONICAL_SYMBOL_PATH)
print("Exists:", CANONICAL_SYMBOL_PATH.exists())

In [ ]:
# =============================================================================
# Step31C2 Cell 65
# Library-size normalization + log1p
# =============================================================================

adata_c2.uns[
    "raw_counts_source"
] = str(
    SCRUBLET_FLAGGED_PATH
)

sc.pp.normalize_total(
    adata_c2,
    target_sum=1e4
)

sc.pp.log1p(
    adata_c2
)

print(
    "Normalized/log1p object:"
)

print(
    adata_c2
)

print(
    "\nRaw counts remain preserved externally at:"
)

print(
    adata_c2.uns[
        "raw_counts_source"
    ]
)

In [ ]:
# =============================================================================
# Step31C2 Cell 66
# Technical gene classes
# =============================================================================

symbols_upper = (
    adata_c2.var[
        "gene_symbol"
    ]
    .astype(str)
    .str.upper()
)

adata_c2.var[
    "mt"
] = (
    symbols_upper
    .str.startswith(
        "MT-"
    )
)

adata_c2.var[
    "ribo"
] = (
    symbols_upper
    .str.startswith(
        (
            "RPL",
            "RPS"
        )
    )
)

print(
    "MT genes:",
    int(
        adata_c2.var[
            "mt"
        ].sum()
    )
)

print(
    "Ribosomal genes:",
    int(
        adata_c2.var[
            "ribo"
        ].sum()
    )
)

In [ ]:
# =============================================================================
# Step31C2 Cell 67
# Sample-aware HVG selection
# =============================================================================

sc.pp.highly_variable_genes(
    adata_c2,
    flavor="seurat",
    n_top_genes=3000,
    batch_key="sample_id",
    inplace=True
)


adata_c2.var[
    "hvg_for_pca"
] = (
    adata_c2.var[
        "highly_variable"
    ]
    &
    ~adata_c2.var[
        "mt"
    ]
    &
    ~adata_c2.var[
        "ribo"
    ]
)


print(
    "Initial HVGs:",
    int(
        adata_c2.var[
            "highly_variable"
        ].sum()
    )
)

print(
    "HVGs used for PCA after MT/ribo exclusion:",
    int(
        adata_c2.var[
            "hvg_for_pca"
        ].sum()
    )
)


print(
    "\nHVG batch support:"
)

display(
    adata_c2.var.loc[
        adata_c2.var[
            "highly_variable"
        ],
        "highly_variable_nbatches"
    ]
    .value_counts()
    .sort_index()
)

In [ ]:
# =============================================================================
# Step31C2 Cell 68
# PCA on shared HVGs
# =============================================================================

import gc

adata_pca = (
    adata_c2[
        :,
        adata_c2.var[
            "hvg_for_pca"
        ]
    ]
    .copy()
)

print(
    "PCA input:",
    adata_pca.shape
)


sc.pp.scale(
    adata_pca,
    max_value=10
)


sc.tl.pca(
    adata_pca,
    n_comps=50,
    svd_solver="arpack",
    random_state=42
)


adata_c2.obsm[
    "X_pca"
] = (
    adata_pca.obsm[
        "X_pca"
    ].copy()
)


adata_c2.obsm[
    "X_pca_unintegrated"
] = (
    adata_pca.obsm[
        "X_pca"
    ].copy()
)


print(
    "PCA shape:",
    adata_c2.obsm[
        "X_pca"
    ].shape
)


del adata_pca
gc.collect()

print(
    "\nPCA: COMPLETE"
)

In [ ]:
# =============================================================================
# Step31C2 Cell 69
# Ensure Harmony dependency
# =============================================================================

import importlib.util
import subprocess
import sys


if importlib.util.find_spec(
    "harmonypy"
) is None:

    print(
        "Installing harmonypy..."
    )

    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "harmonypy"
    ])


import harmonypy

print(
    "harmonypy available: PASS"
)

In [ ]:
# =============================================================================
# Step31C2 Cell 70 — REVISED
# Patient-only Harmony integration
# Robust to harmonypy Z_corr orientation
# =============================================================================

import numpy as np
import harmonypy as hm


# -------------------------------------------------------------------------
# Use the frozen unintegrated PCA representation
# -------------------------------------------------------------------------

X_pca_input = np.asarray(
    adata_c2.obsm[
        "X_pca_unintegrated"
    ],
    dtype=np.float64
)

n_cells, n_pcs = X_pca_input.shape


print(
    "Harmony input shape:",
    X_pca_input.shape
)

print(
    "Harmony covariate:",
    "patient_id"
)


# -------------------------------------------------------------------------
# Run Harmony directly
# -------------------------------------------------------------------------

harmony_out = hm.run_harmony(
    X_pca_input,
    adata_c2.obs,
    vars_use=[
        "patient_id"
    ],
    max_iter_harmony=20,
    random_state=42
)


Z_corr = np.asarray(
    harmony_out.Z_corr
)


print(
    "\nRaw harmonypy Z_corr shape:",
    Z_corr.shape
)


# -------------------------------------------------------------------------
# Robust orientation handling
# -------------------------------------------------------------------------

if Z_corr.shape == (
    n_cells,
    n_pcs
):

    X_harmony = Z_corr

    orientation = (
        "cells × PCs returned directly"
    )


elif Z_corr.shape == (
    n_pcs,
    n_cells
):

    X_harmony = Z_corr.T

    orientation = (
        "PCs × cells returned; transposed once"
    )


else:

    raise ValueError(
        "Unexpected Harmony output shape: "
        f"{Z_corr.shape}; "
        f"expected {(n_cells, n_pcs)} "
        f"or {(n_pcs, n_cells)}"
    )


# Store as float32 to reduce memory
X_harmony = np.asarray(
    X_harmony,
    dtype=np.float32
)


assert X_harmony.shape == (
    adata_c2.n_obs,
    n_pcs
)


adata_c2.obsm[
    "X_pca_harmony_patient"
] = X_harmony


# -------------------------------------------------------------------------
# Record provenance
# -------------------------------------------------------------------------

adata_c2.uns[
    "Step31C2_harmony"
] = {

    "batch_key":
        "patient_id",

    "input_basis":
        "X_pca_unintegrated",

    "output_basis":
        "X_pca_harmony_patient",

    "n_pcs":
        int(n_pcs),

    "max_iter_harmony":
        20,

    "random_state":
        42,

    "orientation_handling":
        orientation
}


print(
    "\nOrientation:",
    orientation
)

print(
    "Final Harmony shape:",
    adata_c2.obsm[
        "X_pca_harmony_patient"
    ].shape
)

print(
    "\nPATIENT-ONLY HARMONY: PASS"
)

In [ ]:
# =============================================================================
# Step31C2 Cell 71
# Patient-Harmony neighbors and UMAP
# =============================================================================

sc.pp.neighbors(
    adata_c2,
    n_neighbors=15,
    n_pcs=30,
    use_rep="X_pca_harmony_patient",
    key_added="neighbors_harmony_patient",
    random_state=42
)


sc.tl.umap(
    adata_c2,
    neighbors_key="neighbors_harmony_patient",
    random_state=42
)


adata_c2.obsm[
    "X_umap_harmony_patient"
] = (
    adata_c2.obsm[
        "X_umap"
    ].copy()
)


print(
    "UMAP:",
    adata_c2.obsm[
        "X_umap_harmony_patient"
    ].shape
)

In [ ]:
# =============================================================================
# Step31C2 Cell 72
# Ensure Leiden dependencies
# =============================================================================

import importlib.util
import subprocess
import sys


missing = []

for pkg in [
    "igraph",
    "leidenalg"
]:
    if importlib.util.find_spec(pkg) is None:
        missing.append(pkg)


if missing:

    print(
        "Installing:",
        missing
    )

    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "igraph",
        "leidenalg"
    ])


print(
    "Leiden dependencies: PASS"
)

In [ ]:
# =============================================================================
# Step31C2 Cell 73
# Preliminary Leiden clustering
# =============================================================================

sc.tl.leiden(
    adata_c2,
    resolution=0.8,
    neighbors_key="neighbors_harmony_patient",
    key_added="leiden_harmony_r08",
    random_state=42
)


print(
    "Clusters:",
    adata_c2.obs[
        "leiden_harmony_r08"
    ].nunique()
)


display(
    adata_c2.obs[
        "leiden_harmony_r08"
    ]
    .value_counts()
    .sort_index()
)

In [ ]:
# =============================================================================
# Step31C2 Cell 74
# Prespecified broad-lineage marker panel
# =============================================================================

LINEAGE_MARKERS = {

    "T_NK": [
        "CD3D",
        "CD3E",
        "TRBC1",
        "TRBC2",
        "NKG7",
        "GNLY"
    ],

    "Myeloid": [
        "LST1",
        "TYROBP",
        "FCER1G",
        "AIF1",
        "CTSS",
        "LILRB1"
    ],

    "B_Plasma": [
        "CD79A",
        "MS4A1",
        "CD37",
        "CD74",
        "MZB1",
        "JCHAIN"
    ],

    "Endothelial": [
        "PECAM1",
        "VWF",
        "EMCN",
        "KDR",
        "ENG",
        "RAMP2"
    ],

    "Fibroblast_Stellate": [
        "COL1A1",
        "COL1A2",
        "COL3A1",
        "DCN",
        "LUM",
        "COL6A1"
    ],

    "Hepatocyte": [
        "ALB",
        "APOA1",
        "APOA2",
        "TTR",
        "ASGR1",
        "CYP2E1"
    ],

    "Epithelial_Cholangiocyte": [
        "EPCAM",
        "KRT19",
        "KRT7",
        "KRT17",
        "KRT23",
        "TACSTD2"
    ],

    "Erythroid": [
        "HBB",
        "HBA1",
        "HBA2",
        "ALAS2"
    ],

    "Platelet": [
        "PPBP",
        "PF4",
        "GP9",
        "ITGA2B",
        "TUBB1"
    ],

    "Cycling": [
        "MKI67",
        "TOP2A",
        "CENPF",
        "UBE2C",
        "CCNB1",
        "CDC20"
    ]
}


print(
    "Programs:",
    len(
        LINEAGE_MARKERS
    )
)

In [ ]:
# =============================================================================
# Step31C2 Cell 75
# Broad marker coverage audit
# =============================================================================

symbol_to_ids = (
    canonical_symbol_map
    .groupby(
        "gene_symbol"
    )[
        "gene_id"
    ]
    .apply(list)
    .to_dict()
)


marker_coverage_rows = []


LINEAGE_GENE_IDS = {}


for program, markers in (
    LINEAGE_MARKERS.items()
):

    ids = []
    present_symbols = []
    missing_symbols = []


    for gene in markers:

        gene_ids = (
            symbol_to_ids.get(
                gene,
                []
            )
        )

        if len(gene_ids) == 1:

            ids.append(
                gene_ids[0]
            )

            present_symbols.append(
                gene
            )

        else:

            missing_symbols.append(
                gene
            )


    LINEAGE_GENE_IDS[
        program
    ] = ids


    marker_coverage_rows.append({

        "program":
            program,

        "n_requested":
            len(markers),

        "n_used":
            len(ids),

        "present_symbols":
            " | ".join(
                present_symbols
            ),

        "missing_or_ambiguous":
            " | ".join(
                missing_symbols
            )
    })


marker_coverage = pd.DataFrame(
    marker_coverage_rows
)


display(
    marker_coverage
)

In [ ]:
# =============================================================================
# Step31C2 Cell 76
# Broad lineage scores from uncorrected log-normalized expression
# =============================================================================

for program, gene_ids in (
    LINEAGE_GENE_IDS.items()
):

    assert len(
        gene_ids
    ) > 0


    score = np.asarray(
        adata_c2[
            :,
            gene_ids
        ].X.mean(
            axis=1
        )
    ).ravel()


    adata_c2.obs[
        f"score_{program}"
    ] = score


print(
    "Broad lineage scores added."
)

In [ ]:
# =============================================================================
# Step31C2 Cell 77
# Standardized broad-lineage scores
# =============================================================================

PROGRAMS = list(
    LINEAGE_MARKERS.keys()
)


for program in PROGRAMS:

    col = (
        f"score_{program}"
    )

    values = (
        adata_c2.obs[
            col
        ].astype(float)
    )

    sd = values.std()

    adata_c2.obs[
        f"z_{program}"
    ] = (
        (
            values
            -
            values.mean()
        )
        /
        sd
    )


print(
    "Z-score programs: COMPLETE"
)

In [ ]:
# =============================================================================
# Step31C2 Cell 78
# Cluster-level lineage / patient / doublet audit
# =============================================================================

cluster_key = (
    "leiden_harmony_r08"
)


base_cluster = (
    adata_c2.obs
    .groupby(
        cluster_key,
        observed=True
    )
    .agg(
        n_cells=(
            "sample_id",
            "size"
        ),

        n_samples=(
            "sample_id",
            "nunique"
        ),

        n_patients=(
            "patient_id",
            "nunique"
        ),

        tumor_fraction=(
            "tissue",
            lambda x:
                (
                    x.astype(str)
                    ==
                    "Tumor"
                ).mean()
        ),

        predicted_doublet_fraction=(
            "predicted_doublet",
            "mean"
        ),

        median_doublet_score=(
            "doublet_score",
            "median"
        )
    )
)


score_cols = [
    f"z_{x}"
    for x in PROGRAMS
]


cluster_scores = (
    adata_c2.obs
    .groupby(
        cluster_key,
        observed=True
    )[
        score_cols
    ]
    .mean()
)


cluster_audit = (
    base_cluster
    .join(
        cluster_scores
    )
    .reset_index()
)


display(
    cluster_audit
)

In [ ]:
# =============================================================================
# Step31C2 Cell 79
# Dominant-sample audit per cluster
# =============================================================================

cluster_sample_counts = (
    adata_c2.obs
    .groupby(
        [
            cluster_key,
            "sample_id"
        ],
        observed=True
    )
    .size()
    .rename(
        "n"
    )
    .reset_index()
)


cluster_totals = (
    cluster_sample_counts
    .groupby(
        cluster_key
    )[
        "n"
    ]
    .sum()
    .rename(
        "cluster_total"
    )
)


cluster_sample_counts = (
    cluster_sample_counts
    .merge(
        cluster_totals,
        on=cluster_key
    )
)


cluster_sample_counts[
    "fraction"
] = (
    cluster_sample_counts[
        "n"
    ]
    /
    cluster_sample_counts[
        "cluster_total"
    ]
)


dominant_sample = (
    cluster_sample_counts
    .sort_values(
        [
            cluster_key,
            "fraction"
        ],
        ascending=[
            True,
            False
        ]
    )
    .drop_duplicates(
        cluster_key
    )
    [
        [
            cluster_key,
            "sample_id",
            "fraction"
        ]
    ]
    .rename(
        columns={
            "sample_id":
                "dominant_sample",

            "fraction":
                "dominant_sample_fraction"
        }
    )
)


cluster_audit = (
    cluster_audit
    .merge(
        dominant_sample,
        on=cluster_key,
        how="left"
    )
)


display(
    cluster_audit
)

In [ ]:
# =============================================================================
# Step31C2 Cell 80
# Preliminary dominant lineage score
# =============================================================================

LINEAGE_ONLY = [
    "T_NK",
    "Myeloid",
    "B_Plasma",
    "Endothelial",
    "Fibroblast_Stellate",
    "Hepatocyte",
    "Epithelial_Cholangiocyte",
    "Erythroid",
    "Platelet"
]


lineage_z_cols = [
    f"z_{x}"
    for x in LINEAGE_ONLY
]


cluster_audit[
    "preliminary_top_program"
] = (
    cluster_audit[
        lineage_z_cols
    ]
    .idxmax(
        axis=1
    )
    .str.replace(
        "z_",
        "",
        regex=False
    )
)


cluster_audit[
    "preliminary_top_z"
] = (
    cluster_audit[
        lineage_z_cols
    ]
    .max(
        axis=1
    )
)


display(
    cluster_audit[
        [
            cluster_key,
            "n_cells",
            "n_patients",
            "tumor_fraction",
            "predicted_doublet_fraction",
            "dominant_sample",
            "dominant_sample_fraction",
            "preliminary_top_program",
            "preliminary_top_z",
            "z_Cycling"
        ]
    ]
)

In [ ]:
# =============================================================================
# Step31C2 Cell 81
# Save preliminary major-cell landscape
# =============================================================================

C2_PATH = (
    DATA_DIR /
    "GSE326201_STEP31C2_PRELIM_MAJORCELL_LANDSCAPE.h5ad"
)


cluster_audit.to_csv(
    AUDIT_DIR /
    "Step31C2_PreliminaryClusterAudit.csv",
    index=False
)


marker_coverage.to_csv(
    AUDIT_DIR /
    "Step31C2_BroadMarkerCoverage.csv",
    index=False
)


cluster_sample_counts.to_csv(
    AUDIT_DIR /
    "Step31C2_ClusterSampleComposition.csv",
    index=False
)


adata_c2.write_h5ad(
    C2_PATH,
    compression="gzip"
)


print(
    "Saved:"
)

print(
    C2_PATH
)

In [ ]:
# =============================================================================
# Step31C2 Cell 82
# REPORT — preliminary major-cell landscape
# =============================================================================

print("=" * 115)

print(
    "STEP31C2 — PRELIMINARY MAJOR-CELL LANDSCAPE REPORT"
)

print("=" * 115)


print(
    "\n[1] INPUT"
)

print(
    "Cells:",
    adata_c2.n_obs
)

print(
    "Genes:",
    adata_c2.n_vars
)

print(
    "Patients:",
    adata_c2.obs[
        "patient_id"
    ].nunique()
)

print(
    "Samples:",
    adata_c2.obs[
        "sample_id"
    ].nunique()
)


print(
    "\n[2] HVG / PCA"
)

print(
    "Initial HVGs:",
    int(
        adata_c2.var[
            "highly_variable"
        ].sum()
    )
)

print(
    "HVGs used for PCA:",
    int(
        adata_c2.var[
            "hvg_for_pca"
        ].sum()
    )
)

print(
    "PCA shape:",
    adata_c2.obsm[
        "X_pca_unintegrated"
    ].shape
)


print(
    "\n[3] INTEGRATION"
)

print(
    "Harmony key: patient_id"
)

print(
    "Harmony PCA shape:",
    adata_c2.obsm[
        "X_pca_harmony_patient"
    ].shape
)

print(
    "Expression matrix altered by Harmony:",
    False
)


print(
    "\n[4] PRELIMINARY CLUSTERS"
)

print(
    "Leiden clusters:",
    adata_c2.obs[
        cluster_key
    ].nunique()
)


display(
    adata_c2.obs[
        cluster_key
    ]
    .value_counts()
    .sort_index()
)


print(
    "\n[5] BROAD MARKER COVERAGE"
)

display(
    marker_coverage
)


print(
    "\n[6] CLUSTER AUDIT"
)

display(
    cluster_audit[
        [
            cluster_key,
            "n_cells",
            "n_samples",
            "n_patients",
            "tumor_fraction",
            "predicted_doublet_fraction",
            "dominant_sample",
            "dominant_sample_fraction",
            "preliminary_top_program",
            "preliminary_top_z",
            "z_Cycling"
        ]
    ]
)


print(
    "\n[7] SCRUBLET CONTEXT"
)

print(
    "Overall provisional doublet fraction:",
    float(
        adata_c2.obs[
            "predicted_doublet"
        ].mean()
    )
)


print(
    "Maximum cluster-level predicted-doublet fraction:",
    float(
        cluster_audit[
            "predicted_doublet_fraction"
        ].max()
    )
)


print(
    "\n[8] CORE50 STATUS"
)

print(
    "Frozen signature membership:",
    50
)

print(
    "Biological coverage:",
    "50/50"
)

print(
    "Alias mapping:",
    CORE50_ALIAS_MAP
)


print(
    "\n[9] CURRENT DECISION BOUNDARY"
)

print(
"""
This is a preliminary broad-cell landscape only.

No major-cell label has yet been frozen.

No malignant-cell population has yet been defined.

No Scrublet-flagged cells have yet been removed.

No Core50 validation score has yet been calculated.

NEXT:
Review cluster lineage programs, patient/sample composition and doublet
enrichment, then freeze broad cell types and final heterotypic-doublet
handling before malignant-cell identification.
"""
)


print(
    "\n[10] FILE"
)

print(
    C2_PATH
)

print(
    "Exists:",
    C2_PATH.exists()
)


print(
    "\nSTEP31C2: COMPLETE — AWAITING CLUSTER REVIEW"
)

print("=" * 115)

## Step31C3A — Cluster marker and epithelial-compartment audit

### Objectives

1. Characterize the 27 preliminary Leiden clusters using cluster marker genes.
2. Identify hepatic/epithelial populations that may not retain mature
   hepatocyte markers.
3. Distinguish proliferating lineage populations from possible mixed-lineage
   doublets.
4. Review Scrublet enrichment in cluster context.
5. Do not define malignant cells yet.
6. Do not calculate Core50 scores yet.

In [ ]:
# =============================================================================
# Step31C3A Cell 84
# Hepatic / epithelial audit marker panel
# =============================================================================

C3_AUDIT_MARKERS = {

    "Hepatic_Mature": [
        "ALB",
        "APOA1",
        "APOA2",
        "TTR",
        "ASGR1",
        "CYP2E1",
        "HP",
        "FGA",
        "FGB"
    ],

    "Hepatic_Epithelial": [
        "KRT8",
        "KRT18",
        "KRT19",
        "EPCAM",
        "KRT7",
        "TACSTD2"
    ],

    "HCC_Associated_Audit": [
        "GPC3",
        "AFP",
        "SPP1",
        "KRT19",
        "EPCAM"
    ],

    "T_Cell": [
        "CD3D",
        "CD3E",
        "TRAC",
        "TRBC1"
    ],

    "NK": [
        "NKG7",
        "GNLY",
        "KLRD1",
        "PRF1"
    ],

    "Myeloid": [
        "LST1",
        "TYROBP",
        "FCER1G",
        "AIF1"
    ],

    "B_Cell": [
        "CD79A",
        "MS4A1",
        "CD37",
        "CD74"
    ],

    "Plasma": [
        "JCHAIN",
        "MZB1",
        "SDC1",
        "IGHG1"
    ],

    "Endothelial": [
        "PECAM1",
        "VWF",
        "EMCN",
        "KDR"
    ],

    "Fibroblast": [
        "COL1A1",
        "COL1A2",
        "DCN",
        "LUM"
    ],

    "Erythroid": [
        "HBB",
        "HBA1",
        "HBA2",
        "ALAS2"
    ],

    "Platelet": [
        "PPBP",
        "PF4",
        "GP9",
        "ITGA2B"
    ],

    "Cycling": [
        "MKI67",
        "TOP2A",
        "UBE2C",
        "CCNB1",
        "CDC20"
    ]
}


print(
    "Audit programs:",
    len(C3_AUDIT_MARKERS)
)

In [ ]:
# =============================================================================
# Step31C3A Cell 85
# Map audit markers to common Ensembl universe
# =============================================================================

C3_AUDIT_IDS = {}

audit_marker_rows = []


for program, markers in C3_AUDIT_MARKERS.items():

    ids = []
    present = []
    missing = []
    ambiguous = []


    for gene in markers:

        hits = (
            canonical_symbol_map.loc[
                canonical_symbol_map[
                    "gene_symbol"
                ] == gene,
                "gene_id"
            ]
            .astype(str)
            .unique()
            .tolist()
        )


        if len(hits) == 1:

            ids.append(
                hits[0]
            )

            present.append(
                gene
            )


        elif len(hits) == 0:

            missing.append(
                gene
            )


        else:

            ambiguous.append(
                gene
            )


    C3_AUDIT_IDS[
        program
    ] = ids


    audit_marker_rows.append({

        "program":
            program,

        "n_requested":
            len(markers),

        "n_used":
            len(ids),

        "present":
            " | ".join(
                present
            ),

        "missing":
            " | ".join(
                missing
            ),

        "ambiguous":
            " | ".join(
                ambiguous
            )
    })


c3_marker_coverage = (
    pd.DataFrame(
        audit_marker_rows
    )
)


display(
    c3_marker_coverage
)


assert all(
    len(x) > 0
    for x in C3_AUDIT_IDS.values()
)

print(
    "\nC3 AUDIT MARKER MAPPING: PASS"
)

In [ ]:
# =============================================================================
# Step31C3A Cell 86
# Audit program scores from log-normalized expression
# =============================================================================

for program, gene_ids in C3_AUDIT_IDS.items():

    values = np.asarray(
        adata_c2[
            :,
            gene_ids
        ].X.mean(
            axis=1
        )
    ).ravel()


    adata_c2.obs[
        f"c3_{program}"
    ] = values


    sd = values.std()

    adata_c2.obs[
        f"c3z_{program}"
    ] = (
        (
            values
            -
            values.mean()
        )
        /
        sd
    )


print(
    "C3 audit scores added."
)

In [ ]:
# =============================================================================
# Step31C3A Cell 87
# Preliminary cluster marker characterization
# =============================================================================

CLUSTER_KEY = (
    "leiden_harmony_r08"
)


sc.tl.rank_genes_groups(
    adata_c2,
    groupby=CLUSTER_KEY,
    method="t-test_overestim_var",
    n_genes=100,
    use_raw=False,
    key_added="rank_genes_c3",
    pts=True
)


print(
    "Cluster marker ranking: COMPLETE"
)

In [ ]:
# =============================================================================
# Step31C3A Cell 88
# Extract top non-technical markers per cluster
# =============================================================================

cluster_marker_rows = []


clusters = sorted(
    adata_c2.obs[
        CLUSTER_KEY
    ]
    .astype(str)
    .unique(),
    key=lambda x: int(x)
)


for cluster in clusters:

    df = sc.get.rank_genes_groups_df(
        adata_c2,
        group=cluster,
        key="rank_genes_c3"
    )


    df[
        "gene_id"
    ] = df[
        "names"
    ].astype(str)


    df[
        "gene_symbol"
    ] = (
        df[
            "gene_id"
        ]
        .map(
            symbol_dict
        )
    )


    # Remove obvious technical mitochondrial / ribosomal markers
    technical = (
        df[
            "gene_symbol"
        ]
        .astype(str)
        .str.upper()
        .str.startswith(
            (
                "MT-",
                "RPL",
                "RPS"
            )
        )
    )


    df = (
        df.loc[
            ~technical
        ]
        .copy()
    )


    df = df.head(
        20
    )


    for rank, (_, r) in enumerate(
        df.iterrows(),
        start=1
    ):

        cluster_marker_rows.append({

            "cluster":
                cluster,

            "rank":
                rank,

            "gene_symbol":
                r[
                    "gene_symbol"
                ],

            "gene_id":
                r[
                    "gene_id"
                ],

            "score":
                r[
                    "scores"
                ],

            "logfoldchange":
                r[
                    "logfoldchanges"
                ],

            "pval_adj":
                r[
                    "pvals_adj"
                ]
        })


cluster_markers_top20 = (
    pd.DataFrame(
        cluster_marker_rows
    )
)


display(
    cluster_markers_top20.head(
        40
    )
)

In [ ]:
# =============================================================================
# Step31C3A Cell 89
# Compact Top10 marker summary
# =============================================================================

top10_marker_summary = (
    cluster_markers_top20.loc[
        cluster_markers_top20[
            "rank"
        ] <= 10
    ]
    .groupby(
        "cluster"
    )[
        "gene_symbol"
    ]
    .apply(
        lambda x:
            " | ".join(
                x.astype(str)
            )
    )
    .reset_index(
        name="top10_markers"
    )
)


display(
    top10_marker_summary
)

In [ ]:
# =============================================================================
# Step31C3A Cell 90
# Refined cluster program audit
# =============================================================================

c3_programs = list(
    C3_AUDIT_MARKERS.keys()
)


c3z_cols = [
    f"c3z_{x}"
    for x in c3_programs
]


cluster_c3_scores = (
    adata_c2.obs
    .groupby(
        CLUSTER_KEY,
        observed=True
    )[
        c3z_cols
    ]
    .mean()
    .reset_index()
)


cluster_c3_scores[
    "top_program"
] = (
    cluster_c3_scores[
        c3z_cols
    ]
    .idxmax(
        axis=1
    )
    .str.replace(
        "c3z_",
        "",
        regex=False
    )
)


cluster_c3_scores[
    "top_z"
] = (
    cluster_c3_scores[
        c3z_cols
    ]
    .max(
        axis=1
    )
)


# second-highest score
score_matrix = (
    cluster_c3_scores[
        c3z_cols
    ].to_numpy()
)


sorted_scores = np.sort(
    score_matrix,
    axis=1
)


cluster_c3_scores[
    "second_z"
] = (
    sorted_scores[
        :,
        -2
    ]
)


cluster_c3_scores[
    "top_minus_second"
] = (
    cluster_c3_scores[
        "top_z"
    ]
    -
    cluster_c3_scores[
        "second_z"
    ]
)


display(
    cluster_c3_scores
)

In [ ]:
# =============================================================================
# Step31C3A Cell 91
# Combined cluster review table
# =============================================================================

c3_cluster_review = (
    cluster_audit
    .merge(
        cluster_c3_scores,
        left_on="leiden_harmony_r08",
        right_on=CLUSTER_KEY,
        how="left",
        suffixes=(
            "",
            "_c3"
        )
    )
)


# Clean duplicated cluster key if necessary
if (
    f"{CLUSTER_KEY}_c3"
    in
    c3_cluster_review.columns
):

    c3_cluster_review = (
        c3_cluster_review.drop(
            columns=[
                f"{CLUSTER_KEY}_c3"
            ]
        )
    )


c3_cluster_review = (
    c3_cluster_review
    .merge(
        top10_marker_summary,
        left_on="leiden_harmony_r08",
        right_on="cluster",
        how="left"
    )
)


display(
    c3_cluster_review[
        [
            "leiden_harmony_r08",
            "n_cells",
            "n_patients",
            "tumor_fraction",
            "predicted_doublet_fraction",
            "dominant_sample",
            "dominant_sample_fraction",
            "top_program",
            "top_z",
            "top_minus_second",
            "c3z_Hepatic_Mature",
            "c3z_Hepatic_Epithelial",
            "c3z_HCC_Associated_Audit",
            "c3z_Cycling",
            "top10_markers"
        ]
    ]
)

In [ ]:
# =============================================================================
# Step31C3A Cell 92
# Special-cluster audit
# =============================================================================

SPECIAL_CLUSTERS = [
    "18",
    "22",
    "23",
    "25",
    "26"
]


special_cluster_audit = (
    c3_cluster_review.loc[
        c3_cluster_review[
            "leiden_harmony_r08"
        ].astype(str)
        .isin(
            SPECIAL_CLUSTERS
        )
    ]
)


display(
    special_cluster_audit[
        [
            "leiden_harmony_r08",
            "n_cells",
            "n_samples",
            "n_patients",
            "tumor_fraction",
            "predicted_doublet_fraction",
            "dominant_sample",
            "dominant_sample_fraction",
            "top_program",
            "top_z",
            "c3z_Cycling",
            "top10_markers"
        ]
    ]
)

In [ ]:
# =============================================================================
# Step31C3A Cell 93
# Hepatic / epithelial candidate audit
# =============================================================================

hepatic_audit = (
    c3_cluster_review[
        [
            "leiden_harmony_r08",
            "n_cells",
            "n_patients",
            "tumor_fraction",
            "dominant_sample",
            "dominant_sample_fraction",
            "c3z_Hepatic_Mature",
            "c3z_Hepatic_Epithelial",
            "c3z_HCC_Associated_Audit",
            "c3z_Cycling",
            "top10_markers"
        ]
    ]
    .sort_values(
        [
            "c3z_Hepatic_Epithelial",
            "c3z_Hepatic_Mature"
        ],
        ascending=False
    )
)


display(
    hepatic_audit
)

In [ ]:
# =============================================================================
# Step31C3A Cell 94
# Save cluster characterization audit
# =============================================================================

cluster_markers_top20.to_csv(
    AUDIT_DIR /
    "Step31C3A_ClusterTop20Markers.csv",
    index=False
)


c3_cluster_review.to_csv(
    AUDIT_DIR /
    "Step31C3A_ClusterReview.csv",
    index=False
)


c3_marker_coverage.to_csv(
    AUDIT_DIR /
    "Step31C3A_AuditMarkerCoverage.csv",
    index=False
)


print(
    "Step31C3A audit tables saved."
)

In [ ]:
# =============================================================================
# Step31C3A Cell 95
# REPORT — cluster characterization
# =============================================================================

print(
    "=" * 120
)

print(
    "STEP31C3A — CLUSTER CHARACTERIZATION AND EPITHELIAL AUDIT"
)

print(
    "=" * 120
)


print(
    "\n[1] DATASET"
)

print(
    "Cells:",
    adata_c2.n_obs
)

print(
    "Clusters:",
    adata_c2.obs[
        CLUSTER_KEY
    ].nunique()
)


print(
    "\n[2] AUDIT MARKER COVERAGE"
)

display(
    c3_marker_coverage
)


print(
    "\n[3] TOP10 CLUSTER MARKERS"
)

display(
    top10_marker_summary
)


print(
    "\n[4] REFINED CLUSTER REVIEW"
)

display(
    c3_cluster_review[
        [
            "leiden_harmony_r08",
            "n_cells",
            "n_patients",
            "tumor_fraction",
            "predicted_doublet_fraction",
            "dominant_sample",
            "dominant_sample_fraction",
            "top_program",
            "top_z",
            "top_minus_second",
            "c3z_Hepatic_Mature",
            "c3z_Hepatic_Epithelial",
            "c3z_HCC_Associated_Audit",
            "c3z_Cycling",
            "top10_markers"
        ]
    ]
)


print(
    "\n[5] HEPATIC / EPITHELIAL AUDIT"
)

display(
    hepatic_audit.head(
        12
    )
)


print(
    "\n[6] SPECIAL CLUSTERS"
)

display(
    special_cluster_audit[
        [
            "leiden_harmony_r08",
            "n_cells",
            "n_patients",
            "tumor_fraction",
            "predicted_doublet_fraction",
            "dominant_sample",
            "dominant_sample_fraction",
            "top_program",
            "top_z",
            "c3z_Cycling",
            "top10_markers"
        ]
    ]
)


print(
    "\n[7] CURRENT BOUNDARY"
)

print(
"""
No broad cell-type labels are frozen yet.

No entire cluster is classified as a doublet solely from Scrublet.

No malignant population is defined yet.

No Core50 score is used for cell identification.

The next decision will integrate:
    cluster marker genes,
    broad lineage programs,
    hepatic/epithelial identity,
    sample/patient distribution,
    and Scrublet enrichment.
"""
)


print(
    "\nSTEP31C3A: COMPLETE"
)

print(
    "NEXT: freeze broad cell types + final heterotypic-doublet handling."
)

print(
    "=" * 120
)

## Step31C3B — Targeted resolution of special clusters

Targets:

- Cluster 18 / 22: determine likely parent lineage of cycling cells.
- Cluster 21: distinguish mast-cell identity from platelet-like signal.
- Cluster 24: distinguish pDC identity from plasma/B-cell signal.
- Cluster 23: quantify myeloid–LSEC mixed-lineage structure and Scrublet overlap.

No malignant-cell identification is performed here.
No Core50 information is used.
No cluster is removed solely because of Scrublet.

In [ ]:
# =============================================================================
# Step31C3B Cell 97
# Targeted marker programs for special-cluster resolution
# =============================================================================

TARGETED_MARKERS = {

    "Mast": [
        "KIT",
        "TPSAB1",
        "TPSB2",
        "CPA3",
        "MS4A2",
        "HDC",
        "HPGDS"
    ],

    "Platelet_Core": [
        "PPBP",
        "PF4",
        "GP9",
        "ITGA2B",
        "TUBB1"
    ],

    "pDC": [
        "LILRA4",
        "IL3RA",
        "CLEC4C",
        "TCF4",
        "GZMB",
        "PLD4"
    ],

    "Plasma_Core": [
        "JCHAIN",
        "MZB1",
        "SDC1",
        "IGHG1"
    ],

    "Myeloid_Core": [
        "LST1",
        "TYROBP",
        "FCER1G",
        "AIF1",
        "CTSS",
        "LILRB1"
    ],

    "LSEC": [
        "CLEC4G",
        "FCN3",
        "OIT3",
        "CRHBP",
        "DNASE1L3",
        "PECAM1"
    ]
}


print(
    "Targeted programs:",
    len(TARGETED_MARKERS)
)

In [ ]:
# =============================================================================
# Step31C3B Cell 98
# Map targeted programs to common Ensembl universe
# =============================================================================

TARGETED_IDS = {}

targeted_rows = []


for program, markers in TARGETED_MARKERS.items():

    ids = []
    present = []
    missing = []
    ambiguous = []


    for gene in markers:

        hits = (
            canonical_symbol_map.loc[
                canonical_symbol_map[
                    "gene_symbol"
                ].astype(str)
                == gene,
                "gene_id"
            ]
            .astype(str)
            .unique()
            .tolist()
        )


        if len(hits) == 1:

            ids.append(
                hits[0]
            )

            present.append(
                gene
            )


        elif len(hits) == 0:

            missing.append(
                gene
            )


        else:

            ambiguous.append(
                gene
            )


    TARGETED_IDS[
        program
    ] = ids


    targeted_rows.append({

        "program":
            program,

        "n_requested":
            len(markers),

        "n_used":
            len(ids),

        "present":
            " | ".join(present),

        "missing":
            " | ".join(missing),

        "ambiguous":
            " | ".join(ambiguous)
    })


targeted_coverage = pd.DataFrame(
    targeted_rows
)


display(
    targeted_coverage
)


assert all(
    len(ids) > 0
    for ids in TARGETED_IDS.values()
)


print(
    "\nTARGETED MARKER MAPPING: PASS"
)

In [ ]:
# =============================================================================
# Step31C3B Cell 99
# Targeted scores and marker-detection counts
# =============================================================================

import scipy.sparse as sp
import numpy as np


def detected_gene_count(
    adata,
    gene_ids
):

    X = adata[
        :,
        gene_ids
    ].X

    if sp.issparse(X):

        return np.asarray(
            (
                X > 0
            ).sum(
                axis=1
            )
        ).ravel()

    else:

        return (
            X > 0
        ).sum(
            axis=1
        )


for program, gene_ids in TARGETED_IDS.items():

    X = adata_c2[
        :,
        gene_ids
    ].X


    mean_score = np.asarray(
        X.mean(
            axis=1
        )
    ).ravel()


    adata_c2.obs[
        f"target_{program}"
    ] = mean_score


    sd = mean_score.std()

    adata_c2.obs[
        f"target_z_{program}"
    ] = (
        (
            mean_score
            -
            mean_score.mean()
        )
        /
        sd
    )


    adata_c2.obs[
        f"target_n_{program}"
    ] = detected_gene_count(
        adata_c2,
        gene_ids
    )


print(
    "Targeted scores + detection counts: COMPLETE"
)

In [ ]:
# =============================================================================
# Step31C3B Cell 100
# Special cluster targeted-program audit
# =============================================================================

REFERENCE_SPECIAL = [
    "3",
    "9",
    "10",
    "13",
    "17",
    "21",
    "23",
    "24"
]


target_cols = []

for program in TARGETED_MARKERS:

    target_cols.extend([
        f"target_z_{program}",
        f"target_n_{program}"
    ])


special_program_summary = (
    adata_c2.obs.loc[
        adata_c2.obs[
            CLUSTER_KEY
        ]
        .astype(str)
        .isin(
            REFERENCE_SPECIAL
        )
    ]
    .groupby(
        CLUSTER_KEY,
        observed=True
    )[
        target_cols
    ]
    .mean()
    .reset_index()
)


display(
    special_program_summary
)

In [ ]:
# =============================================================================
# Step31C3B Cell 101
# Cluster 23 myeloid-LSEC mixed-lineage audit
# =============================================================================

cluster23_mask = (
    adata_c2.obs[
        CLUSTER_KEY
    ].astype(str)
    ==
    "23"
)


cluster23 = (
    adata_c2.obs.loc[
        cluster23_mask
    ].copy()
)


cluster23[
    "strong_myeloid"
] = (
    cluster23[
        "target_n_Myeloid_Core"
    ]
    >=
    2
)


cluster23[
    "strong_lsec"
] = (
    cluster23[
        "target_n_LSEC"
    ]
    >=
    2
)


cluster23[
    "myeloid_lsec_both"
] = (
    cluster23[
        "strong_myeloid"
    ]
    &
    cluster23[
        "strong_lsec"
    ]
)


cluster23[
    "scrublet_and_mixed"
] = (
    cluster23[
        "predicted_doublet"
    ].astype(bool)
    &
    cluster23[
        "myeloid_lsec_both"
    ]
)


cluster23_audit = pd.DataFrame({

    "metric": [
        "n_cells",
        "scrublet_positive",
        "strong_myeloid",
        "strong_lsec",
        "myeloid_lsec_both",
        "scrublet_and_mixed"
    ],

    "n": [
        len(cluster23),

        int(
            cluster23[
                "predicted_doublet"
            ].sum()
        ),

        int(
            cluster23[
                "strong_myeloid"
            ].sum()
        ),

        int(
            cluster23[
                "strong_lsec"
            ].sum()
        ),

        int(
            cluster23[
                "myeloid_lsec_both"
            ].sum()
        ),

        int(
            cluster23[
                "scrublet_and_mixed"
            ].sum()
        )
    ]
})


cluster23_audit[
    "fraction"
] = (
    cluster23_audit[
        "n"
    ]
    /
    len(cluster23)
)


display(
    cluster23_audit
)

In [ ]:
# =============================================================================
# Step31C3B Cell 102
# Clear broad-lineage reference map
# =============================================================================

CLEAR_CLUSTER_MAP = {

    "0": "T",
    "1": "T",
    "2": "NK",

    "3": "Myeloid",

    "4": "Fibroblast",

    "5": "Endothelial",

    "6": "T",
    "7": "NK",

    "8": "Hepatic",

    "9": "Myeloid",

    "10": "Endothelial",

    "11": "Myeloid",

    "12": "T",

    "13": "Myeloid",

    "14": "B",

    "15": "Endothelial",

    "16": "Fibroblast",

    "17": "Plasma",

    "19": "Epithelial",

    "20": "Hepatic",

    # Cluster25 is sample-specific,
    # but broad NK identity is strong.
    "25": "NK",

    # Cluster26 is sample-specific,
    # but broad hepatic identity is strong.
    "26": "Hepatic"
}


clear_labels = (
    adata_c2.obs[
        CLUSTER_KEY
    ]
    .astype(str)
    .map(
        CLEAR_CLUSTER_MAP
    )
)


print(
    "Reference cells:",
    int(
        clear_labels.notna().sum()
    )
)

print(
    "Unresolved cells:",
    int(
        clear_labels.isna().sum()
    )
)

In [ ]:
# =============================================================================
# Step31C3B Cell 103
# Neighbor lineage voting for clusters 18 and 22
# =============================================================================

import pandas as pd
import numpy as np


neighbors_meta = (
    adata_c2.uns[
        "neighbors_harmony_patient"
    ]
)


CONNECTIVITY_KEY = (
    neighbors_meta[
        "connectivities_key"
    ]
)


C = (
    adata_c2.obsp[
        CONNECTIVITY_KEY
    ]
    .tocsr()
)


cycling_mask = (
    adata_c2.obs[
        CLUSTER_KEY
    ]
    .astype(str)
    .isin(
        [
            "18",
            "22"
        ]
    )
)


cycling_indices = np.where(
    cycling_mask
)[0]


vote_rows = []


clear_labels_array = (
    clear_labels.to_numpy()
)


for i in cycling_indices:

    row = C.getrow(i)

    neigh_idx = row.indices
    weights = row.data


    labels = (
        clear_labels_array[
            neigh_idx
        ]
    )


    valid = pd.notna(
        labels
    )


    labels = labels[
        valid
    ]

    weights_valid = weights[
        valid
    ]


    if len(labels) == 0:

        vote_rows.append({

            "cell_id":
                adata_c2.obs_names[i],

            "cluster":
                str(
                    adata_c2.obs.iloc[i][
                        CLUSTER_KEY
                    ]
                ),

            "top_parent_lineage":
                "Unresolved",

            "top_vote_fraction":
                np.nan,

            "n_clear_neighbors":
                0
        })

        continue


    vote = (
        pd.DataFrame({
            "label":
                labels,

            "weight":
                weights_valid
        })
        .groupby(
            "label"
        )[
            "weight"
        ]
        .sum()
        .sort_values(
            ascending=False
        )
    )


    total_weight = (
        vote.sum()
    )


    top_label = (
        vote.index[0]
    )

    top_fraction = (
        vote.iloc[0]
        /
        total_weight
    )


    vote_rows.append({

        "cell_id":
            adata_c2.obs_names[i],

        "cluster":
            str(
                adata_c2.obs.iloc[i][
                    CLUSTER_KEY
                ]
            ),

        "top_parent_lineage":
            top_label,

        "top_vote_fraction":
            float(
                top_fraction
            ),

        "n_clear_neighbors":
            int(
                len(labels)
            )
    })


cycling_parent_votes = (
    pd.DataFrame(
        vote_rows
    )
)


print(
    "Cycling cells audited:",
    len(
        cycling_parent_votes
    )
)


display(
    cycling_parent_votes.head()
)

In [ ]:
# =============================================================================
# Step31C3B Cell 104
# Cycling parent-lineage summary
# =============================================================================

cycling_vote_summary = (
    cycling_parent_votes
    .groupby(
        [
            "cluster",
            "top_parent_lineage"
        ]
    )
    .agg(
        n_cells=(
            "cell_id",
            "size"
        ),

        median_vote_fraction=(
            "top_vote_fraction",
            "median"
        ),

        median_clear_neighbors=(
            "n_clear_neighbors",
            "median"
        )
    )
    .reset_index()
)


cycling_totals = (
    cycling_vote_summary
    .groupby(
        "cluster"
    )[
        "n_cells"
    ]
    .transform(
        "sum"
    )
)


cycling_vote_summary[
    "cell_fraction"
] = (
    cycling_vote_summary[
        "n_cells"
    ]
    /
    cycling_totals
)


display(
    cycling_vote_summary.sort_values(
        [
            "cluster",
            "cell_fraction"
        ],
        ascending=[
            True,
            False
        ]
    )
)

In [ ]:
# =============================================================================
# Step31C3B Cell 105
# Save targeted-resolution audit
# =============================================================================

targeted_coverage.to_csv(
    AUDIT_DIR /
    "Step31C3B_TargetedMarkerCoverage.csv",
    index=False
)


special_program_summary.to_csv(
    AUDIT_DIR /
    "Step31C3B_SpecialClusterPrograms.csv",
    index=False
)


cluster23_audit.to_csv(
    AUDIT_DIR /
    "Step31C3B_Cluster23_MixedLineageAudit.csv",
    index=False
)


cycling_parent_votes.to_csv(
    AUDIT_DIR /
    "Step31C3B_CyclingParentVotes.csv",
    index=False
)


cycling_vote_summary.to_csv(
    AUDIT_DIR /
    "Step31C3B_CyclingParentVoteSummary.csv",
    index=False
)


print(
    "Step31C3B audit outputs saved."
)

In [ ]:
# =============================================================================
# Step31C3B Cell 106
# REPORT — targeted special-cluster resolution
# =============================================================================

print(
    "=" * 120
)

print(
    "STEP31C3B — TARGETED SPECIAL-CLUSTER RESOLUTION"
)

print(
    "=" * 120
)


print(
    "\n[1] TARGETED MARKER COVERAGE"
)

display(
    targeted_coverage
)


print(
    "\n[2] SPECIAL-CLUSTER PROGRAMS"
)

display(
    special_program_summary
)


print(
    "\n[3] CLUSTER 23 — MYELOID / LSEC MIXED-LINEAGE AUDIT"
)

display(
    cluster23_audit
)


print(
    "\n[4] CYCLING CLUSTERS 18 / 22 — PARENT LINEAGE VOTES"
)

display(
    cycling_vote_summary.sort_values(
        [
            "cluster",
            "cell_fraction"
        ],
        ascending=[
            True,
            False
        ]
    )
)


print(
    "\n[5] CURRENT INTERPRETATION BOUNDARY"
)

print(
"""
Most broad cell identities are already well supported.

Clusters 18 and 22 remain cycling populations until their parent-lineage
structure is reviewed.

Cluster 21 is being tested specifically for mast-cell versus platelet identity.

Cluster 24 is being tested specifically for pDC versus plasma-cell identity.

Cluster 23 is not removed as a whole. Scrublet enrichment is interpreted
together with myeloid/LSEC marker coexistence.

No malignant population has been defined.

No Core50 information has been used.
"""
)


print(
    "\nSTEP31C3B: COMPLETE"
)

print(
    "NEXT: broad cell-type lock + conservative heterotypic-doublet exclusion."
)

print(
    "=" * 120
)

In [ ]:
# =============================================================================
# Step31C3C Cell 107
# Resolve parent lineage of cycling clusters using non-cycling programs
# =============================================================================

NONCYCLING_PROGRAMS = [
    "Hepatic_Mature",
    "Hepatic_Epithelial",
    "HCC_Associated_Audit",
    "T_Cell",
    "NK",
    "Myeloid",
    "B_Cell",
    "Plasma",
    "Endothelial",
    "Fibroblast",
    "Erythroid",
    "Platelet"
]

NONCYCLING_Z_COLS = [
    f"c3z_{x}"
    for x in NONCYCLING_PROGRAMS
]


cycling_parent_score = (
    adata_c2.obs.loc[
        adata_c2.obs[
            CLUSTER_KEY
        ].astype(str)
        .isin(
            ["18", "22"]
        )
    ]
    .groupby(
        CLUSTER_KEY,
        observed=True
    )[
        NONCYCLING_Z_COLS
    ]
    .mean()
    .reset_index()
)


cycling_parent_score[
    "top_noncycling_program"
] = (
    cycling_parent_score[
        NONCYCLING_Z_COLS
    ]
    .idxmax(
        axis=1
    )
    .str.replace(
        "c3z_",
        "",
        regex=False
    )
)


cycling_parent_score[
    "top_noncycling_z"
] = (
    cycling_parent_score[
        NONCYCLING_Z_COLS
    ]
    .max(
        axis=1
    )
)


display(
    cycling_parent_score
)


print("\nExpected:")
print("Cluster 18 -> Erythroid")
print("Cluster 22 -> Myeloid")

In [ ]:
# =============================================================================
# Step31C3C Cell 108
# Verify cycling-parent assignments
# =============================================================================

cycling_parent_dict = dict(
    zip(
        cycling_parent_score[
            CLUSTER_KEY
        ].astype(str),
        cycling_parent_score[
            "top_noncycling_program"
        ]
    )
)


print(
    "Cluster 18 parent:",
    cycling_parent_dict.get("18")
)

print(
    "Cluster 22 parent:",
    cycling_parent_dict.get("22")
)


assert (
    cycling_parent_dict.get("18")
    ==
    "Erythroid"
), (
    "Cluster 18 parent lineage differs from expected Erythroid. "
    "Stop and review before freezing."
)


assert (
    cycling_parent_dict.get("22")
    ==
    "Myeloid"
), (
    "Cluster 22 parent lineage differs from expected Myeloid. "
    "Stop and review before freezing."
)


print(
    "\nCYCLING PARENT LINEAGES: PASS"
)

In [ ]:
# =============================================================================
# Step31C3C Cell 109
# Freeze broad cluster annotation
# =============================================================================

BROAD_CLUSTER_MAP = {

    # T cells
    "0":  "T",
    "1":  "T",
    "6":  "T",
    "12": "T",

    # NK
    "2":  "NK",
    "7":  "NK",
    "25": "NK",

    # Myeloid
    "3":  "Myeloid",
    "9":  "Myeloid",
    "11": "Myeloid",
    "13": "Myeloid",

    # Stromal
    "4":  "Fibroblast_Stellate",
    "16": "Fibroblast_Stellate",

    # Endothelial
    "5":  "Endothelial",
    "10": "Endothelial_LSEC",
    "15": "Endothelial",

    # Hepatic / epithelial
    "8":  "Hepatic",
    "20": "Hepatic",
    "26": "Hepatic",
    "19": "Epithelial_Cholangiocyte",

    # B / plasma
    "14": "B",
    "17": "Plasma",

    # Cycling populations
    "18": "Cycling_Erythroid",
    "22": "Cycling_Myeloid",

    # Special immune populations
    "21": "Mast",
    "24": "pDC",

    # Mixed-lineage cluster
    "23": "Ambiguous_Myeloid_LSEC"
}


cluster_ids = set(
    adata_c2.obs[
        CLUSTER_KEY
    ]
    .astype(str)
    .unique()
)


assert (
    cluster_ids
    ==
    set(
        BROAD_CLUSTER_MAP.keys()
    )
), (
    "Not all 27 clusters are represented in BROAD_CLUSTER_MAP."
)


adata_c2.obs[
    "broad_cell_type"
] = (
    adata_c2.obs[
        CLUSTER_KEY
    ]
    .astype(str)
    .map(
        BROAD_CLUSTER_MAP
    )
)


assert (
    adata_c2.obs[
        "broad_cell_type"
    ].notna()
    .all()
)


print(
    "Broad labels assigned:",
    adata_c2.obs[
        "broad_cell_type"
    ].nunique()
)


display(
    adata_c2.obs[
        "broad_cell_type"
    ]
    .value_counts()
)

In [ ]:
# =============================================================================
# Step31C3C Cell 110
# Flag sample-specific rare clusters
# =============================================================================

adata_c2.obs[
    "sample_specific_cluster"
] = (
    adata_c2.obs[
        CLUSTER_KEY
    ]
    .astype(str)
    .isin(
        ["25", "26"]
    )
)


print(
    "Cells in sample-specific rare clusters:",
    int(
        adata_c2.obs[
            "sample_specific_cluster"
        ].sum()
    )
)


display(
    adata_c2.obs.loc[
        adata_c2.obs[
            "sample_specific_cluster"
        ],
        [
            "sample_id",
            CLUSTER_KEY,
            "broad_cell_type"
        ]
    ]
    .value_counts()
)

In [ ]:
# =============================================================================
# Step31C3C Cell 111
# Conservative high-confidence heterotypic-doublet definition
# =============================================================================

adata_c2.obs[
    "high_conf_heterotypic_doublet"
] = (
    adata_c2.obs[
        "predicted_doublet"
    ].astype(bool)
    &
    (
        adata_c2.obs[
            CLUSTER_KEY
        ].astype(str)
        ==
        "23"
    )
    &
    (
        adata_c2.obs[
            "target_n_Myeloid_Core"
        ]
        >=
        2
    )
    &
    (
        adata_c2.obs[
            "target_n_LSEC"
        ]
        >=
        2
    )
)


print(
    "Scrublet-positive cells:",
    int(
        adata_c2.obs[
            "predicted_doublet"
        ].sum()
    )
)


print(
    "High-confidence heterotypic doublets:",
    int(
        adata_c2.obs[
            "high_conf_heterotypic_doublet"
        ].sum()
    )
)


print(
    "Scrublet-positive but retained:",
    int(
        (
            adata_c2.obs[
                "predicted_doublet"
            ]
            &
            ~adata_c2.obs[
                "high_conf_heterotypic_doublet"
            ]
        ).sum()
    )
)


assert (
    int(
        adata_c2.obs[
            "high_conf_heterotypic_doublet"
        ].sum()
    )
    ==
    11
)

In [ ]:
# =============================================================================
# Step31C3C Cell 112 — FIXED
# Build final broad-cell universe
# =============================================================================

keep_mask = (
    ~adata_c2.obs[
        "high_conf_heterotypic_doublet"
    ].astype(bool)
)


print(
    "Cells before exclusion:",
    adata_c2.n_obs
)

print(
    "Cells marked for exclusion:",
    int(
        (
            ~keep_mask
        ).sum()
    )
)


adata_broad = (
    adata_c2[
        keep_mask
    ]
    .copy()
)


print(
    "\nBefore final heterotypic-doublet exclusion:",
    adata_c2.n_obs
)

print(
    "Removed high-confidence heterotypic doublets:",
    adata_c2.n_obs
    -
    adata_broad.n_obs
)

print(
    "Final broad-cell universe:",
    adata_broad.n_obs
)

print(
    "Genes:",
    adata_broad.n_vars
)


assert (
    adata_c2.n_obs
    -
    adata_broad.n_obs
    ==
    int(
        adata_c2.obs[
            "high_conf_heterotypic_doublet"
        ].sum()
    )
)


assert (
    adata_broad.n_obs
    ==
    90980
), (
    f"Unexpected final cell number: {adata_broad.n_obs}"
)


assert (
    adata_broad.n_vars
    ==
    32732
)


print(
    "\nFINAL BROAD-CELL UNIVERSE: PASS"
)

In [ ]:
# =============================================================================
# Step31C3C Cell 113
# Final cluster annotation table
# =============================================================================

broad_cluster_lock = (
    adata_broad.obs
    .groupby(
        [
            CLUSTER_KEY,
            "broad_cell_type"
        ],
        observed=True
    )
    .agg(
        n_cells=(
            "sample_id",
            "size"
        ),

        n_samples=(
            "sample_id",
            "nunique"
        ),

        n_patients=(
            "patient_id",
            "nunique"
        ),

        tumor_fraction=(
            "tissue",
            lambda x:
                (
                    x.astype(str)
                    ==
                    "Tumor"
                ).mean()
        ),

        remaining_scrublet_fraction=(
            "predicted_doublet",
            "mean"
        )
    )
    .reset_index()
)


display(
    broad_cluster_lock
)

In [ ]:
# =============================================================================
# Step31C3C Cell 114
# Broad-cell composition
# =============================================================================

broad_cell_counts = (
    adata_broad.obs[
        "broad_cell_type"
    ]
    .value_counts()
    .rename_axis(
        "broad_cell_type"
    )
    .reset_index(
        name="n_cells"
    )
)


broad_cell_counts[
    "fraction"
] = (
    broad_cell_counts[
        "n_cells"
    ]
    /
    adata_broad.n_obs
)


display(
    broad_cell_counts
)

In [ ]:
# =============================================================================
# Step31C3C Cell 115
# Save broad-cell-type lock
# =============================================================================

BROAD_LOCK_PATH = (
    DATA_DIR /
    "GSE326201_STEP31C3C_BROAD_CELLTYPE_LOCK.h5ad"
)


broad_cluster_lock.to_csv(
    AUDIT_DIR /
    "Step31C3C_BroadClusterAnnotationLock.csv",
    index=False
)


broad_cell_counts.to_csv(
    AUDIT_DIR /
    "Step31C3C_BroadCellTypeCounts.csv",
    index=False
)


adata_c2.obs[
    [
        "sample_id",
        "patient_id",
        "tissue",
        CLUSTER_KEY,
        "broad_cell_type",
        "predicted_doublet",
        "doublet_score",
        "high_conf_heterotypic_doublet",
        "sample_specific_cluster"
    ]
].to_csv(
    AUDIT_DIR /
    "Step31C3C_CellLevelAnnotationAndDoubletAudit.csv"
)


adata_broad.write_h5ad(
    BROAD_LOCK_PATH,
    compression="gzip"
)


print(
    "Saved:"
)

print(
    BROAD_LOCK_PATH
)

print(
    "Exists:",
    BROAD_LOCK_PATH.exists()
)

In [ ]:
# =============================================================================
# Step31C3C Cell 116
# FINAL REPORT — Broad cell-type lock
# =============================================================================

print("=" * 120)

print(
    "STEP31C3C — FINAL BROAD CELL-TYPE AND DOUBLET LOCK"
)

print("=" * 120)


print(
    "\n[1] INPUT"
)

print(
    "QC-pass cells:",
    adata_c2.n_obs
)

print(
    "Genes:",
    adata_c2.n_vars
)


print(
    "\n[2] CYCLING PARENT LINEAGES"
)

display(
    cycling_parent_score[
        [
            CLUSTER_KEY,
            "top_noncycling_program",
            "top_noncycling_z"
        ]
    ]
)


print(
    "\n[3] SPECIAL CLUSTER RESOLUTION"
)

print(
    "Cluster 21 -> Mast"
)

print(
    "Cluster 24 -> pDC"
)

print(
    "Cluster 23 -> Ambiguous_Myeloid_LSEC"
)

print(
    "Cluster 18 -> Cycling_Erythroid"
)

print(
    "Cluster 22 -> Cycling_Myeloid"
)


print(
    "\n[4] DOUBLET POLICY"
)

print(
    "Scrublet-positive:",
    int(
        adata_c2.obs[
            "predicted_doublet"
        ].sum()
    )
)

print(
    "High-confidence heterotypic doublets removed:",
    int(
        adata_c2.obs[
            "high_conf_heterotypic_doublet"
        ].sum()
    )
)

print(
    "Scrublet-positive but retained:",
    int(
        (
            adata_c2.obs[
                "predicted_doublet"
            ]
            &
            ~adata_c2.obs[
                "high_conf_heterotypic_doublet"
            ]
        ).sum()
    )
)


print(
    "\n[5] FINAL BROAD-CELL UNIVERSE"
)

print(
    "Cells:",
    adata_broad.n_obs
)

print(
    "Genes:",
    adata_broad.n_vars
)

print(
    "Patients:",
    adata_broad.obs[
        "patient_id"
    ].nunique()
)

print(
    "Samples:",
    adata_broad.obs[
        "sample_id"
    ].nunique()
)


print(
    "\n[6] BROAD CELL-TYPE COUNTS"
)

display(
    broad_cell_counts
)


print(
    "\n[7] CLUSTER LOCK"
)

display(
    broad_cluster_lock
)


print(
    "\n[8] SPECIAL CAUTION FLAGS"
)

print(
    "Sample-specific clusters:",
    ["25", "26"]
)

print(
    "Cluster 23 retained label:",
    "Ambiguous_Myeloid_LSEC"
)

print(
    "Cluster 23 is excluded from lineage-specific biological claims."
)


print(
    "\n[9] CORE50 STATUS"
)

print(
    "Core50 used for annotation:",
    False
)

print(
    "Frozen biological coverage:",
    "50/50"
)


print(
    "\n[10] CLAIM BOUNDARY"
)

print(
"""
Broad cell identities are now frozen for Step31.

Only 11 cells meeting BOTH Scrublet and incompatible Myeloid/LSEC
mixed-lineage evidence were removed as high-confidence heterotypic doublets.

The remaining Scrublet-positive cells are retained but remain flagged.

Cycling clusters were assigned parent lineages using non-cycling lineage
programs rather than proliferation markers alone.

No malignant-cell identity has yet been assigned.

No Core50 score has been used to define any cell population.
"""
)


print(
    "\n[11] FILE"
)

print(
    BROAD_LOCK_PATH
)

print(
    "Exists:",
    BROAD_LOCK_PATH.exists()
)


print(
    "\nSTEP31C3C: FROZEN"
)

print(
    "NEXT: Step31D — independent malignant-cell identification."
)

print("=" * 120)

## Step31D1 — Independent malignant-cell identification setup

### Objective

Establish the candidate epithelial/hepatic compartments and independent
normal reference populations before malignant-cell classification.

### Principles

- Broad cell identities are inherited unchanged from Step31C3C.
- Core50 is NOT used.
- MS3/MS6 programs are NOT used.
- Step29 regulators are NOT used.
- Tumor origin alone is NOT considered proof of malignancy.
- NonTumor hepatic cells serve as the primary normal-hepatocyte reference.
- Epithelial/Cholangiocyte cells are evaluated separately from the primary
  hepatic compartment.
- CNV-like evidence will be assessed only after reference adequacy and
  gene-coordinate feasibility are confirmed.

In [ ]:
# =============================================================================
# Step31D1 Cell 118
# Reload frozen broad-cell universe
# =============================================================================

import scanpy as sc
import pandas as pd
import numpy as np
from pathlib import Path


BROAD_LOCK_PATH = (
    DATA_DIR /
    "GSE326201_STEP31C3C_BROAD_CELLTYPE_LOCK.h5ad"
)


adata_d = sc.read_h5ad(
    BROAD_LOCK_PATH
)


print(adata_d)

print(
    "\nShape:",
    adata_d.shape
)


assert adata_d.n_obs == 90980
assert adata_d.n_vars == 32732

assert (
    adata_d.obs[
        "broad_cell_type"
    ].notna()
    .all()
)

assert (
    "high_conf_heterotypic_doublet"
    in
    adata_d.obs.columns
)


print(
    "\nSTEP31D INPUT: PASS"
)

In [ ]:
# =============================================================================
# Step31D1 Cell 119
# Define independent malignant-candidate compartments
# =============================================================================

PRIMARY_COMPARTMENT = (
    "Hepatic"
)

SECONDARY_COMPARTMENT = (
    "Epithelial_Cholangiocyte"
)


adata_d.obs[
    "malignant_candidate_compartment"
] = "Other"


adata_d.obs.loc[
    adata_d.obs[
        "broad_cell_type"
    ]
    ==
    PRIMARY_COMPARTMENT,
    "malignant_candidate_compartment"
] = "Hepatic"


adata_d.obs.loc[
    adata_d.obs[
        "broad_cell_type"
    ]
    ==
    SECONDARY_COMPARTMENT,
    "malignant_candidate_compartment"
] = "Epithelial_Cholangiocyte"


print(
    adata_d.obs[
        "malignant_candidate_compartment"
    ]
    .value_counts()
)

In [ ]:
# =============================================================================
# Step31D1 Cell 120
# Hepatic compartment by patient and tissue
# =============================================================================

hepatic = (
    adata_d.obs.loc[
        adata_d.obs[
            "broad_cell_type"
        ]
        ==
        "Hepatic"
    ]
    .copy()
)


hepatic_patient_counts = (
    hepatic
    .groupby(
        [
            "patient_id",
            "tissue"
        ],
        observed=True
    )
    .size()
    .rename(
        "n_cells"
    )
    .reset_index()
)


hepatic_patient_pivot = (
    hepatic_patient_counts
    .pivot(
        index="patient_id",
        columns="tissue",
        values="n_cells"
    )
    .fillna(0)
    .astype(int)
    .reset_index()
)


for col in [
    "Tumor",
    "NonTumor"
]:

    if col not in hepatic_patient_pivot.columns:

        hepatic_patient_pivot[
            col
        ] = 0


hepatic_patient_pivot[
    "total"
] = (
    hepatic_patient_pivot[
        "Tumor"
    ]
    +
    hepatic_patient_pivot[
        "NonTumor"
    ]
)


hepatic_patient_pivot[
    "tumor_fraction"
] = (
    hepatic_patient_pivot[
        "Tumor"
    ]
    /
    hepatic_patient_pivot[
        "total"
    ]
)


display(
    hepatic_patient_pivot
)

In [ ]:
# =============================================================================
# Step31D1 Cell 121
# Hepatic-cluster composition
# =============================================================================

hepatic_cluster_composition = (
    adata_d.obs.loc[
        adata_d.obs[
            "broad_cell_type"
        ]
        ==
        "Hepatic"
    ]
    .groupby(
        [
            "leiden_harmony_r08",
            "tissue",
            "patient_id"
        ],
        observed=True
    )
    .size()
    .rename(
        "n_cells"
    )
    .reset_index()
)


display(
    hepatic_cluster_composition
)

In [ ]:
# =============================================================================
# Step31D1 Cell 122
# Define normal hepatic reference and tumor hepatic candidates
# =============================================================================

normal_hepatic_mask = (
    (
        adata_d.obs[
            "broad_cell_type"
        ]
        ==
        "Hepatic"
    )
    &
    (
        adata_d.obs[
            "tissue"
        ].astype(str)
        ==
        "NonTumor"
    )
)


tumor_hepatic_mask = (
    (
        adata_d.obs[
            "broad_cell_type"
        ]
        ==
        "Hepatic"
    )
    &
    (
        adata_d.obs[
            "tissue"
        ].astype(str)
        ==
        "Tumor"
    )
)


print(
    "Normal hepatic reference:",
    int(
        normal_hepatic_mask.sum()
    )
)


print(
    "Tumor-derived hepatic candidates:",
    int(
        tumor_hepatic_mask.sum()
    )
)


print(
    "\nNormal hepatic reference by patient:"
)

display(
    adata_d.obs.loc[
        normal_hepatic_mask
    ][
        "patient_id"
    ]
    .value_counts()
    .sort_index()
)

In [ ]:
# =============================================================================
# Step31D1 Cell 123
# Normal-hepatocyte reference composition audit
# =============================================================================

normal_hepatic_reference_audit = (
    adata_d.obs.loc[
        normal_hepatic_mask
    ]
    .groupby(
        [
            "patient_id",
            "sample_id",
            "leiden_harmony_r08"
        ],
        observed=True
    )
    .size()
    .rename(
        "n_cells"
    )
    .reset_index()
)


normal_hepatic_reference_audit[
    "fraction_of_reference"
] = (
    normal_hepatic_reference_audit[
        "n_cells"
    ]
    /
    normal_hepatic_mask.sum()
)


display(
    normal_hepatic_reference_audit
    .sort_values(
        "n_cells",
        ascending=False
    )
)


print(
    "\nNumber of patients contributing normal hepatic cells:",
    adata_d.obs.loc[
        normal_hepatic_mask,
        "patient_id"
    ].nunique()
)


print(
    "Number of NonTumor samples contributing normal hepatic cells:",
    adata_d.obs.loc[
        normal_hepatic_mask,
        "sample_id"
    ].nunique()
)

In [ ]:
# =============================================================================
# Step31D1 Cell 124
# Paired hepatic-reference adequacy
# =============================================================================

MIN_REFERENCE_CELLS = 20


hepatic_patient_pivot[
    "has_tumor_hepatic"
] = (
    hepatic_patient_pivot[
        "Tumor"
    ] > 0
)


hepatic_patient_pivot[
    "has_normal_hepatic"
] = (
    hepatic_patient_pivot[
        "NonTumor"
    ] > 0
)


hepatic_patient_pivot[
    "adequate_normal_reference"
] = (
    hepatic_patient_pivot[
        "NonTumor"
    ]
    >=
    MIN_REFERENCE_CELLS
)


hepatic_patient_pivot[
    "paired_adequate"
] = (
    hepatic_patient_pivot[
        "has_tumor_hepatic"
    ]
    &
    hepatic_patient_pivot[
        "adequate_normal_reference"
    ]
)


display(
    hepatic_patient_pivot
)


print(
    "\nPatients with Tumor hepatic cells:",
    int(
        hepatic_patient_pivot[
            "has_tumor_hepatic"
        ].sum()
    )
)


print(
    "Patients with >=20 NonTumor hepatic reference cells:",
    int(
        hepatic_patient_pivot[
            "adequate_normal_reference"
        ].sum()
    )
)


print(
    "Paired patients with adequate hepatic reference:",
    int(
        hepatic_patient_pivot[
            "paired_adequate"
        ].sum()
    )
)

In [ ]:
# =============================================================================
# Step31D1 Cell 125
# Secondary epithelial / cholangiocyte compartment audit
# =============================================================================

epi_mask = (
    adata_d.obs[
        "broad_cell_type"
    ]
    ==
    "Epithelial_Cholangiocyte"
)


epi_patient_counts = (
    adata_d.obs.loc[
        epi_mask
    ]
    .groupby(
        [
            "patient_id",
            "tissue"
        ],
        observed=True
    )
    .size()
    .rename(
        "n_cells"
    )
    .reset_index()
)


epi_patient_pivot = (
    epi_patient_counts
    .pivot(
        index="patient_id",
        columns="tissue",
        values="n_cells"
    )
    .fillna(0)
    .astype(int)
    .reset_index()
)


for col in [
    "Tumor",
    "NonTumor"
]:

    if col not in epi_patient_pivot.columns:

        epi_patient_pivot[
            col
        ] = 0


epi_patient_pivot[
    "total"
] = (
    epi_patient_pivot[
        "Tumor"
    ]
    +
    epi_patient_pivot[
        "NonTumor"
    ]
)


epi_patient_pivot[
    "tumor_fraction"
] = (
    epi_patient_pivot[
        "Tumor"
    ]
    /
    epi_patient_pivot[
        "total"
    ]
)


display(
    epi_patient_pivot
)


print(
    "\nTotal Epithelial/Cholangiocyte cells:",
    int(
        epi_mask.sum()
    )
)

print(
    "Tumor:",
    int(
        (
            epi_mask
            &
            (
                adata_d.obs[
                    "tissue"
                ].astype(str)
                ==
                "Tumor"
            )
        ).sum()
    )
)

print(
    "NonTumor:",
    int(
        (
            epi_mask
            &
            (
                adata_d.obs[
                    "tissue"
                ].astype(str)
                ==
                "NonTumor"
            )
        ).sum()
    )
)

In [ ]:
# =============================================================================
# Step31D1 Cell 126
# Build epithelial/hepatic candidate universe
# =============================================================================

candidate_mask = (
    adata_d.obs[
        "broad_cell_type"
    ]
    .isin(
        [
            "Hepatic",
            "Epithelial_Cholangiocyte"
        ]
    )
)


adata_candidate = (
    adata_d[
        candidate_mask
    ]
    .copy()
)


print(
    "Candidate cells:",
    adata_candidate.n_obs
)

print(
    "Genes:",
    adata_candidate.n_vars
)


display(
    adata_candidate.obs[
        [
            "broad_cell_type",
            "tissue"
        ]
    ]
    .value_counts()
)

In [ ]:
# =============================================================================
# Step31D1 Cell 127
# CNV-like inference feasibility audit
# =============================================================================

import importlib.util


print(
    "adata.var columns:"
)

print(
    list(
        adata_candidate.var.columns
    )
)


coordinate_candidates = {

    "chromosome": [
        "chromosome",
        "chr",
        "chrom"
    ],

    "start": [
        "start",
        "gene_start",
        "start_position"
    ],

    "end": [
        "end",
        "gene_end",
        "end_position"
    ]
}


coordinate_status = {}


for field, options in (
    coordinate_candidates.items()
):

    found = [
        x
        for x in options
        if x in adata_candidate.var.columns
    ]

    coordinate_status[
        field
    ] = found


print(
    "\nCoordinate columns:"
)

print(
    coordinate_status
)


print(
    "\nOptional packages:"
)

print(
    "infercnvpy:",
    importlib.util.find_spec(
        "infercnvpy"
    )
    is not None
)

print(
    "mygene:",
    importlib.util.find_spec(
        "mygene"
    )
    is not None
)


has_coordinates = all(
    len(
        coordinate_status[x]
    ) > 0
    for x in [
        "chromosome",
        "start",
        "end"
    ]
)


print(
    "\nComplete genomic coordinates already available:",
    has_coordinates
)

In [ ]:
# =============================================================================
# Step31D1 Cell 128
# Normal-hepatic reference QC audit
# =============================================================================

normal_hepatic_qc_cols = [
    x
    for x in [
        "total_counts",
        "n_genes_by_counts",
        "pct_counts_mt"
    ]
    if x in adata_d.obs.columns
]


if len(
    normal_hepatic_qc_cols
) > 0:

    normal_hepatic_qc = (
        adata_d.obs.loc[
            normal_hepatic_mask
        ]
        .groupby(
            "sample_id",
            observed=True
        )[
            normal_hepatic_qc_cols
        ]
        .median()
        .reset_index()
    )

    display(
        normal_hepatic_qc
    )

else:

    normal_hepatic_qc = (
        pd.DataFrame()
    )

    print(
        "QC columns not retained in current H5AD; "
        "reference adequacy will rely on the frozen upstream QC."
    )

In [ ]:
# =============================================================================
# Step31D1 Cell 129
# Save malignant-identification setup audit
# =============================================================================

hepatic_patient_pivot.to_csv(
    AUDIT_DIR /
    "Step31D1_HepaticPatientTissueAudit.csv",
    index=False
)


hepatic_cluster_composition.to_csv(
    AUDIT_DIR /
    "Step31D1_HepaticClusterComposition.csv",
    index=False
)


normal_hepatic_reference_audit.to_csv(
    AUDIT_DIR /
    "Step31D1_NormalHepaticReferenceAudit.csv",
    index=False
)


epi_patient_pivot.to_csv(
    AUDIT_DIR /
    "Step31D1_EpithelialPatientTissueAudit.csv",
    index=False
)


candidate_summary = (
    adata_candidate.obs[
        [
            "broad_cell_type",
            "tissue",
            "patient_id",
            "sample_id",
            "leiden_harmony_r08"
        ]
    ]
    .copy()
)


candidate_summary.to_csv(
    AUDIT_DIR /
    "Step31D1_CandidateCellMetadata.csv"
)


print(
    "Step31D1 audit outputs saved."
)

In [ ]:
# =============================================================================
# Step31D1 Cell 130
# REPORT — malignant-cell identification setup
# =============================================================================

print("=" * 120)

print(
    "STEP31D1 — INDEPENDENT MALIGNANT-CELL IDENTIFICATION SETUP"
)

print("=" * 120)


print(
    "\n[1] FROZEN INPUT"
)

print(
    "Cells:",
    adata_d.n_obs
)

print(
    "Genes:",
    adata_d.n_vars
)

print(
    "Patients:",
    adata_d.obs[
        "patient_id"
    ].nunique()
)

print(
    "Samples:",
    adata_d.obs[
        "sample_id"
    ].nunique()
)


print(
    "\n[2] PRIMARY HEPATIC COMPARTMENT"
)

print(
    "Total Hepatic cells:",
    int(
        (
            adata_d.obs[
                "broad_cell_type"
            ]
            ==
            "Hepatic"
        ).sum()
    )
)


print(
    "Tumor-derived Hepatic:",
    int(
        tumor_hepatic_mask.sum()
    )
)

print(
    "NonTumor Hepatic reference:",
    int(
        normal_hepatic_mask.sum()
    )
)


display(
    hepatic_patient_pivot
)


print(
    "\n[3] NORMAL HEPATIC REFERENCE"
)

print(
    "Reference patients:",
    adata_d.obs.loc[
        normal_hepatic_mask,
        "patient_id"
    ].nunique()
)

print(
    "Reference samples:",
    adata_d.obs.loc[
        normal_hepatic_mask,
        "sample_id"
    ].nunique()
)


display(
    normal_hepatic_reference_audit
    .sort_values(
        "n_cells",
        ascending=False
    )
)


print(
    "\n[4] SECONDARY EPITHELIAL / CHOLANGIOCYTE COMPARTMENT"
)

print(
    "Total:",
    int(
        epi_mask.sum()
    )
)


display(
    epi_patient_pivot
)


print(
    "\n[5] TOTAL MALIGNANT-CANDIDATE UNIVERSE"
)

print(
    "Candidate cells:",
    adata_candidate.n_obs
)

display(
    adata_candidate.obs[
        [
            "broad_cell_type",
            "tissue"
        ]
    ]
    .value_counts()
)


print(
    "\n[6] CNV FEASIBILITY"
)

print(
    "Genomic-coordinate columns already available:",
    has_coordinates
)

print(
    "infercnvpy installed:",
    importlib.util.find_spec(
        "infercnvpy"
    )
    is not None
)

print(
    "mygene installed:",
    importlib.util.find_spec(
        "mygene"
    )
    is not None
)


print(
    "\n[7] ANALYTIC BOUNDARY"
)

print(
"""
No malignant-cell label has been assigned.

Tumor-derived Hepatic cells are candidates, not automatically malignant.

NonTumor Hepatic cells are reserved as the primary normal-hepatocyte
reference.

Epithelial/Cholangiocyte cells remain a separate secondary compartment.

Core50, MS3/MS6 programs, Original RiskScore and Step29 regulators have
not been used in malignant-cell identification.

The next step will determine the independent malignancy evidence strategy
from the observed reference adequacy and genomic-coordinate feasibility.
"""
)


print(
    "\nSTEP31D1: COMPLETE"
)

print(
    "NEXT: Step31D2 — independent malignancy evidence / CNV-like analysis."
)

print("=" * 120)

## Step31D2A — Hepatic CNV-like malignancy evidence

### Primary design

- Analyze the frozen Hepatic compartment only.
- Tumor-derived Hepatic cells are malignant candidates, not predefined malignant cells.
- NonTumor Hepatic cells form the pooled normal-hepatocyte reference.
- Gene expression is taken from the uncorrected log-normalized matrix.
- Harmony coordinates are not used to calculate CNV-like signal.
- Genes are ordered by autosomal genomic coordinates using Ensembl gene IDs.
- CNV-like signal is estimated from contiguous genomic expression bins.
- A normal-reference-derived threshold is used only as supporting malignancy evidence.
- Core50, MS3/MS6, Original RiskScore and Step29 regulators remain completely excluded.

### Secondary design

The Epithelial/Cholangiocyte compartment will be analyzed separately using
its own NonTumor epithelial reference and will not be compared directly
against the Hepatic reference.

In [ ]:
# =============================================================================
# Step31D2A Cell 132
# Expression-state audit
# =============================================================================

print(
    "Object:",
    adata_d
)

print(
    "\nX shape:",
    adata_d.X.shape
)

print(
    "\nlog1p metadata:"
)

print(
    adata_d.uns.get(
        "log1p",
        "NOT FOUND"
    )
)

print(
    "\nRaw-count source:"
)

print(
    adata_d.uns.get(
        "raw_counts_source",
        "NOT FOUND"
    )
)


assert adata_d.n_obs == 90980
assert adata_d.n_vars == 32732


print(
    "\nCNV-like analysis will use:"
)

print(
    "uncorrected log-normalized X"
)

print(
    "Harmony expression used:",
    False
)

print(
    "\nEXPRESSION STATE AUDIT: PASS"
)

In [ ]:
# =============================================================================
# Step31D2A Cell 133
# Ensure Ensembl-coordinate query dependency
# =============================================================================

import importlib.util
import subprocess
import sys


if importlib.util.find_spec(
    "mygene"
) is None:

    print(
        "Installing mygene..."
    )

    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "mygene"
    ])


import mygene


print(
    "mygene:",
    mygene.__version__
)

print(
    "GENOMIC-COORDINATE QUERY DEPENDENCY: PASS"
)

In [ ]:
from hcc_release_paths import project_path as _hcc_path
# =============================================================================
# Step31D2A Cell 133A
# Restore frozen Step31 directory paths
# =============================================================================

from pathlib import Path


STEP31_DIR = Path(
    _hcc_path('LIHC_scRNA_Step31_Independent_Validation')
)


DATA_DIR = (
    STEP31_DIR /
    "Data"
)

TABLES_DIR = (
    STEP31_DIR /
    "Tables"
)

FIGURES_DIR = (
    STEP31_DIR /
    "Figures"
)

AUDIT_DIR = (
    STEP31_DIR /
    "Audit"
)

RESOURCES_DIR = (
    STEP31_DIR /
    "Resources"
)


# Ensure directories exist
for p in [
    DATA_DIR,
    TABLES_DIR,
    FIGURES_DIR,
    AUDIT_DIR,
    RESOURCES_DIR
]:
    p.mkdir(
        parents=True,
        exist_ok=True
    )


print("STEP31_DIR:")
print(STEP31_DIR)

print("\nDATA_DIR:")
print(DATA_DIR)

print("\nTABLES_DIR:")
print(TABLES_DIR)

print("\nFIGURES_DIR:")
print(FIGURES_DIR)

print("\nAUDIT_DIR:")
print(AUDIT_DIR)

print("\nRESOURCES_DIR:")
print(RESOURCES_DIR)


assert STEP31_DIR.exists()
assert DATA_DIR.exists()
assert AUDIT_DIR.exists()
assert RESOURCES_DIR.exists()


print(
    "\nSTEP31 DIRECTORY STATE: PASS"
)

In [ ]:
# =============================================================================
# Step31D2A Cell 134
# Build / load Ensembl genomic-coordinate map
# =============================================================================

import time
import pandas as pd
import numpy as np


GENE_COORD_PATH = (
    RESOURCES_DIR /
    "Step31_Ensembl_GRCh38_GeneCoordinates.csv"
)


AUTOSOMES = [
    str(x)
    for x in range(
        1,
        23
    )
]


if GENE_COORD_PATH.exists():

    print(
        "Loading cached coordinate map:"
    )

    print(
        GENE_COORD_PATH
    )

    gene_coordinates = pd.read_csv(
        GENE_COORD_PATH,
        dtype={
            "gene_id": str,
            "chromosome": str
        }
    )


else:

    print(
        "No cached coordinate map found."
    )

    print(
        "Querying Ensembl IDs through MyGene..."
    )


    mg = mygene.MyGeneInfo()


    gene_ids_to_query = (
        adata_d.var_names
        .astype(str)
        .tolist()
    )


    query_results = []


    BATCH_SIZE = 1000


    for start in range(
        0,
        len(gene_ids_to_query),
        BATCH_SIZE
    ):

        batch = (
            gene_ids_to_query[
                start:
                start + BATCH_SIZE
            ]
        )


        print(
            f"{start + 1}"
            f"-"
            f"{min(start + BATCH_SIZE, len(gene_ids_to_query))}"
            f" / "
            f"{len(gene_ids_to_query)}"
        )


        result = mg.querymany(
            batch,
            scopes="ensembl.gene",
            fields=[
                "symbol",
                "genomic_pos"
            ],
            species="human",
            as_dataframe=False,
            verbose=False
        )


        query_results.extend(
            result
        )


        time.sleep(
            0.05
        )


    coordinate_rows = []


    for r in query_results:

        gene_id = str(
            r.get(
                "query",
                ""
            )
        )


        gp = r.get(
            "genomic_pos",
            None
        )


        if gp is None:

            coordinate_rows.append({

                "gene_id":
                    gene_id,

                "symbol":
                    r.get(
                        "symbol",
                        None
                    ),

                "chromosome":
                    None,

                "start":
                    np.nan,

                "end":
                    np.nan,

                "mapping_status":
                    "missing"
            })

            continue


        if isinstance(
            gp,
            dict
        ):

            gp_list = [
                gp
            ]

        elif isinstance(
            gp,
            list
        ):

            gp_list = gp

        else:

            gp_list = []


        autosomal_hits = [
            x
            for x in gp_list
            if str(
                x.get(
                    "chr",
                    ""
                )
            )
            in AUTOSOMES
        ]


        # Keep only uniquely interpretable autosomal mappings
        unique_hits = {
            (
                str(
                    x.get(
                        "chr"
                    )
                ),
                int(
                    x.get(
                        "start"
                    )
                ),
                int(
                    x.get(
                        "end"
                    )
                )
            )
            for x in autosomal_hits
            if (
                x.get(
                    "start"
                )
                is not None
                and
                x.get(
                    "end"
                )
                is not None
            )
        }


        if len(
            unique_hits
        ) == 1:

            chrom, start_pos, end_pos = (
                list(
                    unique_hits
                )[0]
            )

            status = (
                "unique_autosomal"
            )


        elif len(
            unique_hits
        ) == 0:

            chrom = None
            start_pos = np.nan
            end_pos = np.nan

            status = (
                "no_autosomal_mapping"
            )


        else:

            chrom = None
            start_pos = np.nan
            end_pos = np.nan

            status = (
                "ambiguous_autosomal"
            )


        coordinate_rows.append({

            "gene_id":
                gene_id,

            "symbol":
                r.get(
                    "symbol",
                    None
                ),

            "chromosome":
                chrom,

            "start":
                start_pos,

            "end":
                end_pos,

            "mapping_status":
                status
        })


    gene_coordinates = (
        pd.DataFrame(
            coordinate_rows
        )
        .drop_duplicates(
            subset="gene_id",
            keep="first"
        )
    )


    gene_coordinates.to_csv(
        GENE_COORD_PATH,
        index=False
    )


print(
    "\nCoordinate rows:",
    len(
        gene_coordinates
    )
)


print(
    "Unique autosomal mappings:",
    int(
        (
            gene_coordinates[
                "mapping_status"
            ]
            ==
            "unique_autosomal"
        ).sum()
    )
)


print(
    "Saved/cache:",
    GENE_COORD_PATH
)

In [ ]:
# =============================================================================
# Step31D2A Cell 135
# Genomic-coordinate coverage audit
# =============================================================================

coordinate_audit = (
    gene_coordinates[
        "mapping_status"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "mapping_status"
    )
    .reset_index(
        name="n_genes"
    )
)


coordinate_audit[
    "fraction"
] = (
    coordinate_audit[
        "n_genes"
    ]
    /
    len(
        gene_coordinates
    )
)


display(
    coordinate_audit
)


mapped_gene_ids = set(
    gene_coordinates.loc[
        gene_coordinates[
            "mapping_status"
        ]
        ==
        "unique_autosomal",
        "gene_id"
    ].astype(str)
)


print(
    "\nMapped genes present in AnnData:",
    len(
        mapped_gene_ids
        &
        set(
            adata_d.var_names
            .astype(str)
        )
    )
)

In [ ]:
# =============================================================================
# Step31D2A Cell 136
# Build hepatic-only CNV-analysis object
# =============================================================================

hepatic_mask = (
    adata_d.obs[
        "broad_cell_type"
    ]
    ==
    "Hepatic"
)


adata_hep = (
    adata_d[
        hepatic_mask
    ]
    .copy()
)


print(
    "Hepatic cells:",
    adata_hep.n_obs
)

print(
    "Genes:",
    adata_hep.n_vars
)


print(
    "\nTissue:"
)

display(
    adata_hep.obs[
        "tissue"
    ]
    .value_counts()
)


assert adata_hep.n_obs == 5103


print(
    "\nHEPATIC CNV OBJECT: PASS"
)

In [ ]:
# =============================================================================
# Step31D2A Cell 137
# Select autosomal expressed genes for CNV-like analysis
# =============================================================================

import scipy.sparse as sp


mapped_mask = (
    adata_hep.var_names
    .astype(str)
    .isin(
        mapped_gene_ids
    )
)


adata_hep_mapped = (
    adata_hep[
        :,
        mapped_mask
    ]
)


X_mapped = (
    adata_hep_mapped.X
)


if sp.issparse(
    X_mapped
):

    detection_fraction = (
        np.asarray(
            (
                X_mapped > 0
            ).sum(
                axis=0
            )
        ).ravel()
        /
        adata_hep_mapped.n_obs
    )

else:

    detection_fraction = (
        (
            X_mapped > 0
        ).mean(
            axis=0
        )
    )


MIN_DETECTION_FRACTION = 0.01


expressed_gene_ids = (
    adata_hep_mapped.var_names[
        detection_fraction
        >=
        MIN_DETECTION_FRACTION
    ]
    .astype(str)
    .tolist()
)


print(
    "Mapped autosomal genes:",
    adata_hep_mapped.n_vars
)


print(
    "Detected in >=1% Hepatic cells:",
    len(
        expressed_gene_ids
    )
)


assert len(
    expressed_gene_ids
) > 5000

In [ ]:
# =============================================================================
# Step31D2A Cell 138
# Construct fixed autosomal genomic bins
# =============================================================================

BIN_SIZE = 50
MIN_LAST_BIN_SIZE = 25


coord_use = (
    gene_coordinates.loc[
        gene_coordinates[
            "gene_id"
        ].isin(
            expressed_gene_ids
        )
        &
        (
            gene_coordinates[
                "mapping_status"
            ]
            ==
            "unique_autosomal"
        )
    ]
    .copy()
)


coord_use[
    "chromosome_num"
] = (
    coord_use[
        "chromosome"
    ].astype(int)
)


coord_use[
    "midpoint"
] = (
    (
        coord_use[
            "start"
        ]
        +
        coord_use[
            "end"
        ]
    )
    /
    2
)


coord_use = (
    coord_use
    .sort_values(
        [
            "chromosome_num",
            "midpoint",
            "gene_id"
        ]
    )
    .reset_index(
        drop=True
    )
)


cnv_bins = []


for chrom in range(
    1,
    23
):

    chrom_df = (
        coord_use.loc[
            coord_use[
                "chromosome_num"
            ]
            ==
            chrom
        ]
        .reset_index(
            drop=True
        )
    )


    genes = (
        chrom_df[
            "gene_id"
        ]
        .astype(str)
        .tolist()
    )


    for start in range(
        0,
        len(genes),
        BIN_SIZE
    ):

        bin_genes = (
            genes[
                start:
                start + BIN_SIZE
            ]
        )


        if len(
            bin_genes
        ) < MIN_LAST_BIN_SIZE:

            continue


        cnv_bins.append({

            "bin_id":
                f"chr{chrom}_bin"
                f"{start // BIN_SIZE + 1:03d}",

            "chromosome":
                str(chrom),

            "n_genes":
                len(
                    bin_genes
                ),

            "genes":
                bin_genes
        })


print(
    "CNV-like bins:",
    len(
        cnv_bins
    )
)


print(
    "Genes represented in bins:",
    sum(
        x[
            "n_genes"
        ]
        for x in cnv_bins
    )
)

In [ ]:
# =============================================================================
# Step31D2A Cell 139
# Compute bin-level CNV-like expression deviation
# =============================================================================

hep_normal_mask = (
    adata_hep.obs[
        "tissue"
    ].astype(str)
    ==
    "NonTumor"
)


assert int(
    hep_normal_mask.sum()
) == 415


n_cells = (
    adata_hep.n_obs
)

n_bins = (
    len(
        cnv_bins
    )
)


X_cnv_bins = np.zeros(
    (
        n_cells,
        n_bins
    ),
    dtype=np.float32
)


bin_metadata_rows = []


for j, bin_info in enumerate(
    cnv_bins
):

    gene_ids = (
        bin_info[
            "genes"
        ]
    )


    X_bin = (
        adata_hep[
            :,
            gene_ids
        ].X
    )


    cell_bin_mean = np.asarray(
        X_bin.mean(
            axis=1
        )
    ).ravel()


    reference_mean = float(
        cell_bin_mean[
            hep_normal_mask.to_numpy()
        ].mean()
    )


    deviation = (
        cell_bin_mean
        -
        reference_mean
    )


    # Recenter using normal-reference median
    reference_median_shift = float(
        np.median(
            deviation[
                hep_normal_mask.to_numpy()
            ]
        )
    )


    deviation = (
        deviation
        -
        reference_median_shift
    )


    X_cnv_bins[
        :,
        j
    ] = deviation.astype(
        np.float32
    )


    bin_metadata_rows.append({

        "bin_id":
            bin_info[
                "bin_id"
            ],

        "chromosome":
            bin_info[
                "chromosome"
            ],

        "n_genes":
            bin_info[
                "n_genes"
            ],

        "reference_mean_expression":
            reference_mean,

        "reference_median_shift":
            reference_median_shift
    })


cnv_bin_metadata = (
    pd.DataFrame(
        bin_metadata_rows
    )
)


print(
    "CNV-like matrix:",
    X_cnv_bins.shape
)


print(
    "Normal reference cells:",
    int(
        hep_normal_mask.sum()
    )
)

In [ ]:
# =============================================================================
# Step31D2A Cell 140
# Cell-level CNV-like burden
# =============================================================================

CNV_RMS = np.sqrt(
    np.mean(
        X_cnv_bins ** 2,
        axis=1
    )
)


CNV_MAD_MEAN = np.mean(
    np.abs(
        X_cnv_bins
    ),
    axis=1
)


adata_hep.obs[
    "cnv_like_rms"
] = (
    CNV_RMS
)


adata_hep.obs[
    "cnv_like_mean_abs"
] = (
    CNV_MAD_MEAN
)


print(
    "CNV-like RMS range:"
)

print(
    float(
        CNV_RMS.min()
    ),
    "to",
    float(
        CNV_RMS.max()
    )
)

In [ ]:
# =============================================================================
# Step31D2A Cell 141
# Freeze normal-reference-derived CNV-like thresholds
# =============================================================================

normal_rms = (
    adata_hep.obs.loc[
        hep_normal_mask,
        "cnv_like_rms"
    ]
    .astype(float)
    .to_numpy()
)


CNV_Q95 = float(
    np.quantile(
        normal_rms,
        0.95
    )
)


CNV_Q99 = float(
    np.quantile(
        normal_rms,
        0.99
    )
)


normal_median = float(
    np.median(
        normal_rms
    )
)


normal_mad = float(
    np.median(
        np.abs(
            normal_rms
            -
            normal_median
        )
    )
)


CNV_MEDIAN_PLUS_3MAD = (
    normal_median
    +
    3
    *
    1.4826
    *
    normal_mad
)


print(
    "Normal-reference median:",
    normal_median
)

print(
    "Normal-reference q95:",
    CNV_Q95
)

print(
    "Normal-reference q99:",
    CNV_Q99
)

print(
    "Median + 3 robust SD:",
    CNV_MEDIAN_PLUS_3MAD
)


# Primary threshold
CNV_PRIMARY_THRESHOLD = (
    CNV_Q99
)


adata_hep.obs[
    "cnv_like_high_q99"
] = (
    adata_hep.obs[
        "cnv_like_rms"
    ]
    >
    CNV_PRIMARY_THRESHOLD
)


adata_hep.obs[
    "cnv_like_high_q95"
] = (
    adata_hep.obs[
        "cnv_like_rms"
    ]
    >
    CNV_Q95
)


print(
    "\nPRIMARY CNV-LIKE THRESHOLD:",
    CNV_PRIMARY_THRESHOLD
)

In [ ]:
# =============================================================================
# Step31D2A Cell 142
# Tumor-vs-NonTumor hepatic CNV-like summary
# =============================================================================

hepatic_cnv_tissue_summary = (
    adata_hep.obs
    .groupby(
        "tissue",
        observed=True
    )
    .agg(
        n_cells=(
            "cnv_like_rms",
            "size"
        ),

        median_cnv_rms=(
            "cnv_like_rms",
            "median"
        ),

        mean_cnv_rms=(
            "cnv_like_rms",
            "mean"
        ),

        q75_cnv_rms=(
            "cnv_like_rms",
            lambda x:
                x.quantile(
                    0.75
                )
        ),

        q90_cnv_rms=(
            "cnv_like_rms",
            lambda x:
                x.quantile(
                    0.90
                )
        ),

        q99_high_fraction=(
            "cnv_like_high_q99",
            "mean"
        ),

        q95_high_fraction=(
            "cnv_like_high_q95",
            "mean"
        )
    )
    .reset_index()
)


display(
    hepatic_cnv_tissue_summary
)

In [ ]:
# =============================================================================
# Step31D2A Cell 143
# Patient-aware hepatic CNV-like summary
# =============================================================================

hepatic_cnv_patient_summary = (
    adata_hep.obs
    .groupby(
        [
            "patient_id",
            "tissue"
        ],
        observed=True
    )
    .agg(
        n_cells=(
            "cnv_like_rms",
            "size"
        ),

        median_cnv_rms=(
            "cnv_like_rms",
            "median"
        ),

        q99_high_fraction=(
            "cnv_like_high_q99",
            "mean"
        ),

        q95_high_fraction=(
            "cnv_like_high_q95",
            "mean"
        )
    )
    .reset_index()
)


display(
    hepatic_cnv_patient_summary
)

In [ ]:
# =============================================================================
# Step31D2A Cell 144
# Hepatic-cluster CNV-like summary
# =============================================================================

hepatic_cnv_cluster_summary = (
    adata_hep.obs
    .groupby(
        [
            "leiden_harmony_r08",
            "tissue"
        ],
        observed=True
    )
    .agg(
        n_cells=(
            "cnv_like_rms",
            "size"
        ),

        median_cnv_rms=(
            "cnv_like_rms",
            "median"
        ),

        q99_high_fraction=(
            "cnv_like_high_q99",
            "mean"
        ),

        q95_high_fraction=(
            "cnv_like_high_q95",
            "mean"
        )
    )
    .reset_index()
)


display(
    hepatic_cnv_cluster_summary
)

In [ ]:
# =============================================================================
# Step31D2A Cell 145
# Paired-patient CNV-like sensitivity
# =============================================================================

paired_patients = (
    hepatic_patient_pivot.loc[
        hepatic_patient_pivot[
            "paired_adequate"
        ],
        "patient_id"
    ]
    .astype(str)
    .tolist()
)


paired_rows = []


for patient in paired_patients:

    sub = (
        adata_hep.obs.loc[
            adata_hep.obs[
                "patient_id"
            ].astype(str)
            ==
            patient
        ]
    )


    tumor_values = (
        sub.loc[
            sub[
                "tissue"
            ].astype(str)
            ==
            "Tumor",
            "cnv_like_rms"
        ]
        .astype(float)
        .to_numpy()
    )


    normal_values = (
        sub.loc[
            sub[
                "tissue"
            ].astype(str)
            ==
            "NonTumor",
            "cnv_like_rms"
        ]
        .astype(float)
        .to_numpy()
    )


    paired_rows.append({

        "patient_id":
            patient,

        "n_tumor":
            len(
                tumor_values
            ),

        "n_normal":
            len(
                normal_values
            ),

        "tumor_median":
            np.median(
                tumor_values
            ),

        "normal_median":
            np.median(
                normal_values
            ),

        "median_difference":
            np.median(
                tumor_values
            )
            -
            np.median(
                normal_values
            ),

        "tumor_higher_than_normal":
            np.median(
                tumor_values
            )
            >
            np.median(
                normal_values
            )
    })


paired_cnv_sensitivity = (
    pd.DataFrame(
        paired_rows
    )
)


display(
    paired_cnv_sensitivity
)


print(
    "\nPatients with Tumor median > paired NonTumor median:",
    int(
        paired_cnv_sensitivity[
            "tumor_higher_than_normal"
        ].sum()
    ),
    "/",
    len(
        paired_cnv_sensitivity
    )
)

In [ ]:
# =============================================================================
# Step31D2A Cell 146
# Save CNV-like evidence
# =============================================================================

from scipy import sparse


CNV_BIN_MATRIX_PATH = (
    DATA_DIR /
    "GSE326201_STEP31D2A_Hepatic_CNVLikeBins.npz"
)


CNV_HEPATIC_META_PATH = (
    AUDIT_DIR /
    "Step31D2A_HepaticCell_CNVLikeEvidence.csv"
)


sparse.save_npz(
    CNV_BIN_MATRIX_PATH,
    sparse.csr_matrix(
        X_cnv_bins
    )
)


adata_hep.obs[
    [
        "patient_id",
        "sample_id",
        "tissue",
        "leiden_harmony_r08",
        "cnv_like_rms",
        "cnv_like_mean_abs",
        "cnv_like_high_q95",
        "cnv_like_high_q99"
    ]
].to_csv(
    CNV_HEPATIC_META_PATH
)


cnv_bin_metadata.to_csv(
    AUDIT_DIR /
    "Step31D2A_CNVLikeBinMetadata.csv",
    index=False
)


hepatic_cnv_tissue_summary.to_csv(
    AUDIT_DIR /
    "Step31D2A_HepaticCNV_TissueSummary.csv",
    index=False
)


hepatic_cnv_patient_summary.to_csv(
    AUDIT_DIR /
    "Step31D2A_HepaticCNV_PatientSummary.csv",
    index=False
)


hepatic_cnv_cluster_summary.to_csv(
    AUDIT_DIR /
    "Step31D2A_HepaticCNV_ClusterSummary.csv",
    index=False
)


paired_cnv_sensitivity.to_csv(
    AUDIT_DIR /
    "Step31D2A_HepaticCNV_PairedSensitivity.csv",
    index=False
)


print(
    "CNV-like evidence saved."
)

In [ ]:
# =============================================================================
# Step31D2A Cell 147
# REPORT — hepatic CNV-like malignancy evidence
# =============================================================================

print("=" * 120)

print(
    "STEP31D2A — HEPATIC CNV-LIKE MALIGNANCY EVIDENCE"
)

print("=" * 120)


print(
    "\n[1] ANALYSIS UNIVERSE"
)

print(
    "Hepatic cells:",
    adata_hep.n_obs
)

print(
    "Tumor-derived:",
    int(
        (
            adata_hep.obs[
                "tissue"
            ].astype(str)
            ==
            "Tumor"
        ).sum()
    )
)

print(
    "NonTumor reference:",
    int(
        (
            adata_hep.obs[
                "tissue"
            ].astype(str)
            ==
            "NonTumor"
        ).sum()
    )
)


print(
    "\n[2] GENOMIC COORDINATE COVERAGE"
)

display(
    coordinate_audit
)

print(
    "Genes retained after autosomal mapping + >=1% detection:",
    len(
        expressed_gene_ids
    )
)

print(
    "CNV-like genomic bins:",
    len(
        cnv_bins
    )
)


print(
    "\n[3] NORMAL-REFERENCE THRESHOLDS"
)

print(
    "Normal median:",
    normal_median
)

print(
    "Normal q95:",
    CNV_Q95
)

print(
    "Normal q99:",
    CNV_Q99
)

print(
    "Median + 3 robust SD:",
    CNV_MEDIAN_PLUS_3MAD
)

print(
    "Primary threshold:",
    CNV_PRIMARY_THRESHOLD
)


print(
    "\n[4] TISSUE SUMMARY"
)

display(
    hepatic_cnv_tissue_summary
)


print(
    "\n[5] PATIENT-AWARE SUMMARY"
)

display(
    hepatic_cnv_patient_summary
)


print(
    "\n[6] HEPATIC CLUSTER SUMMARY"
)

display(
    hepatic_cnv_cluster_summary
)


print(
    "\n[7] PAIRED-REFERENCE SENSITIVITY"
)

display(
    paired_cnv_sensitivity
)


print(
    "\nTumor median > paired NonTumor median:",
    int(
        paired_cnv_sensitivity[
            "tumor_higher_than_normal"
        ].sum()
    ),
    "/",
    len(
        paired_cnv_sensitivity
    )
)


print(
    "\n[8] CLAIM BOUNDARY"
)

print(
"""
This analysis provides expression-derived CNV-like evidence only.

It does not establish DNA copy-number alterations.

The CNV-like threshold was derived exclusively from NonTumor Hepatic
reference cells and was not optimized against Tumor cells.

No Core50, MS3/MS6 program, Original RiskScore, or Step29 regulator
information was used.

Tumor-derived Hepatic cells remain malignant candidates until the
CNV-like evidence is reviewed jointly with tissue origin, cluster
structure, and patient-level consistency.

The Epithelial/Cholangiocyte compartment is deliberately excluded from
this hepatic-reference CNV analysis and will be evaluated separately.
"""
)


print(
    "\nSTEP31D2A: COMPLETE"
)

print(
    "NEXT: review CNV-like evidence and define the independent malignant-cell lock."
)

print("=" * 120)

## Step31D2B — Independent malignant-hepatic lock

### Primary definition

The primary malignant-supported population is defined as:

1. Frozen broad cell type = Hepatic;
2. Tissue source = Tumor;
3. Belongs to the independently identified hepatic compartment;
4. The Tumor-derived hepatic compartment demonstrates a reproducible
   elevation in expression-derived CNV-like burden relative to the
   NonTumor hepatic reference;
5. This CNV-like shift is directionally reproduced in all paired patients
   with adequate normal-hepatocyte reference.

The normal-reference q99 threshold is NOT used as the primary malignant-cell
classifier because it is intentionally highly specific and has limited
sensitivity at the single-cell level.

Instead, q95 and q99 CNV-like subsets are retained for sensitivity analyses.

No Core50, MS3/MS6, Original RiskScore, or Step29 regulator information is
used in this definition.

In [ ]:
# =============================================================================
# Step31D2B Cell 149
# Verify evidence required for malignant-hepatic lock
# =============================================================================

tumor_hepatic_median = float(
    hepatic_cnv_tissue_summary.loc[
        hepatic_cnv_tissue_summary[
            "tissue"
        ].astype(str)
        ==
        "Tumor",
        "median_cnv_rms"
    ].iloc[0]
)


normal_hepatic_median = float(
    hepatic_cnv_tissue_summary.loc[
        hepatic_cnv_tissue_summary[
            "tissue"
        ].astype(str)
        ==
        "NonTumor",
        "median_cnv_rms"
    ].iloc[0]
)


n_paired_supported = int(
    paired_cnv_sensitivity[
        "tumor_higher_than_normal"
    ].sum()
)


n_paired_total = len(
    paired_cnv_sensitivity
)


print(
    "Tumor Hepatic median CNV-like RMS:",
    tumor_hepatic_median
)

print(
    "NonTumor Hepatic median CNV-like RMS:",
    normal_hepatic_median
)

print(
    "Paired patients supporting Tumor > NonTumor:",
    f"{n_paired_supported}/{n_paired_total}"
)


assert (
    tumor_hepatic_median
    >
    normal_hepatic_median
)


assert (
    n_paired_supported
    ==
    n_paired_total
)


assert (
    n_paired_total
    >=
    3
)


print(
    "\nINDEPENDENT MALIGNANCY SUPPORT: PASS"
)

In [ ]:
# =============================================================================
# Step31D2B Cell 150
# Transfer hepatic CNV-like evidence to full broad-cell universe
# =============================================================================

CNV_COLUMNS = [
    "cnv_like_rms",
    "cnv_like_mean_abs",
    "cnv_like_high_q95",
    "cnv_like_high_q99"
]


for col in CNV_COLUMNS:

    if col in [
        "cnv_like_high_q95",
        "cnv_like_high_q99"
    ]:

        adata_d.obs[
            col
        ] = False

    else:

        adata_d.obs[
            col
        ] = np.nan


    adata_d.obs.loc[
        adata_hep.obs_names,
        col
    ] = (
        adata_hep.obs[
            col
        ]
        .reindex(
            adata_hep.obs_names
        )
        .values
    )


# Restore boolean dtype explicitly
adata_d.obs[
    "cnv_like_high_q95"
] = (
    adata_d.obs[
        "cnv_like_high_q95"
    ].astype(bool)
)

adata_d.obs[
    "cnv_like_high_q99"
] = (
    adata_d.obs[
        "cnv_like_high_q99"
    ].astype(bool)
)


print(
    "CNV evidence transferred to full universe."
)

print(
    "Cells with CNV-like RMS:",
    int(
        adata_d.obs[
            "cnv_like_rms"
        ].notna()
        .sum()
    )
)


assert (
    adata_d.obs[
        "cnv_like_rms"
    ].notna()
    .sum()
    ==
    5103
)

In [ ]:
# =============================================================================
# Step31D2B Cell 151
# Independent malignant-status assignment
# =============================================================================

adata_d.obs[
    "independent_malignancy_status"
] = (
    "Not_assessed_nonhepatic"
)


# NonTumor hepatic reference
adata_d.obs.loc[
    (
        adata_d.obs[
            "broad_cell_type"
        ]
        ==
        "Hepatic"
    )
    &
    (
        adata_d.obs[
            "tissue"
        ].astype(str)
        ==
        "NonTumor"
    ),
    "independent_malignancy_status"
] = (
    "Normal_reference_hepatic"
)


# Primary malignant-supported hepatic cells
adata_d.obs.loc[
    (
        adata_d.obs[
            "broad_cell_type"
        ]
        ==
        "Hepatic"
    )
    &
    (
        adata_d.obs[
            "tissue"
        ].astype(str)
        ==
        "Tumor"
    ),
    "independent_malignancy_status"
] = (
    "Malignant_supported_hepatic"
)


print(
    adata_d.obs[
        "independent_malignancy_status"
    ]
    .value_counts()
)

In [ ]:
# =============================================================================
# Step31D2B Cell 152
# CNV-supported sensitivity tiers within Tumor-derived hepatic cells
# =============================================================================

malignant_primary_mask = (
    adata_d.obs[
        "independent_malignancy_status"
    ]
    ==
    "Malignant_supported_hepatic"
)


adata_d.obs[
    "malignant_cnv_sensitivity_tier"
] = (
    "Not_primary_malignant"
)


adata_d.obs.loc[
    malignant_primary_mask,
    "malignant_cnv_sensitivity_tier"
] = (
    "Primary_only"
)


adata_d.obs.loc[
    malignant_primary_mask
    &
    adata_d.obs[
        "cnv_like_high_q95"
    ],
    "malignant_cnv_sensitivity_tier"
] = (
    "CNV_q95_supported"
)


adata_d.obs.loc[
    malignant_primary_mask
    &
    adata_d.obs[
        "cnv_like_high_q99"
    ],
    "malignant_cnv_sensitivity_tier"
] = (
    "CNV_q99_supported"
)


malignant_tier_counts = (
    adata_d.obs.loc[
        malignant_primary_mask,
        "malignant_cnv_sensitivity_tier"
    ]
    .value_counts()
    .rename_axis(
        "tier"
    )
    .reset_index(
        name="n_cells"
    )
)


malignant_tier_counts[
    "fraction_of_primary_malignant"
] = (
    malignant_tier_counts[
        "n_cells"
    ]
    /
    malignant_primary_mask.sum()
)


display(
    malignant_tier_counts
)

In [ ]:
# =============================================================================
# Step31D2B Cell 153
# Primary malignant-hepatic composition
# =============================================================================

malignant_cluster_composition = (
    adata_d.obs.loc[
        malignant_primary_mask
    ]
    .groupby(
        [
            "leiden_harmony_r08",
            "patient_id"
        ],
        observed=True
    )
    .size()
    .rename(
        "n_cells"
    )
    .reset_index()
)


print(
    "Primary malignant-supported cells:",
    int(
        malignant_primary_mask.sum()
    )
)


print(
    "\nClusters represented:"
)

display(
    adata_d.obs.loc[
        malignant_primary_mask,
        "leiden_harmony_r08"
    ]
    .value_counts()
    .sort_index()
)


assert set(
    adata_d.obs.loc[
        malignant_primary_mask,
        "leiden_harmony_r08"
    ].astype(str)
    .unique()
) == {
    "8",
    "20"
}


print(
    "\nPRIMARY MALIGNANT CLUSTER COMPOSITION: PASS"
)

In [ ]:
# =============================================================================
# Step31D2B Cell 154
# Patient-level malignant-supported cell counts
# =============================================================================

malignant_patient_counts = (
    adata_d.obs.loc[
        malignant_primary_mask
    ]
    .groupby(
        [
            "patient_id",
            "sample_id"
        ],
        observed=True
    )
    .agg(
        n_malignant_cells=(
            "cnv_like_rms",
            "size"
        ),

        median_cnv_like_rms=(
            "cnv_like_rms",
            "median"
        ),

        q95_supported_fraction=(
            "cnv_like_high_q95",
            "mean"
        ),

        q99_supported_fraction=(
            "cnv_like_high_q99",
            "mean"
        )
    )
    .reset_index()
)


display(
    malignant_patient_counts
)

In [ ]:
# =============================================================================
# Step31D2B Cell 155
# Freeze normal hepatic reference composition
# =============================================================================

normal_reference_mask = (
    adata_d.obs[
        "independent_malignancy_status"
    ]
    ==
    "Normal_reference_hepatic"
)


normal_reference_summary = (
    adata_d.obs.loc[
        normal_reference_mask
    ]
    .groupby(
        [
            "patient_id",
            "sample_id",
            "leiden_harmony_r08"
        ],
        observed=True
    )
    .size()
    .rename(
        "n_cells"
    )
    .reset_index()
)


display(
    normal_reference_summary
)


assert (
    normal_reference_mask.sum()
    ==
    415
)

In [ ]:
# =============================================================================
# Step31D2B Cell 156
# Explicit secondary epithelial compartment boundary
# =============================================================================

epi_mask = (
    adata_d.obs[
        "broad_cell_type"
    ]
    ==
    "Epithelial_Cholangiocyte"
)


adata_d.obs[
    "secondary_epithelial_status"
] = (
    "Not_epithelial"
)


adata_d.obs.loc[
    epi_mask
    &
    (
        adata_d.obs[
            "tissue"
        ].astype(str)
        ==
        "NonTumor"
    ),
    "secondary_epithelial_status"
] = (
    "NonTumor_epithelial"
)


adata_d.obs.loc[
    epi_mask
    &
    (
        adata_d.obs[
            "tissue"
        ].astype(str)
        ==
        "Tumor"
    ),
    "secondary_epithelial_status"
] = (
    "Tumor_associated_epithelial_unclassified"
)


display(
    adata_d.obs[
        "secondary_epithelial_status"
    ]
    .value_counts()
)


print(
    """
Tumor-associated Epithelial/Cholangiocyte cells are NOT included in the
primary malignant HCC universe because their malignancy has not been
independently established.
"""
)

In [ ]:
# =============================================================================
# Step31D2B Cell 157
# Build primary independently defined malignant-hepatic universe
# =============================================================================

adata_malignant = (
    adata_d[
        malignant_primary_mask
    ]
    .copy()
)


print(
    adata_malignant
)


print(
    "\nMalignant-supported Hepatic cells:",
    adata_malignant.n_obs
)


print(
    "Genes:",
    adata_malignant.n_vars
)


print(
    "Patients:",
    adata_malignant.obs[
        "patient_id"
    ].nunique()
)


print(
    "Samples:",
    adata_malignant.obs[
        "sample_id"
    ].nunique()
)


assert (
    adata_malignant.n_obs
    ==
    4688
)


assert (
    adata_malignant.n_vars
    ==
    32732
)


assert (
    adata_malignant.obs[
        "patient_id"
    ].nunique()
    ==
    10
)


print(
    "\nMALIGNANT HEPATIC UNIVERSE: PASS"
)

In [ ]:
# =============================================================================
# Step31D2B Cell 158
# Save independent malignant-cell lock
# =============================================================================

MALIGNANT_LOCK_PATH = (
    DATA_DIR /
    "GSE326201_STEP31D2B_INDEPENDENT_MALIGNANT_HEPATIC_LOCK.h5ad"
)


FULL_D2B_PATH = (
    DATA_DIR /
    "GSE326201_STEP31D2B_FULL_WITH_MALIGNANCY_STATUS.h5ad"
)


adata_malignant.write_h5ad(
    MALIGNANT_LOCK_PATH,
    compression="gzip"
)


adata_d.write_h5ad(
    FULL_D2B_PATH,
    compression="gzip"
)


malignant_patient_counts.to_csv(
    AUDIT_DIR /
    "Step31D2B_MalignantPatientCounts.csv",
    index=False
)


malignant_cluster_composition.to_csv(
    AUDIT_DIR /
    "Step31D2B_MalignantClusterPatientComposition.csv",
    index=False
)


normal_reference_summary.to_csv(
    AUDIT_DIR /
    "Step31D2B_NormalHepaticReferenceLock.csv",
    index=False
)


malignant_tier_counts.to_csv(
    AUDIT_DIR /
    "Step31D2B_CNVSupportSensitivityTiers.csv",
    index=False
)


adata_d.obs[
    [
        "patient_id",
        "sample_id",
        "tissue",
        "broad_cell_type",
        "leiden_harmony_r08",
        "independent_malignancy_status",
        "secondary_epithelial_status",
        "cnv_like_rms",
        "cnv_like_high_q95",
        "cnv_like_high_q99",
        "malignant_cnv_sensitivity_tier"
    ]
].to_csv(
    AUDIT_DIR /
    "Step31D2B_CellLevelMalignancyAudit.csv"
)


print(
    "Malignant lock:"
)

print(
    MALIGNANT_LOCK_PATH
)

print(
    "Exists:",
    MALIGNANT_LOCK_PATH.exists()
)


In [ ]:
# =============================================================================
# Step31D2B Cell 159
# FINAL REPORT — independent malignant hepatic lock
# =============================================================================

print("=" * 120)

print(
    "STEP31D2B — INDEPENDENT MALIGNANT HEPATIC LOCK"
)

print("=" * 120)


print(
    "\n[1] EVIDENCE"
)

print(
    "NonTumor Hepatic median CNV-like RMS:",
    normal_hepatic_median
)

print(
    "Tumor Hepatic median CNV-like RMS:",
    tumor_hepatic_median
)

print(
    "Paired patients Tumor > NonTumor:",
    f"{n_paired_supported}/{n_paired_total}"
)


print(
    "\n[2] PRIMARY MALIGNANT DEFINITION"
)

print(
    """
Broad cell type = Hepatic
AND
Tissue = Tumor
AND
independent group-level CNV-like evidence supports malignant-associated
genomic-expression deviation.
"""
)


print(
    "\n[3] PRIMARY MALIGNANT UNIVERSE"
)

print(
    "Cells:",
    adata_malignant.n_obs
)

print(
    "Genes:",
    adata_malignant.n_vars
)

print(
    "Patients:",
    adata_malignant.obs[
        "patient_id"
    ].nunique()
)

print(
    "Samples:",
    adata_malignant.obs[
        "sample_id"
    ].nunique()
)


print(
    "\n[4] CLUSTER COMPOSITION"
)

display(
    adata_malignant.obs[
        "leiden_harmony_r08"
    ]
    .value_counts()
    .sort_index()
)


print(
    "\n[5] PATIENT COMPOSITION"
)

display(
    malignant_patient_counts
)


print(
    "\n[6] CNV-LIKE SENSITIVITY TIERS"
)

display(
    malignant_tier_counts
)


print(
    "\n[7] NORMAL HEPATIC REFERENCE"
)

print(
    "Cells:",
    int(
        normal_reference_mask.sum()
    )
)

print(
    "Patients:",
    adata_d.obs.loc[
        normal_reference_mask,
        "patient_id"
    ].nunique()
)


print(
    "\n[8] SECONDARY EPITHELIAL COMPARTMENT"
)

print(
    "Tumor-associated Epithelial/Cholangiocyte:",
    int(
        (
            adata_d.obs[
                "secondary_epithelial_status"
            ]
            ==
            "Tumor_associated_epithelial_unclassified"
        ).sum()
    )
)

print(
    "Included in primary malignant universe:",
    False
)


print(
    "\n[9] CORE50 / PRIOR BIOLOGY FIREWALL"
)

print(
    "Core50 used:",
    False
)

print(
    "MS3/MS6 used:",
    False
)

print(
    "Original RiskScore used:",
    False
)

print(
    "Step29 regulators used:",
    False
)


print(
    "\n[10] CLAIM BOUNDARY"
)

print(
"""
The primary Step31 malignant-cell universe consists of Tumor-derived
Hepatic cells supported at the compartment level by independently derived
CNV-like evidence.

The q95/q99 normal-reference thresholds are sensitivity tiers and were
not used to select the primary malignant population.

Expression-derived CNV-like burden is supporting evidence and is not
equivalent to DNA copy-number profiling.

Tumor-associated Epithelial/Cholangiocyte cells remain unclassified and
are excluded from the primary HCC malignant-cell universe.

No frozen MS3/Core50 or downstream prior biological information was used
to define malignancy.
"""
)


print(
    "\n[11] FILE"
)

print(
    MALIGNANT_LOCK_PATH
)

print(
    "Exists:",
    MALIGNANT_LOCK_PATH.exists()
)


print(
    "\nSTEP31D2B: FROZEN"
)

print(
    "NEXT: Step31E — frozen MS3 Core50 independent validation."
)

print("=" * 120)

In [ ]:
# =============================================================================
# Step31 Cell 160A
# FIX — robust detection of frozen Core50 Ensembl-ID column
# =============================================================================

import pandas as pd
import numpy as np
import re


print("=" * 110)
print("STEP31 CORE50 MAPPING — ROBUST COLUMN DETECTION")
print("=" * 110)


# -----------------------------------------------------------------------------
# 1. Confirm required objects are already in memory
# -----------------------------------------------------------------------------

assert "adata_malignant" in globals(), (
    "adata_malignant is missing. "
    "Rerun Cell160 from the beginning up to the malignant-object loading step."
)

assert "core50_validation_map" in globals(), (
    "core50_validation_map is missing. "
    "Reload Step31_Core50_FrozenValidationMapping.csv first."
)


print(
    "\nCore50 mapping shape:",
    core50_validation_map.shape
)

print(
    "\nAvailable columns:"
)

for c in core50_validation_map.columns:
    print(" -", repr(c))


assert len(core50_validation_map) == 50, (
    f"Expected 50 frozen Core50 rows, "
    f"found {len(core50_validation_map)}."
)


# -----------------------------------------------------------------------------
# 2. Standardize malignant Ensembl universe
# -----------------------------------------------------------------------------

malignant_gene_ids = set(
    pd.Index(
        adata_malignant.var_names.astype(str)
    )
    .str.replace(
        r"\.\d+$",
        "",
        regex=True
    )
)


# -----------------------------------------------------------------------------
# 3. Scan every column for Ensembl-like values
# -----------------------------------------------------------------------------

candidate_rows = []


for col in core50_validation_map.columns:

    values = (
        core50_validation_map[col]
        .astype(str)
        .str.strip()
        .str.replace(
            r"\.\d+$",
            "",
            regex=True
        )
    )


    ensembl_mask = (
        values
        .str.match(
            r"^ENSG\d+$",
            na=False
        )
    )


    n_ensembl_like = int(
        ensembl_mask.sum()
    )


    if n_ensembl_like == 0:
        continue


    ensembl_values = (
        values.loc[
            ensembl_mask
        ]
        .tolist()
    )


    n_unique = len(
        set(
            ensembl_values
        )
    )


    n_present_in_malignant = sum(
        x in malignant_gene_ids
        for x in ensembl_values
    )


    candidate_rows.append({

        "column":
            col,

        "n_ensembl_like":
            n_ensembl_like,

        "n_unique":
            n_unique,

        "n_present_in_malignant":
            n_present_in_malignant,

        "coverage_fraction":
            (
                n_present_in_malignant
                /
                n_ensembl_like
            )
            if n_ensembl_like > 0
            else np.nan
    })


ensembl_candidates = pd.DataFrame(
    candidate_rows
)


assert len(ensembl_candidates) > 0, (
    "No column containing Ensembl-like IDs was detected "
    "in the frozen Core50 mapping."
)


ensembl_candidates = (
    ensembl_candidates
    .sort_values(
        [
            "n_present_in_malignant",
            "n_unique",
            "n_ensembl_like"
        ],
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


print(
    "\nDetected Ensembl-ID candidate columns:"
)

display(
    ensembl_candidates
)


# -----------------------------------------------------------------------------
# 4. Select the column with maximal coverage in frozen malignant universe
# -----------------------------------------------------------------------------

CORE50_GENE_ID_COL = (
    ensembl_candidates
    .iloc[0][
        "column"
    ]
)


print(
    "\nSelected Core50 Ensembl-ID column:"
)

print(
    CORE50_GENE_ID_COL
)


# -----------------------------------------------------------------------------
# 5. Build frozen Core50 Ensembl ID list
# -----------------------------------------------------------------------------

CORE50_ENSEMBL_IDS = (
    core50_validation_map[
        CORE50_GENE_ID_COL
    ]
    .astype(str)
    .str.strip()
    .str.replace(
        r"\.\d+$",
        "",
        regex=True
    )
    .tolist()
)


print(
    "\nNumber of Core50 IDs:",
    len(
        CORE50_ENSEMBL_IDS
    )
)

print(
    "Unique Core50 IDs:",
    len(
        set(
            CORE50_ENSEMBL_IDS
        )
    )
)


assert len(
    CORE50_ENSEMBL_IDS
) == 50


assert len(
    set(
        CORE50_ENSEMBL_IDS
    )
) == 50, (
    "The selected Ensembl column does not contain "
    "50 unique gene IDs."
)


# -----------------------------------------------------------------------------
# 6. Verify 50/50 coverage in malignant object
# -----------------------------------------------------------------------------

core50_present = [
    x
    for x in CORE50_ENSEMBL_IDS
    if x in malignant_gene_ids
]


core50_missing = [
    x
    for x in CORE50_ENSEMBL_IDS
    if x not in malignant_gene_ids
]


print(
    "\nCoverage in malignant object:",
    f"{len(core50_present)}/50"
)


if len(core50_missing) > 0:

    print(
        "\nMissing IDs:"
    )

    print(
        core50_missing
    )


assert len(
    core50_present
) == 50, (
    "Frozen Core50 Ensembl mapping does not have 50/50 "
    "coverage in the malignant object. Stop before Step31E."
)


# -----------------------------------------------------------------------------
# 7. Restore fixed alias map
# -----------------------------------------------------------------------------

CORE50_ALIAS_MAP = {
    "SGOL1": "SGO1",
    "SGOL2": "SGO2"
}


# -----------------------------------------------------------------------------
# 8. Update handoff manifest with actual detected column
# -----------------------------------------------------------------------------

if "handoff" in globals():

    handoff[
        "core50"
    ][
        "ensembl_mapping_column"
    ] = str(
        CORE50_GENE_ID_COL
    )


    if "HANDOFF_PATH" in globals():

        import json

        with open(
            HANDOFF_PATH,
            "w",
            encoding="utf-8"
        ) as f:

            json.dump(
                handoff,
                f,
                ensure_ascii=False,
                indent=2
            )


# -----------------------------------------------------------------------------
# 9. Final resume check
# -----------------------------------------------------------------------------

print(
    "\n" + "=" * 110
)

print(
    "STEP31 RESUME STATE: READY"
)

print(
    "=" * 110
)


print(
    "\nFrozen through:"
)

print(
    "Step31D2B — Independent Malignant Hepatic Lock"
)


print(
    "\nMalignant universe:"
)

print(
    "Cells:",
    adata_malignant.n_obs
)

print(
    "Genes:",
    adata_malignant.n_vars
)

print(
    "Patients:",
    adata_malignant.obs[
        "patient_id"
    ].nunique()
)


print(
    "\nFrozen Core50:"
)

print(
    "Mapping column:",
    CORE50_GENE_ID_COL
)

print(
    "Genes:",
    len(
        CORE50_ENSEMBL_IDS
    )
)

print(
    "Coverage:",
    f"{len(core50_present)}/50"
)

print(
    "Aliases:",
    CORE50_ALIAS_MAP
)


print(
    "\nNEXT:"
)

print(
    "Step31E — Frozen MS3 Core50 Independent Validation"
)


print(
    "\nSAFE TO PAUSE / RESTART"
)

print("=" * 110)

In [ ]:
from hcc_release_paths import project_path as _hcc_path
# =============================================================================
# Step31 Cell 160 — FINAL
# PAUSE / RESUME CHECKPOINT
# Clean restart entry point for Step31E
# =============================================================================

from pathlib import Path
import json
import pandas as pd
import numpy as np
import scanpy as sc


# -----------------------------------------------------------------------------
# 1. Restore Step31 paths
# -----------------------------------------------------------------------------

STEP31_DIR = Path(
    _hcc_path('LIHC_scRNA_Step31_Independent_Validation')
)

DATA_DIR = STEP31_DIR / "Data"
TABLES_DIR = STEP31_DIR / "Tables"
FIGURES_DIR = STEP31_DIR / "Figures"
AUDIT_DIR = STEP31_DIR / "Audit"
RESOURCES_DIR = STEP31_DIR / "Resources"


for p in [
    DATA_DIR,
    TABLES_DIR,
    FIGURES_DIR,
    AUDIT_DIR,
    RESOURCES_DIR
]:
    p.mkdir(
        parents=True,
        exist_ok=True
    )


# -----------------------------------------------------------------------------
# 2. Frozen file paths
# -----------------------------------------------------------------------------

BROAD_LOCK_PATH = (
    DATA_DIR /
    "GSE326201_STEP31C3C_BROAD_CELLTYPE_LOCK.h5ad"
)

MALIGNANT_LOCK_PATH = (
    DATA_DIR /
    "GSE326201_STEP31D2B_INDEPENDENT_MALIGNANT_HEPATIC_LOCK.h5ad"
)

FULL_D2B_PATH = (
    DATA_DIR /
    "GSE326201_STEP31D2B_FULL_WITH_MALIGNANCY_STATUS.h5ad"
)

CORE50_MAPPING_PATH = (
    AUDIT_DIR /
    "Step31_Core50_FrozenValidationMapping.csv"
)

GENE_COORD_PATH = (
    RESOURCES_DIR /
    "Step31_Ensembl_GRCh38_GeneCoordinates.csv"
)

HANDOFF_PATH = (
    AUDIT_DIR /
    "Step31_PAUSE_HANDOFF.json"
)


# -----------------------------------------------------------------------------
# 3. Verify frozen files
# -----------------------------------------------------------------------------

required_files = {

    "Broad cell-type lock":
        BROAD_LOCK_PATH,

    "Independent malignant lock":
        MALIGNANT_LOCK_PATH,

    "Full malignancy-status object":
        FULL_D2B_PATH,

    "Frozen Core50 validation mapping":
        CORE50_MAPPING_PATH,

    "Ensembl genomic coordinates":
        GENE_COORD_PATH
}


print("=" * 110)
print("STEP31 — PAUSE / RESUME CHECKPOINT")
print("=" * 110)


all_files_ok = True


for name, path in required_files.items():

    exists = path.exists()

    print(
        f"{name:<38}",
        "OK" if exists else "MISSING",
        "|",
        path
    )

    if not exists:
        all_files_ok = False


assert all_files_ok, (
    "One or more frozen Step31 files are missing."
)


# -----------------------------------------------------------------------------
# 4. Load frozen malignant universe
# -----------------------------------------------------------------------------

print(
    "\nLoading frozen malignant universe..."
)


adata_malignant = sc.read_h5ad(
    MALIGNANT_LOCK_PATH
)


assert adata_malignant.n_obs == 4688
assert adata_malignant.n_vars == 32732

assert (
    adata_malignant.obs[
        "patient_id"
    ].nunique()
    ==
    10
)

assert (
    adata_malignant.obs[
        "sample_id"
    ].nunique()
    ==
    10
)

assert (
    set(
        adata_malignant.obs[
            "leiden_harmony_r08"
        ]
        .astype(str)
        .unique()
    )
    ==
    {"8", "20"}
)


print(
    "Malignant object: PASS"
)


# -----------------------------------------------------------------------------
# 5. Load frozen Core50 mapping
# -----------------------------------------------------------------------------

core50_validation_map = pd.read_csv(
    CORE50_MAPPING_PATH
)


print(
    "\nCore50 mapping shape:",
    core50_validation_map.shape
)


print(
    "\nCore50 mapping columns:"
)


for c in core50_validation_map.columns:
    print(" -", repr(c))


assert len(
    core50_validation_map
) == 50, (
    f"Expected 50 frozen Core50 rows, "
    f"found {len(core50_validation_map)}."
)


# -----------------------------------------------------------------------------
# 6. Robustly detect the true Ensembl-ID column
# -----------------------------------------------------------------------------

malignant_gene_ids = set(
    pd.Index(
        adata_malignant.var_names
        .astype(str)
    )
    .str.replace(
        r"\.\d+$",
        "",
        regex=True
    )
)


candidate_rows = []


for col in core50_validation_map.columns:

    values = (
        core50_validation_map[
            col
        ]
        .astype(str)
        .str.strip()
        .str.replace(
            r"\.\d+$",
            "",
            regex=True
        )
    )


    ensembl_mask = (
        values.str.match(
            r"^ENSG\d+$",
            na=False
        )
    )


    n_ensembl_like = int(
        ensembl_mask.sum()
    )


    if n_ensembl_like == 0:
        continue


    ids = (
        values.loc[
            ensembl_mask
        ]
        .tolist()
    )


    n_present = sum(
        x in malignant_gene_ids
        for x in ids
    )


    candidate_rows.append({

        "column":
            col,

        "n_ensembl_like":
            n_ensembl_like,

        "n_unique":
            len(
                set(
                    ids
                )
            ),

        "n_present_in_malignant":
            n_present
    })


ensembl_candidates = pd.DataFrame(
    candidate_rows
)


assert len(
    ensembl_candidates
) > 0, (
    "No Ensembl-like column found in frozen Core50 mapping."
)


ensembl_candidates = (
    ensembl_candidates
    .sort_values(
        [
            "n_present_in_malignant",
            "n_unique",
            "n_ensembl_like"
        ],
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


print(
    "\nDetected Ensembl candidate columns:"
)

display(
    ensembl_candidates
)


CORE50_GENE_ID_COL = (
    ensembl_candidates
    .iloc[0][
        "column"
    ]
)


print(
    "\nSelected Core50 Ensembl column:"
)

print(
    CORE50_GENE_ID_COL
)


# -----------------------------------------------------------------------------
# 7. Build and validate frozen Core50 IDs
# -----------------------------------------------------------------------------

CORE50_ENSEMBL_IDS = (
    core50_validation_map[
        CORE50_GENE_ID_COL
    ]
    .astype(str)
    .str.strip()
    .str.replace(
        r"\.\d+$",
        "",
        regex=True
    )
    .tolist()
)


assert len(
    CORE50_ENSEMBL_IDS
) == 50


assert len(
    set(
        CORE50_ENSEMBL_IDS
    )
) == 50


core50_present = [
    x
    for x in CORE50_ENSEMBL_IDS
    if x in malignant_gene_ids
]


core50_missing = [
    x
    for x in CORE50_ENSEMBL_IDS
    if x not in malignant_gene_ids
]


print(
    "\nCore50 coverage:",
    f"{len(core50_present)}/50"
)


if len(
    core50_missing
) > 0:

    print(
        "Missing:",
        core50_missing
    )


assert len(
    core50_present
) == 50, (
    "Core50 does not have 50/50 coverage. "
    "Stop before Step31E."
)


# -----------------------------------------------------------------------------
# 8. Restore fixed alias map
# -----------------------------------------------------------------------------

CORE50_ALIAS_MAP = {
    "SGOL1": "SGO1",
    "SGOL2": "SGO2"
}


# -----------------------------------------------------------------------------
# 9. Write pause/resume handoff
# -----------------------------------------------------------------------------

handoff = {

    "project":
        "GSE326201 Step31 Independent scRNA Validation",

    "current_frozen_step":
        "Step31D2B",

    "next_step":
        "Step31E — Frozen MS3 Core50 Independent Validation",

    "broad_cell_universe": {
        "n_cells": 90980,
        "n_genes": 32732
    },

    "malignant_hepatic_universe": {
        "n_cells": 4688,
        "n_genes": 32732,
        "n_patients": 10,
        "n_samples": 10,
        "cluster_8": 3652,
        "cluster_20": 1036
    },

    "normal_hepatic_reference": {
        "n_cells": 415,
        "n_patients": 7
    },

    "cnv_like_evidence": {
        "non_tumor_median":
            0.05143716558814049,

        "tumor_median":
            0.08747104555368423,

        "paired_direction_consistency":
            "5/5 Tumor > NonTumor",

        "q99_used_for_primary_selection":
            False
    },

    "core50": {
        "frozen_membership":
            50,

        "biological_coverage":
            "50/50",

        "ensembl_mapping_column":
            str(
                CORE50_GENE_ID_COL
            ),

        "alias_map": {
            "SGOL1": "SGO1",
            "SGOL2": "SGO2"
        },

        "used_for_malignant_identification":
            False
    },

    "frozen_boundaries": [

        "Do not redefine malignant cells using Core50.",

        "Do not reselect Core50 genes.",

        "Do not reconstruct Original RiskScore in scRNA.",

        "Do not redefine MS1-MS8.",

        "Do not reselect Step29 regulators.",

        "Use patient-level inference where possible.",

        "Tumor-associated Epithelial/Cholangiocyte cells remain outside the primary malignant universe."
    ],

    "paths": {

        "broad_lock":
            str(
                BROAD_LOCK_PATH
            ),

        "malignant_lock":
            str(
                MALIGNANT_LOCK_PATH
            ),

        "full_d2b":
            str(
                FULL_D2B_PATH
            ),

        "core50_mapping":
            str(
                CORE50_MAPPING_PATH
            ),

        "gene_coordinates":
            str(
                GENE_COORD_PATH
            )
    }
}


with open(
    HANDOFF_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        handoff,
        f,
        ensure_ascii=False,
        indent=2
    )


# -----------------------------------------------------------------------------
# 10. Final resume report
# -----------------------------------------------------------------------------

print(
    "\n" + "=" * 110
)

print(
    "STEP31 RESUME STATE: READY"
)

print(
    "=" * 110
)


print(
    "\nFrozen through:"
)

print(
    "Step31D2B — Independent Malignant Hepatic Lock"
)


print(
    "\nMalignant universe:"
)

print(
    "Cells:",
    adata_malignant.n_obs
)

print(
    "Genes:",
    adata_malignant.n_vars
)

print(
    "Patients:",
    adata_malignant.obs[
        "patient_id"
    ].nunique()
)

print(
    "Samples:",
    adata_malignant.obs[
        "sample_id"
    ].nunique()
)


print(
    "\nFrozen Core50:"
)

print(
    "Mapping column:",
    CORE50_GENE_ID_COL
)

print(
    "Genes:",
    len(
        CORE50_ENSEMBL_IDS
    )
)

print(
    "Coverage:",
    f"{len(core50_present)}/50"
)

print(
    "Aliases:",
    CORE50_ALIAS_MAP
)


print(
    "\nHandoff:"
)

print(
    HANDOFF_PATH
)


print(
    """
After restarting Jupyter/kernel:

1. Run Cell160 only.
2. Confirm:
       STEP31 RESUME STATE: READY
       Core50 Coverage: 50/50
3. Continue directly with Cell161 / Step31E.

Do NOT rerun Step31A-D for routine continuation.
"""
)


print(
    "SAFE TO PAUSE / RESTART"
)

print("=" * 110)

## Step31E1 — Frozen MS3 Core50 portability and deterministic scoring

### Objective

Evaluate the frozen Step25 MS3 Conserved Core50 in the independently
defined GSE326201 malignant-hepatic universe.

### Frozen principles

- Core50 membership remains exactly 50 genes.
- No gene is reselected, replaced, weighted, or optimized.
- SGOL1/SGOL2 are represented by SGO1/SGO2 in this validation dataset,
  while original frozen symbols are retained for reporting.
- The malignant-cell universe was defined independently of Core50.
- Core50 is analyzed as a continuous program.
- No data-driven Core50-high cutoff is optimized in this step.
- No MS3-like state is declared yet.
- Patient-level consistency is prioritized over cell-level pseudoreplication.

In [ ]:
# =============================================================================
# Step31E1 Cell 162
# Restore and audit frozen Core50 identifiers
# =============================================================================

import pandas as pd
import numpy as np


assert "adata_malignant" in globals()
assert "core50_validation_map" in globals()
assert "CORE50_ENSEMBL_IDS" in globals()

assert adata_malignant.n_obs == 4688
assert len(CORE50_ENSEMBL_IDS) == 50


# -----------------------------------------------------------------------------
# Identify original frozen-symbol column
# Prefer the column containing SGOL1 / SGOL2.
# -----------------------------------------------------------------------------

symbol_candidate_rows = []


for col in core50_validation_map.columns:

    values = (
        core50_validation_map[col]
        .astype(str)
        .str.strip()
    )

    symbol_candidate_rows.append({

        "column":
            col,

        "n_unique":
            values.nunique(),

        "contains_SGOL1":
            bool(
                (
                    values
                    ==
                    "SGOL1"
                ).any()
            ),

        "contains_SGOL2":
            bool(
                (
                    values
                    ==
                    "SGOL2"
                ).any()
            ),

        "n_ensembl_like":
            int(
                values.str.match(
                    r"^ENSG\d+",
                    na=False
                ).sum()
            )
    })


symbol_candidate_audit = pd.DataFrame(
    symbol_candidate_rows
)


display(
    symbol_candidate_audit
)


preferred_symbol_cols = (
    symbol_candidate_audit.loc[
        symbol_candidate_audit[
            "contains_SGOL1"
        ]
        &
        symbol_candidate_audit[
            "contains_SGOL2"
        ],
        "column"
    ]
    .tolist()
)


if len(preferred_symbol_cols) > 0:

    CORE50_ORIGINAL_SYMBOL_COL = (
        preferred_symbol_cols[0]
    )

else:

    # fallback: 50 unique non-Ensembl strings
    fallback = (
        symbol_candidate_audit.loc[
            (
                symbol_candidate_audit[
                    "n_unique"
                ]
                ==
                50
            )
            &
            (
                symbol_candidate_audit[
                    "n_ensembl_like"
                ]
                ==
                0
            ),
            "column"
        ]
        .tolist()
    )

    CORE50_ORIGINAL_SYMBOL_COL = (
        fallback[0]
        if len(fallback) > 0
        else None
    )


assert CORE50_ORIGINAL_SYMBOL_COL is not None


CORE50_ORIGINAL_SYMBOLS = (
    core50_validation_map[
        CORE50_ORIGINAL_SYMBOL_COL
    ]
    .astype(str)
    .str.strip()
    .tolist()
)


assert len(
    CORE50_ORIGINAL_SYMBOLS
) == 50


assert len(
    set(
        CORE50_ORIGINAL_SYMBOLS
    )
) == 50


core50_frozen_table = pd.DataFrame({

    "frozen_symbol":
        CORE50_ORIGINAL_SYMBOLS,

    "validation_ensembl_id":
        CORE50_ENSEMBL_IDS
})


print(
    "Original frozen-symbol column:",
    CORE50_ORIGINAL_SYMBOL_COL
)

print(
    "Core50 genes:",
    len(
        core50_frozen_table
    )
)


display(
    core50_frozen_table
)


print(
    "\nCORE50 FROZEN IDENTIFIERS: PASS"
)

In [ ]:
# =============================================================================
# Step31E1 Cell 163
# Extract frozen Core50 expression matrix
# =============================================================================

import scipy.sparse as sp


X_core = (
    adata_malignant[
        :,
        CORE50_ENSEMBL_IDS
    ].X
)


if sp.issparse(
    X_core
):

    X_core = (
        X_core
        .toarray()
    )


X_core = np.asarray(
    X_core,
    dtype=np.float32
)


print(
    "Core50 expression matrix:",
    X_core.shape
)


assert X_core.shape == (
    4688,
    50
)


print(
    "Finite:",
    bool(
        np.isfinite(
            X_core
        ).all()
    )
)


print(
    "Minimum:",
    float(
        X_core.min()
    )
)

print(
    "Maximum:",
    float(
        X_core.max()
    )
)


print(
    "\nExpression source:"
)

print(
    "Uncorrected log-normalized expression"
)

print(
    "Harmony-corrected expression used:",
    False
)


print(
    "\nCORE50 EXPRESSION EXTRACTION: PASS"
)

In [ ]:
# =============================================================================
# Step31E1 Cell 164
# Frozen Core50 gene-level expression audit
# =============================================================================

core50_gene_mean = (
    X_core.mean(
        axis=0
    )
)


core50_gene_sd = (
    X_core.std(
        axis=0,
        ddof=0
    )
)


core50_gene_detection = (
    (
        X_core > 0
    )
    .mean(
        axis=0
    )
)


core50_gene_audit = (
    core50_frozen_table
    .copy()
)


core50_gene_audit[
    "mean_logexpr"
] = (
    core50_gene_mean
)


core50_gene_audit[
    "sd_logexpr"
] = (
    core50_gene_sd
)


core50_gene_audit[
    "detection_fraction"
] = (
    core50_gene_detection
)


core50_gene_audit[
    "zero_variance"
] = (
    core50_gene_sd
    ==
    0
)


display(
    core50_gene_audit.sort_values(
        "detection_fraction",
        ascending=False
    )
)


print(
    "\nGenes detected in at least one malignant cell:",
    int(
        (
            core50_gene_detection
            >
            0
        ).sum()
    ),
    "/50"
)


print(
    "Zero-variance genes:",
    int(
        (
            core50_gene_sd
            ==
            0
        ).sum()
    )
)

In [ ]:
# =============================================================================
# Step31E1 Cell 165
# Deterministic frozen Core50 scoring
# =============================================================================

gene_mean = (
    X_core.mean(
        axis=0,
        keepdims=True
    )
)


gene_sd = (
    X_core.std(
        axis=0,
        ddof=0,
        keepdims=True
    )
)


zero_sd = (
    gene_sd
    ==
    0
)


safe_gene_sd = (
    gene_sd.copy()
)


safe_gene_sd[
    zero_sd
] = 1.0


X_core_gene_z = (
    (
        X_core
        -
        gene_mean
    )
    /
    safe_gene_sd
)


# Zero-variance genes contribute exactly 0
X_core_gene_z[
    :,
    zero_sd.ravel()
] = 0.0


core50_zmean = (
    X_core_gene_z.mean(
        axis=1
    )
)


core50_mean_logexpr = (
    X_core.mean(
        axis=1
    )
)


core50_detected_n = (
    (
        X_core > 0
    )
    .sum(
        axis=1
    )
)


core50_detected_fraction = (
    core50_detected_n
    /
    50.0
)


adata_malignant.obs[
    "core50_zmean"
] = (
    core50_zmean
)


adata_malignant.obs[
    "core50_mean_logexpr"
] = (
    core50_mean_logexpr
)


adata_malignant.obs[
    "core50_detected_n"
] = (
    core50_detected_n
)


adata_malignant.obs[
    "core50_detected_fraction"
] = (
    core50_detected_fraction
)


print(
    "Primary score: core50_zmean"
)

print(
    "Sensitivity score: core50_mean_logexpr"
)


print(
    "\nCore50 z-mean:"
)

print(
    pd.Series(
        core50_zmean
    ).describe()
)


print(
    "\nDetected Core50 genes per cell:"
)

print(
    pd.Series(
        core50_detected_n
    ).describe()
)

In [ ]:
# =============================================================================
# Step31E1 Cell 166
# Patient-centered Core50 score
# =============================================================================

adata_malignant.obs[
    "core50_z_within_patient"
] = np.nan


for patient in (
    adata_malignant.obs[
        "patient_id"
    ]
    .astype(str)
    .unique()
):

    mask = (
        adata_malignant.obs[
            "patient_id"
        ].astype(str)
        ==
        patient
    )


    values = (
        adata_malignant.obs.loc[
            mask,
            "core50_zmean"
        ]
        .astype(float)
        .to_numpy()
    )


    mu = float(
        values.mean()
    )


    sd = float(
        values.std(
            ddof=0
        )
    )


    if sd > 0:

        centered = (
            (
                values
                -
                mu
            )
            /
            sd
        )

    else:

        centered = np.zeros_like(
            values
        )


    adata_malignant.obs.loc[
        mask,
        "core50_z_within_patient"
    ] = centered


assert (
    adata_malignant.obs[
        "core50_z_within_patient"
    ]
    .notna()
    .all()
)


print(
    "Patient-centered Core50 score: PASS"
)

In [ ]:
# =============================================================================
# Step31E1 Cell 167
# Patient-level Core50 summary
# =============================================================================

core50_patient_summary = (
    adata_malignant.obs
    .groupby(
        [
            "patient_id",
            "sample_id"
        ],
        observed=True
    )
    .agg(

        n_cells=(
            "core50_zmean",
            "size"
        ),

        mean_core50_zmean=(
            "core50_zmean",
            "mean"
        ),

        median_core50_zmean=(
            "core50_zmean",
            "median"
        ),

        q25_core50_zmean=(
            "core50_zmean",
            lambda x:
                x.quantile(
                    0.25
                )
        ),

        q75_core50_zmean=(
            "core50_zmean",
            lambda x:
                x.quantile(
                    0.75
                )
        ),

        mean_core50_logexpr=(
            "core50_mean_logexpr",
            "mean"
        ),

        median_detected_genes=(
            "core50_detected_n",
            "median"
        )
    )
    .reset_index()
)


core50_patient_summary[
    "iqr_core50_zmean"
] = (
    core50_patient_summary[
        "q75_core50_zmean"
    ]
    -
    core50_patient_summary[
        "q25_core50_zmean"
    ]
)


display(
    core50_patient_summary
)

In [ ]:
# =============================================================================
# Step31E1 Cell 168
# Gene-by-patient Core50 portability audit
# =============================================================================

patient_ids = (
    adata_malignant.obs[
        "patient_id"
    ]
    .astype(str)
    .unique()
    .tolist()
)


gene_patient_rows = []


for patient in patient_ids:

    mask = (
        adata_malignant.obs[
            "patient_id"
        ].astype(str)
        ==
        patient
    ).to_numpy()


    X_patient = (
        X_core[
            mask,
            :
        ]
    )


    for j in range(
        50
    ):

        values = (
            X_patient[
                :,
                j
            ]
        )


        gene_patient_rows.append({

            "patient_id":
                patient,

            "frozen_symbol":
                CORE50_ORIGINAL_SYMBOLS[
                    j
                ],

            "validation_ensembl_id":
                CORE50_ENSEMBL_IDS[
                    j
                ],

            "n_cells":
                int(
                    len(
                        values
                    )
                ),

            "detection_fraction":
                float(
                    (
                        values > 0
                    ).mean()
                ),

            "mean_logexpr":
                float(
                    values.mean()
                ),

            "median_logexpr":
                float(
                    np.median(
                        values
                    )
                )
        })


core50_gene_patient = pd.DataFrame(
    gene_patient_rows
)


display(
    core50_gene_patient.head(
        20
    )
)

In [ ]:
# =============================================================================
# Step31E1 Cell 169
# Cross-patient gene portability summary
# =============================================================================

core50_gene_patient_summary = (
    core50_gene_patient
    .groupby(
        [
            "frozen_symbol",
            "validation_ensembl_id"
        ],
        observed=True
    )
    .agg(

        n_patients_any_detection=(
            "detection_fraction",
            lambda x:
                int(
                    (
                        x > 0
                    ).sum()
                )
        ),

        n_patients_detection_ge_5pct=(
            "detection_fraction",
            lambda x:
                int(
                    (
                        x >= 0.05
                    ).sum()
                )
        ),

        median_patient_detection_fraction=(
            "detection_fraction",
            "median"
        ),

        min_patient_detection_fraction=(
            "detection_fraction",
            "min"
        ),

        max_patient_detection_fraction=(
            "detection_fraction",
            "max"
        ),

        median_patient_mean_logexpr=(
            "mean_logexpr",
            "median"
        )
    )
    .reset_index()
)


display(
    core50_gene_patient_summary.sort_values(
        [
            "n_patients_detection_ge_5pct",
            "median_patient_detection_fraction"
        ],
        ascending=[
            True,
            True
        ]
    )
)


print(
    "\nGenes detected in all 10 patients:"
)

print(
    int(
        (
            core50_gene_patient_summary[
                "n_patients_any_detection"
            ]
            ==
            10
        ).sum()
    ),
    "/50"
)


print(
    "\nGenes detected in >=5% cells in all 10 patients:"
)

print(
    int(
        (
            core50_gene_patient_summary[
                "n_patients_detection_ge_5pct"
            ]
            ==
            10
        ).sum()
    ),
    "/50"
)

In [ ]:
# =============================================================================
# Step31E1 Cell 170
# Descriptive Core50 distribution across malignant hepatic clusters
# =============================================================================

core50_cluster_summary = (
    adata_malignant.obs
    .groupby(
        "leiden_harmony_r08",
        observed=True
    )
    .agg(

        n_cells=(
            "core50_zmean",
            "size"
        ),

        mean_core50_zmean=(
            "core50_zmean",
            "mean"
        ),

        median_core50_zmean=(
            "core50_zmean",
            "median"
        ),

        median_core50_logexpr=(
            "core50_mean_logexpr",
            "median"
        ),

        median_detected_genes=(
            "core50_detected_n",
            "median"
        )
    )
    .reset_index()
)


display(
    core50_cluster_summary
)

In [ ]:
# =============================================================================
# Step31E1 Cell 171
# Primary vs sensitivity scoring consistency
# =============================================================================

from scipy.stats import spearmanr


rho_all, p_all = spearmanr(
    adata_malignant.obs[
        "core50_zmean"
    ],
    adata_malignant.obs[
        "core50_mean_logexpr"
    ]
)


score_consistency_rows = []


for patient in patient_ids:

    sub = (
        adata_malignant.obs.loc[
            adata_malignant.obs[
                "patient_id"
            ].astype(str)
            ==
            patient
        ]
    )


    rho, p = spearmanr(
        sub[
            "core50_zmean"
        ],
        sub[
            "core50_mean_logexpr"
        ]
    )


    score_consistency_rows.append({

        "patient_id":
            patient,

        "n_cells":
            len(
                sub
            ),

        "spearman_rho":
            rho,

        "p_value":
            p
    })


core50_score_consistency = pd.DataFrame(
    score_consistency_rows
)


print(
    "Overall Spearman rho:",
    rho_all
)


display(
    core50_score_consistency
)

In [ ]:
# =============================================================================
# Step31E1 Cell 172
# Save frozen Core50-scored malignant universe
# =============================================================================

CORE50_SCORED_PATH = (
    DATA_DIR /
    "GSE326201_STEP31E1_CORE50_SCORED_MALIGNANT.h5ad"
)


adata_malignant.write_h5ad(
    CORE50_SCORED_PATH,
    compression="gzip"
)


core50_gene_audit.to_csv(
    AUDIT_DIR /
    "Step31E1_Core50GeneExpressionAudit.csv",
    index=False
)


core50_patient_summary.to_csv(
    AUDIT_DIR /
    "Step31E1_Core50PatientSummary.csv",
    index=False
)


core50_gene_patient.to_csv(
    AUDIT_DIR /
    "Step31E1_Core50GeneByPatient.csv",
    index=False
)


core50_gene_patient_summary.to_csv(
    AUDIT_DIR /
    "Step31E1_Core50GenePortabilitySummary.csv",
    index=False
)


core50_cluster_summary.to_csv(
    AUDIT_DIR /
    "Step31E1_Core50ClusterSummary.csv",
    index=False
)


core50_score_consistency.to_csv(
    AUDIT_DIR /
    "Step31E1_Core50ScoreConsistency.csv",
    index=False
)


print(
    "Saved:"
)

print(
    CORE50_SCORED_PATH
)

print(
    "Exists:",
    CORE50_SCORED_PATH.exists()
)

In [ ]:
# =============================================================================
# Step31E1 Cell 173
# REPORT — Frozen Core50 portability and scoring
# =============================================================================

print("=" * 120)

print(
    "STEP31E1 — FROZEN MS3 CORE50 PORTABILITY AND SCORING"
)

print("=" * 120)


print(
    "\n[1] INDEPENDENT MALIGNANT UNIVERSE"
)

print(
    "Cells:",
    adata_malignant.n_obs
)

print(
    "Genes:",
    adata_malignant.n_vars
)

print(
    "Patients:",
    adata_malignant.obs[
        "patient_id"
    ].nunique()
)

print(
    "Samples:",
    adata_malignant.obs[
        "sample_id"
    ].nunique()
)


print(
    "\n[2] FROZEN CORE50"
)

print(
    "Frozen genes:",
    len(
        CORE50_ENSEMBL_IDS
    )
)

print(
    "Biological mapping coverage:",
    "50/50"
)

print(
    "Genes detected in at least one malignant cell:",
    int(
        (
            core50_gene_detection
            >
            0
        ).sum()
    ),
    "/50"
)

print(
    "Zero-variance genes:",
    int(
        (
            core50_gene_sd
            ==
            0
        ).sum()
    )
)


print(
    "\n[3] CELL-LEVEL SCORE DISTRIBUTION"
)

print(
    adata_malignant.obs[
        [
            "core50_zmean",
            "core50_mean_logexpr",
            "core50_detected_n"
        ]
    ]
    .describe()
)


print(
    "\n[4] PATIENT-LEVEL SCORE DISTRIBUTION"
)

display(
    core50_patient_summary
)


print(
    "\n[5] CROSS-PATIENT GENE PORTABILITY"
)

print(
    "Genes detected in all 10 patients:",
    int(
        (
            core50_gene_patient_summary[
                "n_patients_any_detection"
            ]
            ==
            10
        ).sum()
    ),
    "/50"
)


print(
    "Genes detected in >=5% of malignant cells in all 10 patients:",
    int(
        (
            core50_gene_patient_summary[
                "n_patients_detection_ge_5pct"
            ]
            ==
            10
        ).sum()
    ),
    "/50"
)


display(
    core50_gene_patient_summary.sort_values(
        [
            "n_patients_detection_ge_5pct",
            "median_patient_detection_fraction"
        ],
        ascending=[
            True,
            True
        ]
    )
)


print(
    "\n[6] MALIGNANT HEPATIC CLUSTERS"
)

display(
    core50_cluster_summary
)


print(
    "\n[7] SCORE ROBUSTNESS"
)

print(
    "Overall primary-vs-sensitivity Spearman rho:",
    rho_all
)


display(
    core50_score_consistency
)


print(
    "\n[8] FIREWALL / CLAIM BOUNDARY"
)

print(
"""
The Step25 MS3 Conserved Core50 membership remains unchanged.

No genes were reselected, replaced, weighted, optimized, or removed.

The primary Core50 activity score is the arithmetic mean of gene-wise
standardized expression across all 50 frozen genes.

The direct mean log-expression score is retained as a sensitivity measure.

No Core50-high cutoff has been optimized.

No new MS3 state has been defined.

No cell was selected into the malignant universe using Core50 information.

Patient-centered scores are retained only for within-patient validation.

The next step will test whether the frozen Core50 activity recapitulates
the prespecified MS3 proliferative / G2M / spindle-CIN biology across
independent patients.
"""
)


print(
    "\n[9] FILE"
)

print(
    CORE50_SCORED_PATH
)

print(
    "Exists:",
    CORE50_SCORED_PATH.exists()
)


print(
    "\nSTEP31E1: COMPLETE"
)

print(
    "NEXT: Step31E2 — independent biological validation of the frozen MS3 Core50."
)

print("=" * 120)

## Step31E2A — Orthogonal biological validation of frozen MS3 Core50

### Objective

Test whether the frozen MS3 Core50 activity is associated with the
prespecified MS3 biological phenotype in the independent malignant-hepatic
universe.

### Validation principles

- Core50 membership remains frozen and unchanged.
- Validation gene programs are independent of Core50.
- Any gene overlapping Core50 is removed from the validation program,
  not from Core50.
- No validation genes are selected using GSE326201 association results.
- Primary inference is based on within-patient direction consistency.
- Cell-level pooled correlations are descriptive only.
- Technical complexity is explicitly audited because Core50 expression
  is sparse in this independent dataset.
- No Core50-high cutoff or new MS3 cluster is defined.v

In [ ]:
# =============================================================================
# Step31E2A Cell 175
# Prespecified orthogonal biological validation programs
# =============================================================================

VALIDATION_PROGRAMS = {

    # Canonical G2/M biology.
    # Core50 overlap will be removed automatically later.
    "G2M_Independent": [
        "HMGB2",
        "CDK1",
        "NUSAP1",
        "UBE2C",
        "BIRC5",
        "TPX2",
        "TOP2A",
        "NDC80",
        "CKS2",
        "NUF2",
        "CKS1B",
        "MKI67",
        "TMPO",
        "CENPF",
        "TACC3",
        "SMC4",
        "CCNB2",
        "CKAP2L",
        "CKAP2",
        "AURKB",
        "BUB1",
        "KIF11",
        "ANP32E",
        "TUBB4B",
        "GTSE1",
        "KIF20B",
        "HJURP",
        "CDCA3",
        "CDC20",
        "TTK",
        "CDC25C",
        "KIF2C",
        "RANGAP1",
        "NCAPD2",
        "DLGAP5",
        "CDCA2",
        "CDCA8",
        "ECT2",
        "KIF23",
        "HMMR",
        "AURKA",
        "PSRC1",
        "ANLN",
        "LBR",
        "CKAP5",
        "CENPE",
        "CTCF",
        "NEK2",
        "G2E3",
        "GAS2L3",
        "CBX5",
        "CENPA"
    ],

    "Spindle_Chromosome_Independent": [
        "KIF11",
        "KIF15",
        "KIF18A",
        "KIF18B",
        "SKA1",
        "SKA2",
        "SKA3",
        "ZWILCH",
        "ZWINT",
        "MAD2L1",
        "MAD2L2",
        "SPC24",
        "SPC25",
        "CENPK",
        "CENPU",
        "CENPW",
        "SMC2",
        "SMC4",
        "NCAPH",
        "NCAPD2",
        "NCAPD3",
        "DSN1",
        "MIS18A",
        "MIS18BP1",
        "NUP107",
        "NUP133",
        "AHCTF1",
        "RCC2",
        "ECT2"
    ],

    "Mitotic_Entry_Independent": [
        "CDK1",
        "CCNA2",
        "CDC25B",
        "CDC25C",
        "WEE1",
        "PKMYT1",
        "FBXO5",
        "CCNF",
        "CKS1B",
        "CKS2",
        "CDKN3",
        "G2E3",
        "MELK",
        "CDC7",
        "DBF4",
        "ESCO2"
    ],

    # Opposing differentiated hepatic programs.
    "Hepatic_Differentiation": [
        "ALB",
        "APOA1",
        "APOA2",
        "TTR",
        "ASGR1",
        "ASGR2",
        "CPS1",
        "ARG1",
        "OTC",
        "PCK1",
        "G6PC",
        "CYP2E1",
        "CYP3A4",
        "SLC27A5",
        "HP",
        "TF",
        "AHSG",
        "FGA",
        "FGB",
        "FGG"
    ],

    "Bile_Acid_Metabolism": [
        "CYP7A1",
        "CYP8B1",
        "BAAT",
        "SLC10A1",
        "ABCB11",
        "ABCB4",
        "SLC27A5",
        "HSD17B4",
        "AKR1D1",
        "HSD3B7",
        "AMACR",
        "ACOX2",
        "EPHX1",
        "UGT2B4",
        "UGT2B7",
        "SULT2A1",
        "SLCO1B1",
        "SLCO1B3"
    ]
}


EXPECTED_DIRECTION = {

    "G2M_Independent":
        "positive",

    "Spindle_Chromosome_Independent":
        "positive",

    "Mitotic_Entry_Independent":
        "positive",

    "Hepatic_Differentiation":
        "negative",

    "Bile_Acid_Metabolism":
        "negative"
}


print(
    "Validation programs:",
    len(
        VALIDATION_PROGRAMS
    )
)

for k, v in VALIDATION_PROGRAMS.items():

    print(
        f"{k}: {len(v)} requested genes"
    )

In [ ]:
# =============================================================================
# Step31E2A Cell 176
# Map validation programs and remove Core50 overlap
# =============================================================================

import pandas as pd
import numpy as np


# Load the frozen coordinate/symbol resource generated in Step31D
gene_coordinates = pd.read_csv(
    GENE_COORD_PATH,
    dtype={
        "gene_id": str,
        "symbol": str,
        "chromosome": str
    }
)


malignant_gene_universe = set(
    adata_malignant.var_names
    .astype(str)
)


coord_for_mapping = (
    gene_coordinates.loc[
        gene_coordinates[
            "gene_id"
        ].astype(str)
        .isin(
            malignant_gene_universe
        )
        &
        gene_coordinates[
            "symbol"
        ].notna()
    ][
        [
            "gene_id",
            "symbol"
        ]
    ]
    .copy()
)


coord_for_mapping[
    "gene_id"
] = (
    coord_for_mapping[
        "gene_id"
    ].astype(str)
)


coord_for_mapping[
    "symbol"
] = (
    coord_for_mapping[
        "symbol"
    ]
    .astype(str)
    .str.strip()
)


symbol_to_ids = (
    coord_for_mapping
    .groupby(
        "symbol"
    )[
        "gene_id"
    ]
    .apply(
        lambda x:
            sorted(
                set(
                    x
                )
            )
    )
    .to_dict()
)


core50_id_set = set(
    CORE50_ENSEMBL_IDS
)


VALIDATION_PROGRAM_IDS = {}

validation_mapping_rows = []


for program, symbols in VALIDATION_PROGRAMS.items():

    final_ids = []


    for symbol in symbols:

        hits = (
            symbol_to_ids.get(
                symbol,
                []
            )
        )


        if len(hits) == 0:

            status = (
                "missing"
            )

            selected_id = None


        elif len(hits) > 1:

            status = (
                "ambiguous"
            )

            selected_id = None


        else:

            selected_id = (
                hits[0]
            )


            if selected_id in core50_id_set:

                status = (
                    "removed_core50_overlap"
                )

            else:

                status = (
                    "retained"
                )

                final_ids.append(
                    selected_id
                )


        validation_mapping_rows.append({

            "program":
                program,

            "requested_symbol":
                symbol,

            "selected_ensembl_id":
                selected_id,

            "mapping_status":
                status
        })


    VALIDATION_PROGRAM_IDS[
        program
    ] = list(
        dict.fromkeys(
            final_ids
        )
    )


validation_mapping_audit = pd.DataFrame(
    validation_mapping_rows
)


validation_program_coverage = (
    validation_mapping_audit
    .groupby(
        "program",
        observed=True
    )
    .agg(

        n_requested=(
            "requested_symbol",
            "size"
        ),

        n_retained=(
            "mapping_status",
            lambda x:
                int(
                    (
                        x
                        ==
                        "retained"
                    ).sum()
                )
        ),

        n_core50_overlap_removed=(
            "mapping_status",
            lambda x:
                int(
                    (
                        x
                        ==
                        "removed_core50_overlap"
                    ).sum()
                )
        ),

        n_missing=(
            "mapping_status",
            lambda x:
                int(
                    (
                        x
                        ==
                        "missing"
                    ).sum()
                )
        ),

        n_ambiguous=(
            "mapping_status",
            lambda x:
                int(
                    (
                        x
                        ==
                        "ambiguous"
                    ).sum()
                )
        )
    )
    .reset_index()
)


display(
    validation_program_coverage
)


for program, ids in (
    VALIDATION_PROGRAM_IDS.items()
):

    print(
        program,
        "->",
        len(ids),
        "orthogonal genes"
    )


assert all(
    len(ids) >= 5
    for ids in VALIDATION_PROGRAM_IDS.values()
), (
    "One or more validation programs have <5 usable orthogonal genes."
)


assert all(
    len(
        set(ids)
        &
        core50_id_set
    )
    ==
    0
    for ids in VALIDATION_PROGRAM_IDS.values()
)


print(
    "\nCORE50 OVERLAP FIREWALL: PASS"
)

In [ ]:
# =============================================================================
# Step31E2A Cell 177
# Validation-program expression coverage
# =============================================================================

import scipy.sparse as sp


validation_gene_rows = []


for program, ids in (
    VALIDATION_PROGRAM_IDS.items()
):

    X = (
        adata_malignant[
            :,
            ids
        ].X
    )


    if sp.issparse(
        X
    ):

        X = X.toarray()


    X = np.asarray(
        X,
        dtype=np.float32
    )


    detection = (
        X > 0
    ).mean(
        axis=0
    )


    sd = (
        X.std(
            axis=0,
            ddof=0
        )
    )


    for gene_id, det, gene_sd in zip(
        ids,
        detection,
        sd
    ):

        validation_gene_rows.append({

            "program":
                program,

            "gene_id":
                gene_id,

            "detection_fraction":
                float(
                    det
                ),

            "sd_logexpr":
                float(
                    gene_sd
                )
        })


validation_gene_audit = pd.DataFrame(
    validation_gene_rows
)


validation_detection_summary = (
    validation_gene_audit
    .groupby(
        "program",
        observed=True
    )
    .agg(

        n_genes=(
            "gene_id",
            "size"
        ),

        median_detection_fraction=(
            "detection_fraction",
            "median"
        ),

        n_detected_any=(
            "detection_fraction",
            lambda x:
                int(
                    (
                        x > 0
                    ).sum()
                )
        ),

        n_zero_variance=(
            "sd_logexpr",
            lambda x:
                int(
                    (
                        x == 0
                    ).sum()
                )
        )
    )
    .reset_index()
)


display(
    validation_detection_summary
)

In [ ]:
# =============================================================================
# Step31E2A Cell 178
# Score orthogonal validation programs
# =============================================================================

def score_gene_zmean(
    adata,
    gene_ids
):

    X = (
        adata[
            :,
            gene_ids
        ].X
    )


    if sp.issparse(
        X
    ):

        X = X.toarray()


    X = np.asarray(
        X,
        dtype=np.float32
    )


    mu = (
        X.mean(
            axis=0,
            keepdims=True
        )
    )


    sd = (
        X.std(
            axis=0,
            ddof=0,
            keepdims=True
        )
    )


    zero_sd = (
        sd == 0
    )


    safe_sd = (
        sd.copy()
    )

    safe_sd[
        zero_sd
    ] = 1.0


    Z = (
        (
            X - mu
        )
        /
        safe_sd
    )


    Z[
        :,
        zero_sd.ravel()
    ] = 0.0


    score = (
        Z.mean(
            axis=1
        )
    )


    detected_n = (
        X > 0
    ).sum(
        axis=1
    )


    return (
        score,
        detected_n
    )


for program, ids in (
    VALIDATION_PROGRAM_IDS.items()
):

    score, detected_n = (
        score_gene_zmean(
            adata_malignant,
            ids
        )
    )


    adata_malignant.obs[
        f"program_{program}"
    ] = score


    adata_malignant.obs[
        f"program_{program}_detected_n"
    ] = detected_n


print(
    "Orthogonal phenotype scores: COMPLETE"
)

In [ ]:
# =============================================================================
# Step31E2A Cell 179
# Technical-complexity audit
# =============================================================================

from scipy.stats import spearmanr


TECHNICAL_COVARIATES = [
    x
    for x in [
        "n_genes_by_counts",
        "total_counts",
        "pct_counts_mt"
    ]
    if x in adata_malignant.obs.columns
]


print(
    "Available technical covariates:",
    TECHNICAL_COVARIATES
)


technical_rows = []


for patient in (
    adata_malignant.obs[
        "patient_id"
    ]
    .astype(str)
    .unique()
):

    sub = (
        adata_malignant.obs.loc[
            adata_malignant.obs[
                "patient_id"
            ].astype(str)
            ==
            patient
        ]
    )


    for covariate in (
        TECHNICAL_COVARIATES
    ):

        rho, p = spearmanr(
            sub[
                "core50_zmean"
            ],
            sub[
                covariate
            ],
            nan_policy="omit"
        )


        technical_rows.append({

            "patient_id":
                patient,

            "n_cells":
                len(
                    sub
                ),

            "covariate":
                covariate,

            "spearman_rho":
                rho,

            "p_value":
                p
        })


core50_technical_audit = pd.DataFrame(
    technical_rows
)


display(
    core50_technical_audit
)

In [ ]:
# =============================================================================
# Step31E2A Cell 180
# Pooled descriptive phenotype correlations
# =============================================================================

pooled_rows = []


for program in (
    VALIDATION_PROGRAM_IDS.keys()
):

    rho, p = spearmanr(
        adata_malignant.obs[
            "core50_zmean"
        ],
        adata_malignant.obs[
            f"program_{program}"
        ],
        nan_policy="omit"
    )


    pooled_rows.append({

        "program":
            program,

        "expected_direction":
            EXPECTED_DIRECTION[
                program
            ],

        "spearman_rho":
            rho,

        "p_value":
            p
    })


pooled_core50_biology = pd.DataFrame(
    pooled_rows
)


display(
    pooled_core50_biology
)

In [ ]:
# =============================================================================
# Step31E2A Cell 181
# Within-patient Core50-to-biology correlations
# =============================================================================

patient_biology_rows = []


patient_ids = (
    adata_malignant.obs[
        "patient_id"
    ]
    .astype(str)
    .unique()
    .tolist()
)


for patient in patient_ids:

    sub = (
        adata_malignant.obs.loc[
            adata_malignant.obs[
                "patient_id"
            ].astype(str)
            ==
            patient
        ]
    )


    for program in (
        VALIDATION_PROGRAM_IDS.keys()
    ):

        rho, p = spearmanr(
            sub[
                "core50_zmean"
            ],
            sub[
                f"program_{program}"
            ],
            nan_policy="omit"
        )


        patient_biology_rows.append({

            "patient_id":
                patient,

            "n_cells":
                len(
                    sub
                ),

            "program":
                program,

            "expected_direction":
                EXPECTED_DIRECTION[
                    program
                ],

            "spearman_rho":
                rho,

            "p_value":
                p
        })


core50_biology_by_patient = (
    pd.DataFrame(
        patient_biology_rows
    )
)


display(
    core50_biology_by_patient
)

In [ ]:
# =============================================================================
# Step31E2A Cell 182
# Patient-level direction consistency
# =============================================================================

def direction_supported(
    rho,
    expected
):

    if pd.isna(
        rho
    ):

        return False


    if expected == "positive":

        return rho > 0


    if expected == "negative":

        return rho < 0


    return False


core50_biology_by_patient[
    "direction_supported"
] = (
    core50_biology_by_patient.apply(
        lambda r:
            direction_supported(
                r[
                    "spearman_rho"
                ],
                r[
                    "expected_direction"
                ]
            ),
        axis=1
    )
)


biology_direction_summary = (
    core50_biology_by_patient
    .groupby(
        [
            "program",
            "expected_direction"
        ],
        observed=True
    )
    .agg(

        n_patients=(
            "patient_id",
            "nunique"
        ),

        n_direction_supported=(
            "direction_supported",
            "sum"
        ),

        median_rho=(
            "spearman_rho",
            "median"
        ),

        min_rho=(
            "spearman_rho",
            "min"
        ),

        max_rho=(
            "spearman_rho",
            "max"
        )
    )
    .reset_index()
)


biology_direction_summary[
    "direction_fraction"
] = (
    biology_direction_summary[
        "n_direction_supported"
    ]
    /
    biology_direction_summary[
        "n_patients"
    ]
)


display(
    biology_direction_summary
)

In [ ]:
# =============================================================================
# Step31E2A Cell 183
# Sensitivity analysis among cells with detectable Core50 expression
# =============================================================================

core50_detectable_mask = (
    adata_malignant.obs[
        "core50_detected_n"
    ]
    >=
    1
)


print(
    "Cells with >=1 Core50 gene detected:",
    int(
        core50_detectable_mask.sum()
    ),
    "/",
    adata_malignant.n_obs
)


detectable_patient_rows = []


for patient in patient_ids:

    mask = (
        (
            adata_malignant.obs[
                "patient_id"
            ].astype(str)
            ==
            patient
        )
        &
        core50_detectable_mask
    )


    sub = (
        adata_malignant.obs.loc[
            mask
        ]
    )


    if len(
        sub
    ) < 10:

        continue


    for program in (
        VALIDATION_PROGRAM_IDS.keys()
    ):

        rho, p = spearmanr(
            sub[
                "core50_zmean"
            ],
            sub[
                f"program_{program}"
            ],
            nan_policy="omit"
        )


        detectable_patient_rows.append({

            "patient_id":
                patient,

            "n_cells":
                len(
                    sub
                ),

            "program":
                program,

            "expected_direction":
                EXPECTED_DIRECTION[
                    program
                ],

            "spearman_rho":
                rho,

            "p_value":
                p
        })


core50_biology_detectable = (
    pd.DataFrame(
        detectable_patient_rows
    )
)


if len(
    core50_biology_detectable
) > 0:

    core50_biology_detectable[
        "direction_supported"
    ] = (
        core50_biology_detectable.apply(
            lambda r:
                direction_supported(
                    r[
                        "spearman_rho"
                    ],
                    r[
                        "expected_direction"
                    ]
                ),
            axis=1
        )
    )


display(
    core50_biology_detectable
)

In [ ]:
# =============================================================================
# Step31E2A Cell 184
# Technical-complexity-controlled partial Spearman sensitivity
# =============================================================================

from scipy.stats import rankdata


MIN_PARTIAL_N = 20


def rank_residualize(
    y,
    covariate_matrix
):

    y = np.asarray(
        y,
        dtype=float
    )


    y_rank = rankdata(
        y,
        method="average"
    )


    X_cols = [
        np.ones(
            len(
                y
            )
        )
    ]


    for j in range(
        covariate_matrix.shape[1]
    ):

        cov_rank = rankdata(
            covariate_matrix[
                :,
                j
            ],
            method="average"
        )

        X_cols.append(
            cov_rank
        )


    X = np.column_stack(
        X_cols
    )


    beta = np.linalg.lstsq(
        X,
        y_rank,
        rcond=None
    )[0]


    residual = (
        y_rank
        -
        X @ beta
    )


    return residual


partial_rows = []


if len(
    TECHNICAL_COVARIATES
) > 0:

    for patient in patient_ids:

        sub = (
            adata_malignant.obs.loc[
                adata_malignant.obs[
                    "patient_id"
                ].astype(str)
                ==
                patient
            ]
            .copy()
        )


        needed = (
            [
                "core50_zmean"
            ]
            +
            TECHNICAL_COVARIATES
            +
            [
                f"program_{x}"
                for x in
                VALIDATION_PROGRAM_IDS.keys()
            ]
        )


        sub = (
            sub[
                needed
            ]
            .dropna()
        )


        if len(
            sub
        ) < MIN_PARTIAL_N:

            continue


        C = (
            sub[
                TECHNICAL_COVARIATES
            ]
            .astype(float)
            .to_numpy()
        )


        core_resid = (
            rank_residualize(
                sub[
                    "core50_zmean"
                ].to_numpy(),
                C
            )
        )


        for program in (
            VALIDATION_PROGRAM_IDS.keys()
        ):

            phenotype_resid = (
                rank_residualize(
                    sub[
                        f"program_{program}"
                    ].to_numpy(),
                    C
                )
            )


            rho, p = spearmanr(
                core_resid,
                phenotype_resid
            )


            partial_rows.append({

                "patient_id":
                    patient,

                "n_cells":
                    len(
                        sub
                    ),

                "program":
                    program,

                "expected_direction":
                    EXPECTED_DIRECTION[
                        program
                    ],

                "partial_spearman_rho":
                    rho,

                "p_value":
                    p
            })


partial_core50_biology = (
    pd.DataFrame(
        partial_rows
    )
)


if len(
    partial_core50_biology
) > 0:

    partial_core50_biology[
        "direction_supported"
    ] = (
        partial_core50_biology.apply(
            lambda r:
                direction_supported(
                    r[
                        "partial_spearman_rho"
                    ],
                    r[
                        "expected_direction"
                    ]
                ),
            axis=1
        )
    )


display(
    partial_core50_biology
)

In [ ]:
# =============================================================================
# Step31E2A Cell 185
# Partial-correlation direction summary
# =============================================================================

if len(
    partial_core50_biology
) > 0:

    partial_direction_summary = (
        partial_core50_biology
        .groupby(
            [
                "program",
                "expected_direction"
            ],
            observed=True
        )
        .agg(

            n_patients=(
                "patient_id",
                "nunique"
            ),

            n_direction_supported=(
                "direction_supported",
                "sum"
            ),

            median_partial_rho=(
                "partial_spearman_rho",
                "median"
            ),

            min_partial_rho=(
                "partial_spearman_rho",
                "min"
            ),

            max_partial_rho=(
                "partial_spearman_rho",
                "max"
            )
        )
        .reset_index()
    )


    partial_direction_summary[
        "direction_fraction"
    ] = (
        partial_direction_summary[
            "n_direction_supported"
        ]
        /
        partial_direction_summary[
            "n_patients"
        ]
    )


else:

    partial_direction_summary = (
        pd.DataFrame()
    )


display(
    partial_direction_summary
)

In [ ]:
# =============================================================================
# Step31E2A Cell 186
# Save orthogonal biological-validation audit
# =============================================================================

E2A_PATH = (
    DATA_DIR /
    "GSE326201_STEP31E2A_CORE50_ORTHOGONAL_BIOLOGY.h5ad"
)


adata_malignant.write_h5ad(
    E2A_PATH,
    compression="gzip"
)


validation_mapping_audit.to_csv(
    AUDIT_DIR /
    "Step31E2A_ValidationProgramMapping.csv",
    index=False
)


validation_program_coverage.to_csv(
    AUDIT_DIR /
    "Step31E2A_ValidationProgramCoverage.csv",
    index=False
)


validation_gene_audit.to_csv(
    AUDIT_DIR /
    "Step31E2A_ValidationGeneDetection.csv",
    index=False
)


pooled_core50_biology.to_csv(
    AUDIT_DIR /
    "Step31E2A_PooledCore50BiologyCorrelation.csv",
    index=False
)


core50_biology_by_patient.to_csv(
    AUDIT_DIR /
    "Step31E2A_PatientCore50BiologyCorrelation.csv",
    index=False
)


biology_direction_summary.to_csv(
    AUDIT_DIR /
    "Step31E2A_PatientDirectionSummary.csv",
    index=False
)


core50_technical_audit.to_csv(
    AUDIT_DIR /
    "Step31E2A_Core50TechnicalComplexityAudit.csv",
    index=False
)


core50_biology_detectable.to_csv(
    AUDIT_DIR /
    "Step31E2A_Core50DetectableCellSensitivity.csv",
    index=False
)


partial_core50_biology.to_csv(
    AUDIT_DIR /
    "Step31E2A_PartialSpearmanByPatient.csv",
    index=False
)


partial_direction_summary.to_csv(
    AUDIT_DIR /
    "Step31E2A_PartialSpearmanDirectionSummary.csv",
    index=False
)


print(
    "Saved:"
)

print(
    E2A_PATH
)

print(
    "Exists:",
    E2A_PATH.exists()
)

In [ ]:
# =============================================================================
# Step31E2A Cell 187
# REPORT — orthogonal biological validation
# =============================================================================

print("=" * 120)

print(
    "STEP31E2A — ORTHOGONAL BIOLOGICAL VALIDATION OF FROZEN MS3 CORE50"
)

print("=" * 120)


print(
    "\n[1] MALIGNANT UNIVERSE"
)

print(
    "Cells:",
    adata_malignant.n_obs
)

print(
    "Patients:",
    adata_malignant.obs[
        "patient_id"
    ].nunique()
)


print(
    "\n[2] CORE50 SPARSITY CONTEXT"
)

print(
    "Median Core50 genes detected per cell:",
    float(
        adata_malignant.obs[
            "core50_detected_n"
        ].median()
    )
)

print(
    "Cells with >=1 Core50 gene detected:",
    int(
        (
            adata_malignant.obs[
                "core50_detected_n"
            ]
            >=
            1
        ).sum()
    ),
    "/",
    adata_malignant.n_obs
)


print(
    "\n[3] ORTHOGONAL PROGRAM COVERAGE"
)

display(
    validation_program_coverage
)


print(
    "\n[4] PROGRAM EXPRESSION COVERAGE"
)

display(
    validation_detection_summary
)


print(
    "\n[5] POOLED DESCRIPTIVE CORRELATIONS"
)

display(
    pooled_core50_biology
)


print(
    "\n[6] PRIMARY WITHIN-PATIENT DIRECTION CONSISTENCY"
)

display(
    biology_direction_summary
)


print(
    "\n[7] PATIENT-LEVEL CORRELATIONS"
)

display(
    core50_biology_by_patient
)


print(
    "\n[8] CORE50-TECHNICAL COMPLEXITY AUDIT"
)

print(
    "Technical covariates:",
    TECHNICAL_COVARIATES
)

display(
    core50_technical_audit
)


print(
    "\n[9] DETECTABLE-CORE50 SENSITIVITY"
)

display(
    core50_biology_detectable
)


print(
    "\n[10] TECHNICAL-COMPLEXITY-CONTROLLED SENSITIVITY"
)

display(
    partial_direction_summary
)


print(
    "\n[11] FIREWALL / CLAIM BOUNDARY"
)

print(
"""
All phenotype-validation programs are orthogonal to the frozen Core50 at
the Ensembl-gene level.

Any Core50-overlapping genes were removed from validation programs before
scoring.

No validation genes were selected according to their association with
Core50 in GSE326201.

Pooled cell-level correlations are descriptive only.

The primary evidence is within-patient direction consistency.

Because Core50 expression is sparse, technical-complexity associations
and complexity-controlled partial-Spearman analyses are explicitly
reported.

No Core50-high cutoff and no new MS3-like cluster has been defined.

This step does not yet declare external biological validation successful;
the result will be judged after review of cross-patient direction,
effect magnitude, sparsity, and technical-complexity sensitivity.
"""
)


print(
    "\n[12] FILE"
)

print(
    E2A_PATH
)

print(
    "Exists:",
    E2A_PATH.exists()
)


print(
    "\nSTEP31E2A: COMPLETE"
)

print(
    "NEXT: review evidence before any MS3 external-validation claim."
)

print("=" * 120)

## Step31E2B — Raw-count patient-pseudobulk robustness validation

### Rationale

Single-cell Core50 expression is sparse in GSE326201.

To reduce dropout and avoid cell-level pseudoreplication, the frozen
malignant-cell identities are projected back to the frozen raw-count
matrix and aggregated by patient.

### Principles

- The malignant universe remains the frozen 4,688-cell Step31D2B universe.
- Raw counts are aggregated without gene reselection.
- Core50 remains exactly the frozen 50 genes.
- Orthogonal phenotype programs remain exactly those defined in Step31E2A.
- No patient is weighted by its number of cells in correlation analyses.
- Pseudobulk scores are deterministic and unweighted.
- Results are descriptive because only 10 independent patients are available.

In [ ]:
# =============================================================================
# Step31E2B Cell 189
# Recover frozen malignant cells from raw-count object
# =============================================================================

import scanpy as sc
import pandas as pd
import numpy as np
import scipy.sparse as sp


RAW_FLAGGED_PATH = (
    DATA_DIR /
    "GSE326201_STEP31C1_QC_SCRUBLET_FLAGGED_RAW.h5ad"
)


assert RAW_FLAGGED_PATH.exists()


print(
    "Opening raw-count object in backed mode..."
)


adata_raw_backed = sc.read_h5ad(
    RAW_FLAGGED_PATH,
    backed="r"
)


print(
    "Raw object:",
    adata_raw_backed.shape
)


raw_obs_index = pd.Index(
    adata_raw_backed.obs_names
    .astype(str)
)


malignant_cell_ids = (
    adata_malignant.obs_names
    .astype(str)
    .tolist()
)


cell_pos = raw_obs_index.get_indexer(
    malignant_cell_ids
)


assert np.all(
    cell_pos >= 0
), (
    "Some frozen malignant cells are absent "
    "from the raw-count object."
)


print(
    "Matched malignant cells:",
    len(
        cell_pos
    )
)


# Load only the 4,688 malignant cells into memory
adata_raw_malignant = (
    adata_raw_backed[
        cell_pos,
        :
    ]
    .to_memory()
)


# Close backed file
try:
    adata_raw_backed.file.close()
except Exception:
    pass


# Transfer frozen malignant metadata
for col in [
    "patient_id",
    "sample_id",
    "tissue",
    "leiden_harmony_r08"
]:

    adata_raw_malignant.obs[
        col
    ] = (
        adata_malignant.obs.loc[
            adata_raw_malignant.obs_names,
            col
        ]
        .values
    )


assert (
    adata_raw_malignant.n_obs
    ==
    4688
)


assert (
    adata_raw_malignant.obs[
        "patient_id"
    ].nunique()
    ==
    10
)


print(
    "\nRaw malignant object:",
    adata_raw_malignant
)

In [ ]:
# =============================================================================
# Step31E2B Cell 190
# Raw-count integrity audit
# =============================================================================

X_raw = (
    adata_raw_malignant.X
)


if sp.issparse(
    X_raw
):

    raw_values = (
        X_raw.data
    )

else:

    raw_values = (
        np.asarray(
            X_raw
        ).ravel()
    )


print(
    "Minimum raw value:",
    float(
        raw_values.min()
    )
)


print(
    "Maximum raw value:",
    float(
        raw_values.max()
    )
)


integer_like = bool(
    np.allclose(
        raw_values,
        np.round(
            raw_values
        )
    )
)


nonnegative = bool(
    np.all(
        raw_values
        >=
        0
    )
)


print(
    "Integer-like:",
    integer_like
)

print(
    "Non-negative:",
    nonnegative
)


assert integer_like
assert nonnegative


print(
    "\nRAW COUNT INTEGRITY: PASS"
)

In [ ]:
# =============================================================================
# Step31E2B Cell 191
# Verify frozen pseudobulk gene universe
# =============================================================================

assert "CORE50_ENSEMBL_IDS" in globals()
assert "VALIDATION_PROGRAM_IDS" in globals()


all_validation_ids = list(
    dict.fromkeys(
        CORE50_ENSEMBL_IDS
        +
        [
            gene_id
            for ids in
            VALIDATION_PROGRAM_IDS.values()
            for gene_id in ids
        ]
    )
)


raw_gene_universe = set(
    adata_raw_malignant.var_names
    .astype(str)
)


missing_pb_genes = [
    x
    for x in all_validation_ids
    if x not in raw_gene_universe
]


print(
    "Frozen genes required:",
    len(
        all_validation_ids
    )
)

print(
    "Missing from raw object:",
    len(
        missing_pb_genes
    )
)


if missing_pb_genes:

    print(
        missing_pb_genes
    )


assert len(
    missing_pb_genes
) == 0


print(
    "\nPSEUDOBULK GENE COVERAGE: PASS"
)

In [ ]:
# =============================================================================
# Step31E2B Cell 192
# Aggregate raw counts by patient
# =============================================================================

patient_order = sorted(
    adata_raw_malignant.obs[
        "patient_id"
    ]
    .astype(str)
    .unique()
)


pseudobulk_counts = []

pseudobulk_meta_rows = []


for patient in patient_order:

    mask = (
        adata_raw_malignant.obs[
            "patient_id"
        ].astype(str)
        ==
        patient
    ).to_numpy()


    X_patient = (
        adata_raw_malignant.X[
            mask,
            :
        ]
    )


    patient_counts = np.asarray(
        X_patient.sum(
            axis=0
        )
    ).ravel()


    pseudobulk_counts.append(
        patient_counts
    )


    pseudobulk_meta_rows.append({

        "patient_id":
            patient,

        "sample_id":
            str(
                adata_raw_malignant.obs.loc[
                    mask,
                    "sample_id"
                ].iloc[0]
            ),

        "n_malignant_cells":
            int(
                mask.sum()
            ),

        "library_size":
            float(
                patient_counts.sum()
            )
    })


pseudobulk_counts = np.vstack(
    pseudobulk_counts
)


pseudobulk_meta = pd.DataFrame(
    pseudobulk_meta_rows
)


print(
    "Pseudobulk count matrix:",
    pseudobulk_counts.shape
)


display(
    pseudobulk_meta
)


assert pseudobulk_counts.shape == (
    10,
    32732
)

In [ ]:
# =============================================================================
# Step31E2B Cell 193
# Library-size normalization
# =============================================================================

library_sizes = (
    pseudobulk_counts.sum(
        axis=1,
        keepdims=True
    )
)


assert np.all(
    library_sizes > 0
)


pseudobulk_cpm = (
    pseudobulk_counts
    /
    library_sizes
    *
    1e6
)


pseudobulk_logcpm = np.log1p(
    pseudobulk_cpm
)


print(
    "log1p CPM matrix:",
    pseudobulk_logcpm.shape
)


print(
    "Finite:",
    bool(
        np.isfinite(
            pseudobulk_logcpm
        ).all()
    )
)

In [ ]:
# =============================================================================
# Step31E2B Cell 194
# Deterministic patient-pseudobulk program scoring
# =============================================================================

pb_var_index = pd.Index(
    adata_raw_malignant.var_names
    .astype(str)
)


def pseudobulk_zmean(
    gene_ids
):

    idx = pb_var_index.get_indexer(
        gene_ids
    )


    assert np.all(
        idx >= 0
    )


    X = pseudobulk_logcpm[
        :,
        idx
    ]


    mu = X.mean(
        axis=0,
        keepdims=True
    )


    sd = X.std(
        axis=0,
        ddof=0,
        keepdims=True
    )


    zero_sd = (
        sd == 0
    )


    safe_sd = (
        sd.copy()
    )

    safe_sd[
        zero_sd
    ] = 1.0


    Z = (
        X - mu
    ) / safe_sd


    Z[
        :,
        zero_sd.ravel()
    ] = 0


    return Z.mean(
        axis=1
    )


pseudobulk_meta[
    "pb_core50"
] = pseudobulk_zmean(
    CORE50_ENSEMBL_IDS
)


for program, ids in (
    VALIDATION_PROGRAM_IDS.items()
):

    pseudobulk_meta[
        f"pb_{program}"
    ] = pseudobulk_zmean(
        ids
    )


display(
    pseudobulk_meta
)

In [ ]:
# =============================================================================
# Step31E2B Cell 195
# Core50 patient-pseudobulk detection audit
# =============================================================================

core50_idx = (
    pb_var_index.get_indexer(
        CORE50_ENSEMBL_IDS
    )
)


core50_pb_counts = (
    pseudobulk_counts[
        :,
        core50_idx
    ]
)


core50_pb_detection = (
    core50_pb_counts
    >
    0
)


pseudobulk_meta[
    "core50_genes_detected"
] = (
    core50_pb_detection.sum(
        axis=1
    )
)


print(
    "Core50 genes detected per patient pseudobulk:"
)


display(
    pseudobulk_meta[
        [
            "patient_id",
            "n_malignant_cells",
            "library_size",
            "core50_genes_detected"
        ]
    ]
)


core50_pb_gene_detection = pd.DataFrame({

    "frozen_symbol":
        CORE50_ORIGINAL_SYMBOLS,

    "validation_ensembl_id":
        CORE50_ENSEMBL_IDS,

    "n_patients_detected":
        core50_pb_detection.sum(
            axis=0
        )
})


print(
    "\nCore50 genes detected in all 10 patient pseudobulks:",
    int(
        (
            core50_pb_gene_detection[
                "n_patients_detected"
            ]
            ==
            10
        ).sum()
    ),
    "/50"
)


display(
    core50_pb_gene_detection.sort_values(
        "n_patients_detected"
    )
)

In [ ]:
# =============================================================================
# Step31E2B Cell 196
# Patient-level pseudobulk correlations
# =============================================================================

from scipy.stats import spearmanr


pb_correlation_rows = []


for program in (
    VALIDATION_PROGRAM_IDS.keys()
):

    rho, p = spearmanr(
        pseudobulk_meta[
            "pb_core50"
        ],
        pseudobulk_meta[
            f"pb_{program}"
        ]
    )


    expected = (
        EXPECTED_DIRECTION[
            program
        ]
    )


    supported = (
        rho > 0
        if expected == "positive"
        else rho < 0
    )


    pb_correlation_rows.append({

        "program":
            program,

        "expected_direction":
            expected,

        "spearman_rho":
            rho,

        "p_value":
            p,

        "direction_supported":
            supported
    })


pseudobulk_correlations = pd.DataFrame(
    pb_correlation_rows
)


display(
    pseudobulk_correlations
)

In [ ]:
# =============================================================================
# Step31E2B Cell 197
# Leave-one-patient-out pseudobulk sensitivity
# =============================================================================

lopo_rows = []


for program in (
    VALIDATION_PROGRAM_IDS.keys()
):

    expected = (
        EXPECTED_DIRECTION[
            program
        ]
    )


    for left_out in patient_order:

        keep = (
            pseudobulk_meta[
                "patient_id"
            ].astype(str)
            !=
            left_out
        )


        rho, p = spearmanr(
            pseudobulk_meta.loc[
                keep,
                "pb_core50"
            ],
            pseudobulk_meta.loc[
                keep,
                f"pb_{program}"
            ]
        )


        supported = (
            rho > 0
            if expected == "positive"
            else rho < 0
        )


        lopo_rows.append({

            "program":
                program,

            "expected_direction":
                expected,

            "left_out_patient":
                left_out,

            "n_patients":
                int(
                    keep.sum()
                ),

            "spearman_rho":
                rho,

            "direction_supported":
                supported
        })


pseudobulk_lopo = pd.DataFrame(
    lopo_rows
)


pseudobulk_lopo_summary = (
    pseudobulk_lopo
    .groupby(
        [
            "program",
            "expected_direction"
        ],
        observed=True
    )
    .agg(

        n_lopo=(
            "left_out_patient",
            "size"
        ),

        n_direction_supported=(
            "direction_supported",
            "sum"
        ),

        median_rho=(
            "spearman_rho",
            "median"
        ),

        min_rho=(
            "spearman_rho",
            "min"
        ),

        max_rho=(
            "spearman_rho",
            "max"
        )
    )
    .reset_index()
)


pseudobulk_lopo_summary[
    "direction_fraction"
] = (
    pseudobulk_lopo_summary[
        "n_direction_supported"
    ]
    /
    pseudobulk_lopo_summary[
        "n_lopo"
    ]
)


display(
    pseudobulk_lopo_summary
)

In [ ]:
# =============================================================================
# Step31E2B Cell 198
# Small-cell-count patient sensitivity
# =============================================================================

MIN_PSEUDOBULK_CELLS = 20


adequate_pb_mask = (
    pseudobulk_meta[
        "n_malignant_cells"
    ]
    >=
    MIN_PSEUDOBULK_CELLS
)


print(
    "Patients retained:",
    int(
        adequate_pb_mask.sum()
    ),
    "/10"
)


print(
    "Excluded:"
)


display(
    pseudobulk_meta.loc[
        ~adequate_pb_mask,
        [
            "patient_id",
            "n_malignant_cells"
        ]
    ]
)


pb_adequate_rows = []


for program in (
    VALIDATION_PROGRAM_IDS.keys()
):

    rho, p = spearmanr(
        pseudobulk_meta.loc[
            adequate_pb_mask,
            "pb_core50"
        ],
        pseudobulk_meta.loc[
            adequate_pb_mask,
            f"pb_{program}"
        ]
    )


    expected = (
        EXPECTED_DIRECTION[
            program
        ]
    )


    pb_adequate_rows.append({

        "program":
            program,

        "expected_direction":
            expected,

        "n_patients":
            int(
                adequate_pb_mask.sum()
            ),

        "spearman_rho":
            rho,

        "p_value":
            p,

        "direction_supported":
            (
                rho > 0
                if expected == "positive"
                else rho < 0
            )
    })


pseudobulk_adequate_sensitivity = (
    pd.DataFrame(
        pb_adequate_rows
    )
)


display(
    pseudobulk_adequate_sensitivity
)

In [ ]:
# =============================================================================
# Step31E2B Cell 199
# Patient-pseudobulk technical audit
# =============================================================================

tech_patient = (
    adata_malignant.obs
    .groupby(
        "patient_id",
        observed=True
    )
    .agg(

        median_n_genes=(
            "n_genes_by_counts",
            "median"
        ),

        median_total_counts=(
            "total_counts",
            "median"
        ),

        median_pct_mt=(
            "pct_counts_mt",
            "median"
        )
    )
    .reset_index()
)


tech_patient[
    "patient_id"
] = (
    tech_patient[
        "patient_id"
    ].astype(str)
)


pb_technical = (
    pseudobulk_meta
    .merge(
        tech_patient,
        on="patient_id",
        how="left"
    )
)


pb_technical_rows = []


for covariate in [
    "n_malignant_cells",
    "library_size",
    "median_n_genes",
    "median_total_counts",
    "median_pct_mt"
]:

    rho, p = spearmanr(
        pb_technical[
            "pb_core50"
        ],
        pb_technical[
            covariate
        ]
    )


    pb_technical_rows.append({

        "covariate":
            covariate,

        "spearman_rho":
            rho,

        "p_value":
            p
    })


pseudobulk_technical_audit = (
    pd.DataFrame(
        pb_technical_rows
    )
)


display(
    pseudobulk_technical_audit
)

In [ ]:
# =============================================================================
# Step31E2B Cell 200
# Save patient-pseudobulk robustness audit
# =============================================================================

PB_COUNTS_PATH = (
    DATA_DIR /
    "GSE326201_STEP31E2B_MalignantPatientPseudobulkCounts.npz"
)


sp.save_npz(
    PB_COUNTS_PATH,
    sp.csr_matrix(
        pseudobulk_counts
    )
)


pseudobulk_meta.to_csv(
    AUDIT_DIR /
    "Step31E2B_PatientPseudobulkScores.csv",
    index=False
)


core50_pb_gene_detection.to_csv(
    AUDIT_DIR /
    "Step31E2B_Core50PseudobulkDetection.csv",
    index=False
)


pseudobulk_correlations.to_csv(
    AUDIT_DIR /
    "Step31E2B_PseudobulkCorrelations.csv",
    index=False
)


pseudobulk_lopo.to_csv(
    AUDIT_DIR /
    "Step31E2B_PseudobulkLOPO.csv",
    index=False
)


pseudobulk_lopo_summary.to_csv(
    AUDIT_DIR /
    "Step31E2B_PseudobulkLOPOSummary.csv",
    index=False
)


pseudobulk_adequate_sensitivity.to_csv(
    AUDIT_DIR /
    "Step31E2B_PseudobulkAdequateCellSensitivity.csv",
    index=False
)


pseudobulk_technical_audit.to_csv(
    AUDIT_DIR /
    "Step31E2B_PseudobulkTechnicalAudit.csv",
    index=False
)


print(
    "Step31E2B outputs saved."
)

In [ ]:
# =============================================================================
# Step31E2B Cell 201
# REPORT — raw-count patient-pseudobulk robustness
# =============================================================================

print("=" * 120)

print(
    "STEP31E2B — RAW-COUNT PATIENT-PSEUDOBULK ROBUSTNESS"
)

print("=" * 120)


print(
    "\n[1] PSEUDOBULK UNIVERSE"
)

print(
    "Malignant cells:",
    adata_raw_malignant.n_obs
)

print(
    "Patients:",
    len(
        patient_order
    )
)

print(
    "Genes:",
    adata_raw_malignant.n_vars
)


display(
    pseudobulk_meta[
        [
            "patient_id",
            "sample_id",
            "n_malignant_cells",
            "library_size",
            "core50_genes_detected"
        ]
    ]
)


print(
    "\n[2] CORE50 PSEUDOBULK PORTABILITY"
)

print(
    "Core50 genes detected in all 10 patients:",
    int(
        (
            core50_pb_gene_detection[
                "n_patients_detected"
            ]
            ==
            10
        ).sum()
    ),
    "/50"
)


display(
    core50_pb_gene_detection.sort_values(
        "n_patients_detected"
    )
)


print(
    "\n[3] PRIMARY PATIENT-LEVEL PSEUDOBULK CORRELATIONS"
)

display(
    pseudobulk_correlations
)


print(
    "\n[4] LEAVE-ONE-PATIENT-OUT ROBUSTNESS"
)

display(
    pseudobulk_lopo_summary
)


print(
    "\n[5] >=20-MALIGNANT-CELL PATIENT SENSITIVITY"
)

display(
    pseudobulk_adequate_sensitivity
)


print(
    "\n[6] PATIENT-LEVEL TECHNICAL AUDIT"
)

display(
    pseudobulk_technical_audit
)


print(
    "\n[7] CLAIM BOUNDARY"
)

print(
"""
Raw counts from the independently frozen malignant-cell universe were
aggregated to the patient level before normalization.

Each patient contributes one independent pseudobulk observation regardless
of the number of malignant cells.

The frozen Core50 and orthogonal phenotype programs were not reselected,
reweighted, or optimized.

Pseudobulk analysis is intended to reduce single-cell dropout and
pseudoreplication, but the cohort contains only 10 independent patients.

Leave-one-patient-out results are therefore emphasized for robustness.

No Core50-high cutoff or new MS3-like state is defined in this analysis.
"""
)


print(
    "\nSTEP31E2B: COMPLETE"
)

print(
    "NEXT: integrate E2A + E2B and freeze the external Core50 validation claim."
)

print("=" * 120)

## Step31E Final Lock — External validation of the frozen MS3 Core50

### Primary external-validation claim

The frozen Step25 MS3 Conserved Core50 independently recapitulates a
proliferative / G2M / mitotic-entry / spindle–chromosome-segregation
phenotype in GSE326201 malignant HCC cells.

This conclusion is supported by:

1. complete 50/50 biological identifier portability;
2. an independently defined malignant-cell universe;
3. orthogonal validation programs with Core50-overlapping genes removed;
4. consistent positive within-patient associations;
5. technically adjusted single-cell sensitivity;
6. patient-level raw-count pseudobulk validation;
7. leave-one-patient-out directional robustness.

### Secondary biological claim

The expected reciprocal association with hepatic differentiation and
bile-acid metabolic programs is supported at the patient-pseudobulk
level but is weaker and less consistent at single-cell resolution.

### Boundaries

- No Core50 genes were reselected.
- No Core50-high cutoff was optimized.
- No new MS3 cluster was discovered.
- No malignant cells were selected using Core50.
- Expression association does not establish causal state transition.
- Technical complexity remains a potential contributor and is explicitly
  retained as a limitation.

In [ ]:
# =============================================================================
# Step31E Final Lock Cell 203
# Integrate E1 + E2A + E2B evidence
# =============================================================================

import pandas as pd
import numpy as np


# -----------------------------------------------------------------------------
# Primary pseudobulk evidence
# -----------------------------------------------------------------------------

pb_final = (
    pseudobulk_correlations[
        [
            "program",
            "expected_direction",
            "spearman_rho",
            "p_value",
            "direction_supported"
        ]
    ]
    .copy()
)


pb_final = (
    pb_final
    .merge(
        pseudobulk_lopo_summary[
            [
                "program",
                "n_direction_supported",
                "n_lopo",
                "median_rho",
                "min_rho",
                "max_rho",
                "direction_fraction"
            ]
        ],
        on="program",
        how="left"
    )
)


pb_final = (
    pb_final
    .merge(
        pseudobulk_adequate_sensitivity[
            [
                "program",
                "n_patients",
                "spearman_rho",
                "p_value",
                "direction_supported"
            ]
        ].rename(
            columns={
                "n_patients":
                    "adequate_n_patients",

                "spearman_rho":
                    "adequate_spearman_rho",

                "p_value":
                    "adequate_p_value",

                "direction_supported":
                    "adequate_direction_supported"
            }
        ),
        on="program",
        how="left"
    )
)


# -----------------------------------------------------------------------------
# Single-cell evidence
# -----------------------------------------------------------------------------

singlecell_final = (
    biology_direction_summary[
        [
            "program",
            "n_patients",
            "n_direction_supported",
            "median_rho",
            "direction_fraction"
        ]
    ]
    .rename(
        columns={
            "n_patients":
                "singlecell_n_patients",

            "n_direction_supported":
                "singlecell_n_direction_supported",

            "median_rho":
                "singlecell_median_rho",

            "direction_fraction":
                "singlecell_direction_fraction"
        }
    )
)


partial_final = (
    partial_direction_summary[
        [
            "program",
            "n_patients",
            "n_direction_supported",
            "median_partial_rho",
            "direction_fraction"
        ]
    ]
    .rename(
        columns={
            "n_patients":
                "partial_n_patients",

            "n_direction_supported":
                "partial_n_direction_supported",

            "direction_fraction":
                "partial_direction_fraction"
        }
    )
)


final_validation_audit = (
    pb_final
    .merge(
        singlecell_final,
        on="program",
        how="left"
    )
    .merge(
        partial_final,
        on="program",
        how="left"
    )
)


display(
    final_validation_audit
)

In [ ]:
# =============================================================================
# Step31E Final Lock Cell 204
# Freeze interpretation level
# =============================================================================

PRIMARY_MS3_PROGRAMS = [
    "G2M_Independent",
    "Mitotic_Entry_Independent",
    "Spindle_Chromosome_Independent"
]


SECONDARY_RECIPROCAL_PROGRAMS = [
    "Hepatic_Differentiation",
    "Bile_Acid_Metabolism"
]


final_validation_audit[
    "claim_role"
] = np.where(
    final_validation_audit[
        "program"
    ].isin(
        PRIMARY_MS3_PROGRAMS
    ),
    "Primary_MS3_biology",
    "Secondary_reciprocal_biology"
)


print(
    "PRIMARY MS3 BIOLOGY:"
)

display(
    final_validation_audit.loc[
        final_validation_audit[
            "claim_role"
        ]
        ==
        "Primary_MS3_biology"
    ]
)


print(
    "\nSECONDARY RECIPROCAL BIOLOGY:"
)

display(
    final_validation_audit.loc[
        final_validation_audit[
            "claim_role"
        ]
        ==
        "Secondary_reciprocal_biology"
    ]
)

In [ ]:
# =============================================================================
# Step31E Final Lock Cell 205
# Validation-lock checks
# =============================================================================

primary_audit = (
    final_validation_audit.loc[
        final_validation_audit[
            "program"
        ].isin(
            PRIMARY_MS3_PROGRAMS
        )
    ]
)


secondary_audit = (
    final_validation_audit.loc[
        final_validation_audit[
            "program"
        ].isin(
            SECONDARY_RECIPROCAL_PROGRAMS
        )
    ]
)


# All primary programs must have correct pseudobulk direction
assert (
    primary_audit[
        "direction_supported"
    ].all()
)


# All primary programs must maintain direction in all LOPO analyses
assert (
    primary_audit[
        "direction_fraction"
    ]
    .eq(
        1.0
    )
    .all()
)


# All secondary programs also maintain LOPO expected direction
assert (
    secondary_audit[
        "direction_fraction"
    ]
    .eq(
        1.0
    )
    .all()
)


print(
    "Primary MS3 pseudobulk direction: PASS"
)

print(
    "Primary MS3 LOPO robustness: PASS"
)

print(
    "Secondary reciprocal LOPO direction: PASS"
)


print(
    "\nFINAL INTERPRETATION:"
)

print(
    """
Primary:
Frozen MS3 Core50 externally validates proliferative / G2M /
mitotic-entry / spindle-chromosome-segregation biology.

Secondary:
Reciprocal hepatic-differentiation / bile-acid-metabolic behavior is
supported at patient-pseudobulk resolution but remains weaker at
single-cell resolution.
"""
)

In [ ]:
# =============================================================================
# Step31E Final Lock Cell 206
# Save frozen external-validation claim
# =============================================================================

import json


STEP31E_FINAL_AUDIT_PATH = (
    AUDIT_DIR /
    "Step31E_FINAL_EXTERNAL_VALIDATION_AUDIT.csv"
)


STEP31E_FINAL_LOCK_PATH = (
    AUDIT_DIR /
    "Step31E_FINAL_CLAIM_LOCK.json"
)


final_validation_audit.to_csv(
    STEP31E_FINAL_AUDIT_PATH,
    index=False
)


step31e_final_lock = {

    "step":
        "Step31E",

    "status":
        "FROZEN",

    "dataset":
        "GSE326201",

    "malignant_cells":
        4688,

    "patients":
        10,

    "core50_membership":
        50,

    "core50_reselected":
        False,

    "malignancy_defined_using_core50":
        False,

    "primary_claim":
        (
            "The frozen MS3 Core50 independently recapitulates "
            "proliferative/G2M, mitotic-entry, and "
            "spindle-chromosome-segregation biology in the "
            "GSE326201 malignant HCC compartment."
        ),

    "secondary_claim":
        (
            "Reciprocal hepatic-differentiation and bile-acid-metabolic "
            "associations are supported at the patient-pseudobulk level "
            "but are weaker and less consistent at single-cell resolution."
        ),

    "primary_programs":
        PRIMARY_MS3_PROGRAMS,

    "secondary_programs":
        SECONDARY_RECIPROCAL_PROGRAMS,

    "pseudobulk_patients":
        10,

    "lopo_direction_consistency":
        "10/10 leave-one-patient-out analyses retained the prespecified direction for all five programs.",

    "limitations": [

        "Core50 expression is sparse at single-cell resolution.",

        "Only 10 independent patients are available.",

        "Patient-level Core50 score is associated with some per-cell technical-complexity metrics.",

        "Expression-derived associations do not establish causal state transition.",

        "The reciprocal differentiated/metabolic arm is weaker than the proliferative/mitotic arm."
    ]
}


with open(
    STEP31E_FINAL_LOCK_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        step31e_final_lock,
        f,
        ensure_ascii=False,
        indent=2
    )


print(
    "Final audit:"
)

print(
    STEP31E_FINAL_AUDIT_PATH
)


print(
    "\nFinal claim lock:"
)

print(
    STEP31E_FINAL_LOCK_PATH
)

In [ ]:
# =============================================================================
# Step31E Final Lock Cell 207
# FINAL REPORT
# =============================================================================

print("=" * 120)

print(
    "STEP31E — FROZEN MS3 CORE50 EXTERNAL VALIDATION FINAL LOCK"
)

print("=" * 120)


print(
    "\n[1] VALIDATION UNIVERSE"
)

print(
    "Independent malignant cells:",
    4688
)

print(
    "Independent patients:",
    10
)

print(
    "Frozen Core50 genes:",
    50
)


print(
    "\n[2] PRIMARY MS3 BIOLOGY"
)

display(
    final_validation_audit.loc[
        final_validation_audit[
            "claim_role"
        ]
        ==
        "Primary_MS3_biology"
    ][
        [
            "program",
            "spearman_rho",
            "p_value",
            "direction_fraction",
            "singlecell_direction_fraction",
            "partial_direction_fraction"
        ]
    ]
)


print(
    "\n[3] SECONDARY RECIPROCAL BIOLOGY"
)

display(
    final_validation_audit.loc[
        final_validation_audit[
            "claim_role"
        ]
        ==
        "Secondary_reciprocal_biology"
    ][
        [
            "program",
            "spearman_rho",
            "p_value",
            "direction_fraction",
            "singlecell_direction_fraction",
            "partial_direction_fraction"
        ]
    ]
)


print(
    "\n[4] PSEUDOBULK LOPO ROBUSTNESS"
)

display(
    pseudobulk_lopo_summary
)


print(
    "\n[5] TECHNICAL LIMITATION"
)

display(
    pseudobulk_technical_audit
)


print(
    "\n[6] FINAL FROZEN CLAIM"
)

print(
"""
PRIMARY CLAIM

The frozen Step25 MS3 Conserved Core50 independently recapitulates a
proliferative / G2M / mitotic-entry / spindle-chromosome-segregation
phenotype in the GSE326201 malignant HCC compartment.

SECONDARY CLAIM

The reciprocal hepatic-differentiation / bile-acid-metabolic relationship
is directionally supported at patient-pseudobulk resolution, but is weaker
and less consistently reproduced at single-cell resolution.

BOUNDARY

The result supports independent biological reproducibility of the frozen
MS3 program. It does not establish causal state transition, and the
reciprocal MS3-MS6 axis should not be described as uniformly reproduced
at single-cell resolution.
"""
)


print(
    "\n[7] FIREWALL"
)

print(
    "Core50 reselected:",
    False
)

print(
    "Core50 cutoff optimized:",
    False
)

print(
    "Malignancy defined using Core50:",
    False
)

print(
    "New MS3 cluster discovered:",
    False
)


print(
    "\n[8] FILES"
)

print(
    STEP31E_FINAL_AUDIT_PATH
)

print(
    STEP31E_FINAL_LOCK_PATH
)


print(
    "\nSTEP31E: FROZEN"
)

print(
    "NEXT: Step31F — independent validation of the reciprocal MS3-like / MS6-like biological axis."
)

print("=" * 120)

## Step31F — Independent validation of the reciprocal MS3-like / MS6-like biological axis

### Objective

Test whether the independently defined MS3-like proliferative/mitotic pole
and MS6-like hepatic-differentiated/metabolic pole form a reproducible
reciprocal biological continuum in GSE326201 malignant HCC cells.

### Frozen design

MS3-like pole:
- G2M_Independent
- Spindle_Chromosome_Independent
- Mitotic_Entry_Independent

MS6-like pole:
- Hepatic_Differentiation
- Bile_Acid_Metabolism

Core50 is NOT used to construct either pole.

No genes, clusters, cutoffs, or states are reselected.

In [ ]:
# =============================================================================
# Step31F Cell 209
# Construct independent MS3-like and MS6-like biological poles
# =============================================================================

import numpy as np
import pandas as pd


MS3_POLE_PROGRAMS = [
    "G2M_Independent",
    "Spindle_Chromosome_Independent",
    "Mitotic_Entry_Independent"
]

MS6_POLE_PROGRAMS = [
    "Hepatic_Differentiation",
    "Bile_Acid_Metabolism"
]


required_cols = [
    f"program_{x}"
    for x in (
        MS3_POLE_PROGRAMS
        +
        MS6_POLE_PROGRAMS
    )
]


assert all(
    x in adata_malignant.obs.columns
    for x in required_cols
)


def standardize_vector(x):

    x = np.asarray(
        x,
        dtype=float
    )

    sd = x.std(
        ddof=0
    )

    if sd == 0:
        return np.zeros_like(x)

    return (
        x - x.mean()
    ) / sd


# Equal-weight standardized program scores
ms3_components = np.column_stack([
    standardize_vector(
        adata_malignant.obs[
            f"program_{x}"
        ]
    )
    for x in MS3_POLE_PROGRAMS
])


ms6_components = np.column_stack([
    standardize_vector(
        adata_malignant.obs[
            f"program_{x}"
        ]
    )
    for x in MS6_POLE_PROGRAMS
])


adata_malignant.obs[
    "ms3_like_pole"
] = (
    ms3_components.mean(
        axis=1
    )
)


adata_malignant.obs[
    "ms6_like_pole"
] = (
    ms6_components.mean(
        axis=1
    )
)


adata_malignant.obs[
    "ms3_ms6_reciprocal_axis"
] = (
    standardize_vector(
        adata_malignant.obs[
            "ms3_like_pole"
        ]
    )
    -
    standardize_vector(
        adata_malignant.obs[
            "ms6_like_pole"
        ]
    )
)


print(
    "MS3-like pole:",
    MS3_POLE_PROGRAMS
)

print(
    "MS6-like pole:",
    MS6_POLE_PROGRAMS
)

print(
    "Core50 used to construct poles:",
    False
)


print(
    "\nRECIPROCAL POLES: READY"
)

In [ ]:
# =============================================================================
# Step31F Cell 210
# Within-patient reciprocal-axis validation
# =============================================================================

from scipy.stats import spearmanr


reciprocal_patient_rows = []


patient_ids = (
    adata_malignant.obs[
        "patient_id"
    ]
    .astype(str)
    .unique()
    .tolist()
)


for patient in patient_ids:

    sub = (
        adata_malignant.obs.loc[
            adata_malignant.obs[
                "patient_id"
            ].astype(str)
            ==
            patient
        ]
    )


    rho, p = spearmanr(
        sub[
            "ms3_like_pole"
        ],
        sub[
            "ms6_like_pole"
        ]
    )


    reciprocal_patient_rows.append({

        "patient_id":
            patient,

        "n_cells":
            len(sub),

        "spearman_rho":
            rho,

        "p_value":
            p,

        "expected_negative":
            bool(
                rho < 0
            )
    })


reciprocal_by_patient = pd.DataFrame(
    reciprocal_patient_rows
)


display(
    reciprocal_by_patient
)


print(
    "\nPatients with MS3 pole inversely related to MS6 pole:",
    int(
        reciprocal_by_patient[
            "expected_negative"
        ].sum()
    ),
    "/",
    len(
        reciprocal_by_patient
    )
)


print(
    "Median patient rho:",
    float(
        reciprocal_by_patient[
            "spearman_rho"
        ].median()
    )
)

In [ ]:
# =============================================================================
# Step31F Cell 211
# Technical-complexity-controlled reciprocal relationship
# =============================================================================

from scipy.stats import rankdata


TECHNICAL_COVARIATES_F = [
    x
    for x in [
        "n_genes_by_counts",
        "total_counts",
        "pct_counts_mt"
    ]
    if x in adata_malignant.obs.columns
]


def rank_residualize_f(
    y,
    C
):

    y = np.asarray(
        y,
        dtype=float
    )

    y_rank = rankdata(
        y,
        method="average"
    )


    X = [
        np.ones(
            len(y)
        )
    ]


    for j in range(
        C.shape[1]
    ):

        X.append(
            rankdata(
                C[:, j],
                method="average"
            )
        )


    X = np.column_stack(
        X
    )


    beta = np.linalg.lstsq(
        X,
        y_rank,
        rcond=None
    )[0]


    return (
        y_rank
        -
        X @ beta
    )


partial_reciprocal_rows = []


for patient in patient_ids:

    sub = (
        adata_malignant.obs.loc[
            adata_malignant.obs[
                "patient_id"
            ].astype(str)
            ==
            patient
        ]
        .copy()
    )


    needed = [
        "ms3_like_pole",
        "ms6_like_pole"
    ] + TECHNICAL_COVARIATES_F


    sub = (
        sub[
            needed
        ]
        .dropna()
    )


    # Skip very small patient
    if len(sub) < 20:
        continue


    C = (
        sub[
            TECHNICAL_COVARIATES_F
        ]
        .astype(float)
        .to_numpy()
    )


    ms3_resid = (
        rank_residualize_f(
            sub[
                "ms3_like_pole"
            ],
            C
        )
    )


    ms6_resid = (
        rank_residualize_f(
            sub[
                "ms6_like_pole"
            ],
            C
        )
    )


    rho, p = spearmanr(
        ms3_resid,
        ms6_resid
    )


    partial_reciprocal_rows.append({

        "patient_id":
            patient,

        "n_cells":
            len(sub),

        "partial_spearman_rho":
            rho,

        "p_value":
            p,

        "expected_negative":
            bool(
                rho < 0
            )
    })


partial_reciprocal = pd.DataFrame(
    partial_reciprocal_rows
)


display(
    partial_reciprocal
)


print(
    "\nAdjusted negative direction:",
    int(
        partial_reciprocal[
            "expected_negative"
        ].sum()
    ),
    "/",
    len(
        partial_reciprocal
    )
)


print(
    "Median adjusted rho:",
    float(
        partial_reciprocal[
            "partial_spearman_rho"
        ].median()
    )
)

In [ ]:
# =============================================================================
# Step31F Cell 212
# Patient-pseudobulk reciprocal-axis validation
# =============================================================================

assert "pseudobulk_meta" in globals()


for program in (
    MS3_POLE_PROGRAMS
    +
    MS6_POLE_PROGRAMS
):

    assert (
        f"pb_{program}"
        in
        pseudobulk_meta.columns
    )


pb_ms3_components = np.column_stack([
    standardize_vector(
        pseudobulk_meta[
            f"pb_{x}"
        ]
    )
    for x in MS3_POLE_PROGRAMS
])


pb_ms6_components = np.column_stack([
    standardize_vector(
        pseudobulk_meta[
            f"pb_{x}"
        ]
    )
    for x in MS6_POLE_PROGRAMS
])


pseudobulk_meta[
    "pb_ms3_like_pole"
] = (
    pb_ms3_components.mean(
        axis=1
    )
)


pseudobulk_meta[
    "pb_ms6_like_pole"
] = (
    pb_ms6_components.mean(
        axis=1
    )
)


pseudobulk_meta[
    "pb_reciprocal_axis"
] = (
    standardize_vector(
        pseudobulk_meta[
            "pb_ms3_like_pole"
        ]
    )
    -
    standardize_vector(
        pseudobulk_meta[
            "pb_ms6_like_pole"
        ]
    )
)


pb_axis_rho, pb_axis_p = spearmanr(
    pseudobulk_meta[
        "pb_ms3_like_pole"
    ],
    pseudobulk_meta[
        "pb_ms6_like_pole"
    ]
)


display(
    pseudobulk_meta[
        [
            "patient_id",
            "n_malignant_cells",
            "pb_ms3_like_pole",
            "pb_ms6_like_pole",
            "pb_reciprocal_axis"
        ]
    ]
)


print(
    "\nPseudobulk MS3-vs-MS6 Spearman rho:",
    pb_axis_rho
)

print(
    "P value:",
    pb_axis_p
)

print(
    "Expected negative direction:",
    bool(
        pb_axis_rho < 0
    )
)

In [ ]:
# =============================================================================
# Step31F Cell 213
# Leave-one-patient-out reciprocal-axis robustness
# =============================================================================

reciprocal_lopo_rows = []


for patient in (
    pseudobulk_meta[
        "patient_id"
    ]
    .astype(str)
):

    keep = (
        pseudobulk_meta[
            "patient_id"
        ].astype(str)
        !=
        patient
    )


    rho, p = spearmanr(
        pseudobulk_meta.loc[
            keep,
            "pb_ms3_like_pole"
        ],
        pseudobulk_meta.loc[
            keep,
            "pb_ms6_like_pole"
        ]
    )


    reciprocal_lopo_rows.append({

        "left_out_patient":
            patient,

        "n_patients":
            int(
                keep.sum()
            ),

        "spearman_rho":
            rho,

        "p_value":
            p,

        "expected_negative":
            bool(
                rho < 0
            )
    })


reciprocal_lopo = pd.DataFrame(
    reciprocal_lopo_rows
)


display(
    reciprocal_lopo
)


print(
    "\nLOPO negative direction:",
    int(
        reciprocal_lopo[
            "expected_negative"
        ].sum()
    ),
    "/",
    len(
        reciprocal_lopo
    )
)


print(
    "Median LOPO rho:",
    float(
        reciprocal_lopo[
            "spearman_rho"
        ].median()
    )
)

print(
    "Range:",
    float(
        reciprocal_lopo[
            "spearman_rho"
        ].min()
    ),
    "to",
    float(
        reciprocal_lopo[
            "spearman_rho"
        ].max()
    )
)

In [ ]:
# =============================================================================
# Step31F Cell 214
# Save reciprocal-axis validation
# =============================================================================

STEP31F_PATH = (
    DATA_DIR /
    "GSE326201_STEP31F_RECIPROCAL_MS3_MS6_AXIS.h5ad"
)


adata_malignant.write_h5ad(
    STEP31F_PATH,
    compression="gzip"
)


reciprocal_by_patient.to_csv(
    AUDIT_DIR /
    "Step31F_WithinPatientReciprocalAxis.csv",
    index=False
)


partial_reciprocal.to_csv(
    AUDIT_DIR /
    "Step31F_AdjustedReciprocalAxis.csv",
    index=False
)


pseudobulk_meta[
    [
        "patient_id",
        "n_malignant_cells",
        "pb_ms3_like_pole",
        "pb_ms6_like_pole",
        "pb_reciprocal_axis"
    ]
].to_csv(
    AUDIT_DIR /
    "Step31F_PseudobulkReciprocalAxis.csv",
    index=False
)


reciprocal_lopo.to_csv(
    AUDIT_DIR /
    "Step31F_PseudobulkReciprocalAxis_LOPO.csv",
    index=False
)


print(
    "Saved:"
)

print(
    STEP31F_PATH
)

print(
    "Exists:",
    STEP31F_PATH.exists()
)

In [ ]:
# =============================================================================
# Step31F Cell 215
# REPORT — reciprocal MS3-like / MS6-like axis
# =============================================================================

print("=" * 120)

print(
    "STEP31F — INDEPENDENT VALIDATION OF THE RECIPROCAL MS3-LIKE / MS6-LIKE AXIS"
)

print("=" * 120)


print(
    "\n[1] ANALYSIS UNIVERSE"
)

print(
    "Malignant cells:",
    adata_malignant.n_obs
)

print(
    "Patients:",
    adata_malignant.obs[
        "patient_id"
    ].nunique()
)


print(
    "\n[2] AXIS DEFINITION"
)

print(
    "MS3-like pole:",
    MS3_POLE_PROGRAMS
)

print(
    "MS6-like pole:",
    MS6_POLE_PROGRAMS
)

print(
    "Core50 used to construct axis:",
    False
)


print(
    "\n[3] WITHIN-PATIENT RECIPROCAL RELATIONSHIP"
)

display(
    reciprocal_by_patient
)

print(
    "Negative direction:",
    int(
        reciprocal_by_patient[
            "expected_negative"
        ].sum()
    ),
    "/",
    len(
        reciprocal_by_patient
    )
)

print(
    "Median rho:",
    float(
        reciprocal_by_patient[
            "spearman_rho"
        ].median()
    )
)


print(
    "\n[4] TECHNICAL-COMPLEXITY-CONTROLLED"
)

display(
    partial_reciprocal
)

print(
    "Adjusted negative direction:",
    int(
        partial_reciprocal[
            "expected_negative"
        ].sum()
    ),
    "/",
    len(
        partial_reciprocal
    )
)

print(
    "Median adjusted rho:",
    float(
        partial_reciprocal[
            "partial_spearman_rho"
        ].median()
    )
)


print(
    "\n[5] PATIENT-PSEUDOBULK"
)

print(
    "Spearman rho:",
    pb_axis_rho
)

print(
    "P value:",
    pb_axis_p
)

print(
    "Expected negative direction:",
    bool(
        pb_axis_rho < 0
    )
)


print(
    "\n[6] PSEUDOBULK LOPO"
)

display(
    reciprocal_lopo
)

print(
    "Negative direction:",
    int(
        reciprocal_lopo[
            "expected_negative"
        ].sum()
    ),
    "/",
    len(
        reciprocal_lopo
    )
)

print(
    "Median LOPO rho:",
    float(
        reciprocal_lopo[
            "spearman_rho"
        ].median()
    )
)


print(
    "\n[7] CLAIM BOUNDARY"
)

print(
"""
The MS3-like and MS6-like poles were constructed exclusively from
prespecified orthogonal biological programs.

Core50 was not used to define either pole.

No clustering, gene selection, state discovery, or cutoff optimization
was performed.

The analysis tests reciprocal biological organization only.

A negative MS3-like versus MS6-like association supports a reciprocal
continuum compatible with the frozen Step24-27 biological axis, but does
not establish lineage conversion or causal state transition.
"""
)


print(
    "\n[8] FILE"
)

print(
    STEP31F_PATH
)

print(
    "Exists:",
    STEP31F_PATH.exists()
)


print(
    "\nSTEP31F: COMPLETE"
)

print(
    "NEXT: review evidence before freezing the reciprocal-axis claim."
)

print("=" * 120)

In [ ]:
# =============================================================================
# Step31F Cell 216
# Freeze reciprocal-axis interpretation
# =============================================================================

import json


STEP31F_FINAL_LOCK_PATH = (
    AUDIT_DIR /
    "Step31F_FINAL_RECIPROCAL_AXIS_LOCK.json"
)


step31f_final_lock = {

    "step":
        "Step31F",

    "status":
        "FROZEN",

    "malignant_cells":
        4688,

    "patients":
        10,

    "axis_definition": {

        "MS3_like":
            [
                "G2M_Independent",
                "Spindle_Chromosome_Independent",
                "Mitotic_Entry_Independent"
            ],

        "MS6_like":
            [
                "Hepatic_Differentiation",
                "Bile_Acid_Metabolism"
            ],

        "core50_used":
            False
    },

    "single_cell_unadjusted": {

        "negative_patients":
            "2/10",

        "median_rho":
            float(
                reciprocal_by_patient[
                    "spearman_rho"
                ].median()
            )
    },

    "single_cell_adjusted": {

        "negative_patients":
            (
                f"{int(partial_reciprocal['expected_negative'].sum())}"
                f"/{len(partial_reciprocal)}"
            ),

        "median_partial_rho":
            float(
                partial_reciprocal[
                    "partial_spearman_rho"
                ].median()
            )
    },

    "patient_pseudobulk": {

        "spearman_rho":
            float(
                pb_axis_rho
            ),

        "p_value":
            float(
                pb_axis_p
            )
    },

    "lopo": {

        "negative_direction":
            (
                f"{int(reciprocal_lopo['expected_negative'].sum())}"
                f"/{len(reciprocal_lopo)}"
            ),

        "median_rho":
            float(
                reciprocal_lopo[
                    "spearman_rho"
                ].median()
            )
    },

    "frozen_claim":
        (
            "The reciprocal MS3-like proliferative/mitotic versus "
            "MS6-like hepatic/metabolic organization is supported "
            "at the patient-pseudobulk level and after adjustment "
            "for technical complexity, but is not uniformly reproduced "
            "at the unadjusted single-cell level."
        ),

    "claim_boundary": [

        "No new malignant state was discovered.",

        "No clustering or gene reselection was performed.",

        "Core50 was not used to construct the reciprocal axis.",

        "The result supports reciprocal biological organization, not lineage conversion.",

        "The axis should not be described as uniformly reproduced at raw single-cell resolution."
    ]
}


with open(
    STEP31F_FINAL_LOCK_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        step31f_final_lock,
        f,
        ensure_ascii=False,
        indent=2
    )


print(
    "Saved:"
)

print(
    STEP31F_FINAL_LOCK_PATH
)

print(
    "Exists:",
    STEP31F_FINAL_LOCK_PATH.exists()
)

In [ ]:
# =============================================================================
# Step31F Cell 217
# Frozen reciprocal-axis evidence summary
# =============================================================================

step31f_summary = pd.DataFrame([

    {
        "analysis":
            "Unadjusted within-patient",

        "result":
            (
                f"{int(reciprocal_by_patient['expected_negative'].sum())}"
                f"/{len(reciprocal_by_patient)} negative"
            ),

        "effect":
            float(
                reciprocal_by_patient[
                    "spearman_rho"
                ].median()
            )
    },

    {
        "analysis":
            "Adjusted within-patient",

        "result":
            (
                f"{int(partial_reciprocal['expected_negative'].sum())}"
                f"/{len(partial_reciprocal)} negative"
            ),

        "effect":
            float(
                partial_reciprocal[
                    "partial_spearman_rho"
                ].median()
            )
    },

    {
        "analysis":
            "Patient pseudobulk",

        "result":
            "Negative",

        "effect":
            float(
                pb_axis_rho
            )
    },

    {
        "analysis":
            "Pseudobulk LOPO",

        "result":
            (
                f"{int(reciprocal_lopo['expected_negative'].sum())}"
                f"/{len(reciprocal_lopo)} negative"
            ),

        "effect":
            float(
                reciprocal_lopo[
                    "spearman_rho"
                ].median()
            )
    }
])


display(
    step31f_summary
)


step31f_summary.to_csv(
    AUDIT_DIR /
    "Step31F_FINAL_ReciprocalAxisSummary.csv",
    index=False
)

In [ ]:
# =============================================================================
# Step31F Cell 218
# FINAL REPORT
# =============================================================================

print("=" * 120)

print(
    "STEP31F — RECIPROCAL MS3-LIKE / MS6-LIKE AXIS FINAL LOCK"
)

print("=" * 120)


print(
    "\n[1] UNADJUSTED SINGLE-CELL"
)

print(
    "Negative patients:",
    int(
        reciprocal_by_patient[
            "expected_negative"
        ].sum()
    ),
    "/",
    len(
        reciprocal_by_patient
    )
)

print(
    "Median rho:",
    float(
        reciprocal_by_patient[
            "spearman_rho"
        ].median()
    )
)


print(
    "\n[2] TECHNICAL-ADJUSTED SINGLE-CELL"
)

print(
    "Negative patients:",
    int(
        partial_reciprocal[
            "expected_negative"
        ].sum()
    ),
    "/",
    len(
        partial_reciprocal
    )
)

print(
    "Median partial rho:",
    float(
        partial_reciprocal[
            "partial_spearman_rho"
        ].median()
    )
)


print(
    "\n[3] PATIENT PSEUDOBULK"
)

print(
    "Spearman rho:",
    pb_axis_rho
)

print(
    "P value:",
    pb_axis_p
)


print(
    "\n[4] LOPO ROBUSTNESS"
)

print(
    "Negative direction:",
    int(
        reciprocal_lopo[
            "expected_negative"
        ].sum()
    ),
    "/",
    len(
        reciprocal_lopo
    )
)

print(
    "Median rho:",
    float(
        reciprocal_lopo[
            "spearman_rho"
        ].median()
    )
)


print(
    "\n[5] FINAL FROZEN CLAIM"
)

print(
"""
The reciprocal MS3-like proliferative/mitotic versus MS6-like
hepatic/metabolic organization is supported at the patient-pseudobulk
level and after adjustment for technical complexity, but is not uniformly
reproduced at the unadjusted single-cell level.

This finding supports reciprocal biological organization compatible with
the frozen Step24-27 axis, but does not demonstrate lineage conversion or
causal state transition.
"""
)


print(
    "\nSTEP31F: FROZEN"
)

print(
    "NEXT: Step31G — validation of the frozen Step29 regulator architecture."
)

print("=" * 120)

## Step31G — Independent validation of the frozen Step29 regulator architecture

### Objective

Validate whether the frozen Step29 regulator architecture is associated
with the corresponding independently defined biological poles in GSE326201.

### Frozen regulators

MS3 side:
- FOXM1
- MYBL2
- E2F8
- TFDP1
- E2F1
- BRCA1

MS6 side:
- HNF4A
- NR1H4
- NR1I3
- CEBPA
- FOXA3

### Boundaries

- No new transcription factors are selected.
- No GRN or regulon discovery is rerun.
- No TF is weighted according to GSE326201 results.
- Regulator expression is not equivalent to regulon activity.
- Primary inference emphasizes patient-level pseudobulk and LOPO robustness.

In [ ]:
# =============================================================================
# Step31G Cell 220
# Map frozen Step29 regulators to the independent dataset
# =============================================================================

import pandas as pd
import numpy as np


MS3_FROZEN_REGULATORS = [
    "FOXM1",
    "MYBL2",
    "E2F8",
    "TFDP1",
    "E2F1",
    "BRCA1"
]


MS6_FROZEN_REGULATORS = [
    "HNF4A",
    "NR1H4",
    "NR1I3",
    "CEBPA",
    "FOXA3"
]


ALL_FROZEN_REGULATORS = (
    MS3_FROZEN_REGULATORS
    +
    MS6_FROZEN_REGULATORS
)


# Rebuild symbol -> Ensembl mapping from the frozen resource
gene_coordinates_g = pd.read_csv(
    GENE_COORD_PATH,
    dtype={
        "gene_id": str,
        "symbol": str
    }
)


malignant_gene_universe = set(
    adata_malignant.var_names
    .astype(str)
)


mapping_source = (
    gene_coordinates_g.loc[
        gene_coordinates_g[
            "gene_id"
        ].astype(str).isin(
            malignant_gene_universe
        )
        &
        gene_coordinates_g[
            "symbol"
        ].notna(),
        [
            "gene_id",
            "symbol"
        ]
    ]
    .copy()
)


mapping_source[
    "symbol"
] = (
    mapping_source[
        "symbol"
    ]
    .astype(str)
    .str.strip()
)


mapping_source[
    "gene_id"
] = (
    mapping_source[
        "gene_id"
    ].astype(str)
)


symbol_to_ids_g = (
    mapping_source
    .groupby(
        "symbol"
    )[
        "gene_id"
    ]
    .apply(
        lambda x:
            sorted(
                set(x)
            )
    )
    .to_dict()
)


regulator_mapping_rows = []


for symbol in ALL_FROZEN_REGULATORS:

    hits = (
        symbol_to_ids_g.get(
            symbol,
            []
        )
    )


    regulator_mapping_rows.append({

        "regulator":
            symbol,

        "side":
            (
                "MS3"
                if symbol
                in MS3_FROZEN_REGULATORS
                else "MS6"
            ),

        "n_matches":
            len(hits),

        "ensembl_id":
            (
                hits[0]
                if len(hits) == 1
                else None
            ),

        "mapping_status":
            (
                "unique"
                if len(hits) == 1
                else
                "missing"
                if len(hits) == 0
                else
                "ambiguous"
            )
    })


regulator_mapping = pd.DataFrame(
    regulator_mapping_rows
)


display(
    regulator_mapping
)


assert (
    regulator_mapping[
        "mapping_status"
    ]
    .eq(
        "unique"
    )
    .all()
), (
    "One or more frozen Step29 regulators are not uniquely mapped."
)


MS3_REGULATOR_IDS = (
    regulator_mapping.loc[
        regulator_mapping[
            "side"
        ]
        ==
        "MS3",
        "ensembl_id"
    ]
    .tolist()
)


MS6_REGULATOR_IDS = (
    regulator_mapping.loc[
        regulator_mapping[
            "side"
        ]
        ==
        "MS6",
        "ensembl_id"
    ]
    .tolist()
)


print(
    "\nMS3 regulators mapped:",
    len(
        MS3_REGULATOR_IDS
    ),
    "/6"
)

print(
    "MS6 regulators mapped:",
    len(
        MS6_REGULATOR_IDS
    ),
    "/5"
)


print(
    "\nFROZEN REGULATOR MAPPING: PASS"
)

In [ ]:
# =============================================================================
# Step31G Cell 221
# Frozen regulator expression coverage
# =============================================================================

import scipy.sparse as sp


regulator_expression_rows = []


for _, row in regulator_mapping.iterrows():

    gene_id = (
        row[
            "ensembl_id"
        ]
    )


    X = (
        adata_malignant[
            :,
            [gene_id]
        ].X
    )


    if sp.issparse(
        X
    ):

        values = (
            X.toarray()
            .ravel()
        )

    else:

        values = (
            np.asarray(
                X
            )
            .ravel()
        )


    regulator_expression_rows.append({

        "regulator":
            row[
                "regulator"
            ],

        "side":
            row[
                "side"
            ],

        "ensembl_id":
            gene_id,

        "detection_fraction":
            float(
                (
                    values > 0
                ).mean()
            ),

        "mean_logexpr":
            float(
                values.mean()
            ),

        "sd_logexpr":
            float(
                values.std(
                    ddof=0
                )
            )
    })


regulator_expression_audit = (
    pd.DataFrame(
        regulator_expression_rows
    )
)


display(
    regulator_expression_audit
)


print(
    "\nRegulators detected at least once:",
    int(
        (
            regulator_expression_audit[
                "detection_fraction"
            ]
            >
            0
        ).sum()
    ),
    "/",
    len(
        regulator_expression_audit
    )
)

In [ ]:
# =============================================================================
# Step31G Cell 222
# Deterministic frozen regulator-module scoring
# =============================================================================

def score_fixed_regulator_module(
    adata,
    gene_ids
):

    X = (
        adata[
            :,
            gene_ids
        ].X
    )


    if sp.issparse(
        X
    ):

        X = X.toarray()


    X = np.asarray(
        X,
        dtype=np.float32
    )


    mu = (
        X.mean(
            axis=0,
            keepdims=True
        )
    )


    sd = (
        X.std(
            axis=0,
            ddof=0,
            keepdims=True
        )
    )


    zero_sd = (
        sd == 0
    )


    safe_sd = (
        sd.copy()
    )

    safe_sd[
        zero_sd
    ] = 1.0


    Z = (
        X - mu
    ) / safe_sd


    Z[
        :,
        zero_sd.ravel()
    ] = 0


    return (
        Z.mean(
            axis=1
        )
    )


adata_malignant.obs[
    "step29_ms3_regulator_module"
] = (
    score_fixed_regulator_module(
        adata_malignant,
        MS3_REGULATOR_IDS
    )
)


adata_malignant.obs[
    "step29_ms6_regulator_module"
] = (
    score_fixed_regulator_module(
        adata_malignant,
        MS6_REGULATOR_IDS
    )
)


print(
    "MS3 regulator module: 6 frozen regulators"
)

print(
    "MS6 regulator module: 5 frozen regulators"
)

print(
    "Data-driven weighting:",
    False
)

In [ ]:
# =============================================================================
# Step31G Cell 223
# Within-patient regulator-to-pole validation
# =============================================================================

from scipy.stats import spearmanr


regulator_pole_patient_rows = []


patient_ids = (
    adata_malignant.obs[
        "patient_id"
    ]
    .astype(str)
    .unique()
    .tolist()
)


for patient in patient_ids:

    sub = (
        adata_malignant.obs.loc[
            adata_malignant.obs[
                "patient_id"
            ].astype(str)
            ==
            patient
        ]
    )


    # MS3 regulator module -> MS3 biological pole
    rho, p = spearmanr(
        sub[
            "step29_ms3_regulator_module"
        ],
        sub[
            "ms3_like_pole"
        ]
    )


    regulator_pole_patient_rows.append({

        "patient_id":
            patient,

        "n_cells":
            len(sub),

        "architecture":
            "MS3_regulators_to_MS3_pole",

        "spearman_rho":
            rho,

        "p_value":
            p,

        "expected_positive":
            bool(
                rho > 0
            )
    })


    # MS6 regulator module -> MS6 biological pole
    rho, p = spearmanr(
        sub[
            "step29_ms6_regulator_module"
        ],
        sub[
            "ms6_like_pole"
        ]
    )


    regulator_pole_patient_rows.append({

        "patient_id":
            patient,

        "n_cells":
            len(sub),

        "architecture":
            "MS6_regulators_to_MS6_pole",

        "spearman_rho":
            rho,

        "p_value":
            p,

        "expected_positive":
            bool(
                rho > 0
            )
    })


regulator_pole_by_patient = (
    pd.DataFrame(
        regulator_pole_patient_rows
    )
)


display(
    regulator_pole_by_patient
)


regulator_pole_patient_summary = (
    regulator_pole_by_patient
    .groupby(
        "architecture",
        observed=True
    )
    .agg(

        n_patients=(
            "patient_id",
            "nunique"
        ),

        n_positive=(
            "expected_positive",
            "sum"
        ),

        median_rho=(
            "spearman_rho",
            "median"
        ),

        min_rho=(
            "spearman_rho",
            "min"
        ),

        max_rho=(
            "spearman_rho",
            "max"
        )
    )
    .reset_index()
)


display(
    regulator_pole_patient_summary
)

In [ ]:
# =============================================================================
# Step31G Cell 224
# Technical-complexity-adjusted regulator architecture
# =============================================================================

adjusted_regulator_rows = []


for patient in patient_ids:

    sub = (
        adata_malignant.obs.loc[
            adata_malignant.obs[
                "patient_id"
            ].astype(str)
            ==
            patient
        ]
        .copy()
    )


    if len(
        sub
    ) < 20:

        continue


    needed = [
        "step29_ms3_regulator_module",
        "step29_ms6_regulator_module",
        "ms3_like_pole",
        "ms6_like_pole"
    ] + TECHNICAL_COVARIATES_F


    sub = (
        sub[
            needed
        ]
        .dropna()
    )


    C = (
        sub[
            TECHNICAL_COVARIATES_F
        ]
        .astype(float)
        .to_numpy()
    )


    # MS3
    regulator_resid = (
        rank_residualize_f(
            sub[
                "step29_ms3_regulator_module"
            ],
            C
        )
    )


    pole_resid = (
        rank_residualize_f(
            sub[
                "ms3_like_pole"
            ],
            C
        )
    )


    rho, p = spearmanr(
        regulator_resid,
        pole_resid
    )


    adjusted_regulator_rows.append({

        "patient_id":
            patient,

        "n_cells":
            len(sub),

        "architecture":
            "MS3_regulators_to_MS3_pole",

        "partial_spearman_rho":
            rho,

        "p_value":
            p,

        "expected_positive":
            bool(
                rho > 0
            )
    })


    # MS6
    regulator_resid = (
        rank_residualize_f(
            sub[
                "step29_ms6_regulator_module"
            ],
            C
        )
    )


    pole_resid = (
        rank_residualize_f(
            sub[
                "ms6_like_pole"
            ],
            C
        )
    )


    rho, p = spearmanr(
        regulator_resid,
        pole_resid
    )


    adjusted_regulator_rows.append({

        "patient_id":
            patient,

        "n_cells":
            len(sub),

        "architecture":
            "MS6_regulators_to_MS6_pole",

        "partial_spearman_rho":
            rho,

        "p_value":
            p,

        "expected_positive":
            bool(
                rho > 0
            )
    })


adjusted_regulator_architecture = (
    pd.DataFrame(
        adjusted_regulator_rows
    )
)


adjusted_regulator_summary = (
    adjusted_regulator_architecture
    .groupby(
        "architecture",
        observed=True
    )
    .agg(

        n_patients=(
            "patient_id",
            "nunique"
        ),

        n_positive=(
            "expected_positive",
            "sum"
        ),

        median_partial_rho=(
            "partial_spearman_rho",
            "median"
        ),

        min_partial_rho=(
            "partial_spearman_rho",
            "min"
        ),

        max_partial_rho=(
            "partial_spearman_rho",
            "max"
        )
    )
    .reset_index()
)


display(
    adjusted_regulator_summary
)

In [ ]:
# =============================================================================
# Step31G Cell 225
# Patient-pseudobulk validation of frozen regulator architecture
# =============================================================================

assert "pseudobulk_logcpm" in globals()
assert "pb_var_index" in globals()
assert "pseudobulk_meta" in globals()


def pseudobulk_regulator_zmean(
    gene_ids
):

    idx = (
        pb_var_index.get_indexer(
            gene_ids
        )
    )


    assert np.all(
        idx >= 0
    )


    X = (
        pseudobulk_logcpm[
            :,
            idx
        ]
    )


    mu = (
        X.mean(
            axis=0,
            keepdims=True
        )
    )


    sd = (
        X.std(
            axis=0,
            ddof=0,
            keepdims=True
        )
    )


    zero_sd = (
        sd == 0
    )


    safe_sd = (
        sd.copy()
    )

    safe_sd[
        zero_sd
    ] = 1.0


    Z = (
        X - mu
    ) / safe_sd


    Z[
        :,
        zero_sd.ravel()
    ] = 0


    return (
        Z.mean(
            axis=1
        )
    )


pseudobulk_meta[
    "pb_step29_ms3_regulators"
] = (
    pseudobulk_regulator_zmean(
        MS3_REGULATOR_IDS
    )
)


pseudobulk_meta[
    "pb_step29_ms6_regulators"
] = (
    pseudobulk_regulator_zmean(
        MS6_REGULATOR_IDS
    )
)


pb_ms3_reg_rho, pb_ms3_reg_p = (
    spearmanr(
        pseudobulk_meta[
            "pb_step29_ms3_regulators"
        ],
        pseudobulk_meta[
            "pb_ms3_like_pole"
        ]
    )
)


pb_ms6_reg_rho, pb_ms6_reg_p = (
    spearmanr(
        pseudobulk_meta[
            "pb_step29_ms6_regulators"
        ],
        pseudobulk_meta[
            "pb_ms6_like_pole"
        ]
    )
)


print(
    "MS3 regulator module -> MS3 pole:"
)

print(
    "rho =",
    pb_ms3_reg_rho,
    "p =",
    pb_ms3_reg_p
)


print(
    "\nMS6 regulator module -> MS6 pole:"
)

print(
    "rho =",
    pb_ms6_reg_rho,
    "p =",
    pb_ms6_reg_p
)

In [ ]:
# =============================================================================
# Step31G Cell 226
# LOPO robustness of frozen regulator architecture
# =============================================================================

regulator_lopo_rows = []


for left_out in (
    pseudobulk_meta[
        "patient_id"
    ].astype(str)
):

    keep = (
        pseudobulk_meta[
            "patient_id"
        ].astype(str)
        !=
        left_out
    )


    # MS3
    rho, p = spearmanr(
        pseudobulk_meta.loc[
            keep,
            "pb_step29_ms3_regulators"
        ],
        pseudobulk_meta.loc[
            keep,
            "pb_ms3_like_pole"
        ]
    )


    regulator_lopo_rows.append({

        "architecture":
            "MS3_regulators_to_MS3_pole",

        "left_out_patient":
            left_out,

        "spearman_rho":
            rho,

        "p_value":
            p,

        "expected_positive":
            bool(
                rho > 0
            )
    })


    # MS6
    rho, p = spearmanr(
        pseudobulk_meta.loc[
            keep,
            "pb_step29_ms6_regulators"
        ],
        pseudobulk_meta.loc[
            keep,
            "pb_ms6_like_pole"
        ]
    )


    regulator_lopo_rows.append({

        "architecture":
            "MS6_regulators_to_MS6_pole",

        "left_out_patient":
            left_out,

        "spearman_rho":
            rho,

        "p_value":
            p,

        "expected_positive":
            bool(
                rho > 0
            )
    })


regulator_lopo = (
    pd.DataFrame(
        regulator_lopo_rows
    )
)


regulator_lopo_summary = (
    regulator_lopo
    .groupby(
        "architecture",
        observed=True
    )
    .agg(

        n_lopo=(
            "left_out_patient",
            "size"
        ),

        n_positive=(
            "expected_positive",
            "sum"
        ),

        median_rho=(
            "spearman_rho",
            "median"
        ),

        min_rho=(
            "spearman_rho",
            "min"
        ),

        max_rho=(
            "spearman_rho",
            "max"
        )
    )
    .reset_index()
)


display(
    regulator_lopo_summary
)

In [ ]:
# =============================================================================
# Step31G Cell 227
# Save frozen-regulator validation
# =============================================================================

STEP31G_PATH = (
    DATA_DIR /
    "GSE326201_STEP31G_STEP29_REGULATOR_VALIDATION.h5ad"
)


adata_malignant.write_h5ad(
    STEP31G_PATH,
    compression="gzip"
)


regulator_mapping.to_csv(
    AUDIT_DIR /
    "Step31G_FrozenRegulatorMapping.csv",
    index=False
)


regulator_expression_audit.to_csv(
    AUDIT_DIR /
    "Step31G_FrozenRegulatorExpressionAudit.csv",
    index=False
)


regulator_pole_by_patient.to_csv(
    AUDIT_DIR /
    "Step31G_RegulatorPoleByPatient.csv",
    index=False
)


regulator_pole_patient_summary.to_csv(
    AUDIT_DIR /
    "Step31G_RegulatorPolePatientSummary.csv",
    index=False
)


adjusted_regulator_summary.to_csv(
    AUDIT_DIR /
    "Step31G_AdjustedRegulatorArchitectureSummary.csv",
    index=False
)


regulator_lopo.to_csv(
    AUDIT_DIR /
    "Step31G_RegulatorArchitecture_LOPO.csv",
    index=False
)


regulator_lopo_summary.to_csv(
    AUDIT_DIR /
    "Step31G_RegulatorArchitecture_LOPOSummary.csv",
    index=False
)


print(
    "Saved:",
    STEP31G_PATH
)

print(
    "Exists:",
    STEP31G_PATH.exists()
)

In [ ]:
# =============================================================================
# Step31G Cell 228
# REPORT — frozen Step29 regulator architecture validation
# =============================================================================

print("=" * 120)

print(
    "STEP31G — INDEPENDENT VALIDATION OF THE FROZEN STEP29 REGULATOR ARCHITECTURE"
)

print("=" * 120)


print(
    "\n[1] FROZEN REGULATORS"
)

print(
    "MS3 side:",
    MS3_FROZEN_REGULATORS
)

print(
    "MS6 side:",
    MS6_FROZEN_REGULATORS
)

print(
    "Mapped:",
    len(
        regulator_mapping
    ),
    "/11"
)


print(
    "\n[2] EXPRESSION COVERAGE"
)

display(
    regulator_expression_audit
)


print(
    "\n[3] WITHIN-PATIENT ARCHITECTURE"
)

display(
    regulator_pole_patient_summary
)


print(
    "\n[4] TECHNICAL-ADJUSTED ARCHITECTURE"
)

display(
    adjusted_regulator_summary
)


print(
    "\n[5] PATIENT-PSEUDOBULK"
)

print(
    "MS3 regulators -> MS3 pole:"
)

print(
    "rho:",
    pb_ms3_reg_rho,
    "p:",
    pb_ms3_reg_p
)


print(
    "\nMS6 regulators -> MS6 pole:"
)

print(
    "rho:",
    pb_ms6_reg_rho,
    "p:",
    pb_ms6_reg_p
)


print(
    "\n[6] PSEUDOBULK LOPO"
)

display(
    regulator_lopo_summary
)


print(
    "\n[7] CLAIM BOUNDARY"
)

print(
"""
Only the frozen Step29 regulator sets were tested.

No new regulators were selected and no GRN/regulon discovery was rerun.

The analysis evaluates whether frozen regulator expression architecture is
aligned with the corresponding independently defined biological pole.

TF expression is not equivalent to regulon activity or causal regulatory
activity.

Patient-level pseudobulk and LOPO robustness are emphasized because
single-cell TF expression may be sparse.
"""
)


print(
    "\n[8] FILE"
)

print(
    STEP31G_PATH
)

print(
    "Exists:",
    STEP31G_PATH.exists()
)


print(
    "\nSTEP31G: COMPLETE"
)

print(
    "NEXT: review evidence and freeze the Step29 regulator-validation claim."
)

print("=" * 120)

In [ ]:
# =============================================================================
# Step31G Cell 229 — FINAL SAFE VERSION
# =============================================================================

import json

ms3_lopo_row = regulator_lopo_summary.loc[
    regulator_lopo_summary["architecture"]
    ==
    "MS3_regulators_to_MS3_pole"
].iloc[0]

ms6_lopo_row = regulator_lopo_summary.loc[
    regulator_lopo_summary["architecture"]
    ==
    "MS6_regulators_to_MS6_pole"
].iloc[0]


STEP31G_FINAL_LOCK_PATH = (
    AUDIT_DIR /
    "Step31G_FINAL_STEP29_REGULATOR_VALIDATION_LOCK.json"
)


step31g_final_lock = {

    "step": "Step31G",
    "status": "FROZEN",

    "primary_article_claim": (
        "The frozen Step29 MS3 regulator architecture "
        "(FOXM1, MYBL2, E2F8, TFDP1, E2F1 and BRCA1) "
        "was independently associated with the MS3-like "
        "proliferative/mitotic pole at the patient-pseudobulk level, "
        "with preserved positive direction in all leave-one-patient-out analyses."
    ),

    "secondary_article_observation": (
        "The frozen MS6 regulator set "
        "(HNF4A, NR1H4, NR1I3, CEBPA and FOXA3) "
        "showed cell-level concordance with the hepatic/metabolic pole, "
        "but was not reproduced at the patient-pseudobulk level."
    ),

    "ms3_pseudobulk": {
        "rho": float(pb_ms3_reg_rho),
        "p_value": float(pb_ms3_reg_p),
        "lopo_positive": (
            f"{int(ms3_lopo_row['n_positive'])}/"
            f"{int(ms3_lopo_row['n_lopo'])}"
        ),
        "lopo_median_rho": float(
            ms3_lopo_row["median_rho"]
        )
    },

    "ms6_pseudobulk": {
        "rho": float(pb_ms6_reg_rho),
        "p_value": float(pb_ms6_reg_p),
        "lopo_positive": (
            f"{int(ms6_lopo_row['n_positive'])}/"
            f"{int(ms6_lopo_row['n_lopo'])}"
        ),
        "lopo_median_rho": float(
            ms6_lopo_row["median_rho"]
        )
    },

    "claim_boundary": [
        "No new regulators were selected.",
        "No external GRN discovery was performed.",
        "TF expression is not equivalent to regulon activity.",
        "MS3 regulatory architecture is the primary externally supported result.",
        "MS6 regulatory architecture is retained as secondary cell-level evidence only."
    ]
}


with open(
    STEP31G_FINAL_LOCK_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        step31g_final_lock,
        f,
        ensure_ascii=False,
        indent=2
    )


print("=" * 100)
print("STEP31G — FINAL LOCK")
print("=" * 100)

print(
    "\nMS3 regulator architecture:"
)

print(
    "Pseudobulk rho =",
    pb_ms3_reg_rho
)

print(
    "LOPO positive =",
    f"{int(ms3_lopo_row['n_positive'])}/"
    f"{int(ms3_lopo_row['n_lopo'])}"
)


print(
    "\nMS6 regulator architecture:"
)

print(
    "Pseudobulk rho =",
    pb_ms6_reg_rho
)

print(
    "LOPO positive =",
    f"{int(ms6_lopo_row['n_positive'])}/"
    f"{int(ms6_lopo_row['n_lopo'])}"
)


print(
    "\nPRIMARY ARTICLE INTERPRETATION:"
)

print(
    """
External validation supports the frozen MS3-associated regulator
architecture as part of the proliferative/mitotic high-risk program.

The MS6 regulator architecture is retained only as secondary
cell-level concordance because patient-level pseudobulk validation
was not reproduced.
"""
)


print(
    "\nSTEP31G: FROZEN"
)

print(
    "NEXT: Step31H — targeted validation of frozen Step30 communication routes."
)

## Step31H — Targeted validation of frozen Step30 signaling architecture

### Primary route

MS3-associated receiver-side architecture:

IFNAR1 → PIK3CA / PIK3R1 → AKT1 → BRCA1

### Secondary route

MS6-associated receiver-side architecture:

RXRA → NR1H4

### Scope

- Only frozen Step30 routes are evaluated.
- No ligand-receptor discovery is rerun.
- No new signaling components are selected.
- No MS5-like state is rediscovered in GSE326201.
- MMP9–IFNAR1 remains exploratory and is not promoted to a primary claim.
- Expression concordance does not establish biochemical activation,
  physical signaling, or causality.

In [ ]:
# =============================================================================
# Step31H Cell 231
# Map frozen Step30 signaling-route genes
# =============================================================================

import pandas as pd
import numpy as np


MS3_FROZEN_ROUTE = [
    "IFNAR1",
    "PIK3CA",
    "PIK3R1",
    "AKT1",
    "BRCA1"
]

MS6_FROZEN_ROUTE = [
    "RXRA",
    "NR1H4"
]


STEP30_ROUTE_GENES = (
    MS3_FROZEN_ROUTE
    +
    MS6_FROZEN_ROUTE
)


gene_coordinates_h = pd.read_csv(
    GENE_COORD_PATH,
    dtype={
        "gene_id": str,
        "symbol": str
    }
)


malignant_gene_universe = set(
    adata_malignant.var_names.astype(str)
)


mapping_source_h = (
    gene_coordinates_h.loc[
        gene_coordinates_h["gene_id"]
        .astype(str)
        .isin(malignant_gene_universe)
        &
        gene_coordinates_h["symbol"].notna(),
        [
            "gene_id",
            "symbol"
        ]
    ]
    .copy()
)


mapping_source_h["symbol"] = (
    mapping_source_h["symbol"]
    .astype(str)
    .str.strip()
)


mapping_source_h["gene_id"] = (
    mapping_source_h["gene_id"]
    .astype(str)
)


symbol_to_ids_h = (
    mapping_source_h
    .groupby("symbol")["gene_id"]
    .apply(
        lambda x:
            sorted(
                set(x)
            )
    )
    .to_dict()
)


route_mapping_rows = []


for symbol in STEP30_ROUTE_GENES:

    hits = symbol_to_ids_h.get(
        symbol,
        []
    )

    route_mapping_rows.append({

        "symbol":
            symbol,

        "route":
            (
                "MS3"
                if symbol in MS3_FROZEN_ROUTE
                else "MS6"
            ),

        "n_matches":
            len(hits),

        "ensembl_id":
            hits[0]
            if len(hits) == 1
            else None,

        "mapping_status":
            (
                "unique"
                if len(hits) == 1
                else
                "missing"
                if len(hits) == 0
                else
                "ambiguous"
            )
    })


route_mapping = pd.DataFrame(
    route_mapping_rows
)


display(
    route_mapping
)


assert (
    route_mapping[
        "mapping_status"
    ]
    .eq("unique")
    .all()
)


MS3_ROUTE_IDS = (
    route_mapping.loc[
        route_mapping["route"]
        ==
        "MS3",
        "ensembl_id"
    ]
    .tolist()
)


MS6_ROUTE_IDS = (
    route_mapping.loc[
        route_mapping["route"]
        ==
        "MS6",
        "ensembl_id"
    ]
    .tolist()
)


assert len(MS3_ROUTE_IDS) == 5
assert len(MS6_ROUTE_IDS) == 2


print(
    "\nFrozen Step30 route mapping: PASS"
)

In [ ]:
# =============================================================================
# Step31H Cell 232
# Route-gene expression coverage
# =============================================================================

import scipy.sparse as sp


route_expression_rows = []


for _, row in route_mapping.iterrows():

    gene_id = row["ensembl_id"]


    X = (
        adata_malignant[
            :,
            [gene_id]
        ].X
    )


    if sp.issparse(X):

        values = (
            X.toarray()
            .ravel()
        )

    else:

        values = (
            np.asarray(X)
            .ravel()
        )


    route_expression_rows.append({

        "symbol":
            row["symbol"],

        "route":
            row["route"],

        "ensembl_id":
            gene_id,

        "detection_fraction":
            float(
                (
                    values > 0
                ).mean()
            ),

        "mean_logexpr":
            float(
                values.mean()
            ),

        "sd_logexpr":
            float(
                values.std(
                    ddof=0
                )
            )
    })


route_expression_audit = (
    pd.DataFrame(
        route_expression_rows
    )
)


display(
    route_expression_audit
)

In [ ]:
# =============================================================================
# Step31H Cell 233
# Deterministic frozen signaling-route modules
# =============================================================================

def fixed_route_zmean(
    adata,
    gene_ids
):

    X = (
        adata[
            :,
            gene_ids
        ].X
    )


    if sp.issparse(X):

        X = X.toarray()


    X = np.asarray(
        X,
        dtype=np.float32
    )


    mu = (
        X.mean(
            axis=0,
            keepdims=True
        )
    )


    sd = (
        X.std(
            axis=0,
            ddof=0,
            keepdims=True
        )
    )


    zero_sd = (
        sd == 0
    )


    safe_sd = sd.copy()

    safe_sd[
        zero_sd
    ] = 1.0


    Z = (
        X - mu
    ) / safe_sd


    Z[
        :,
        zero_sd.ravel()
    ] = 0


    return (
        Z.mean(
            axis=1
        )
    )


adata_malignant.obs[
    "step30_ms3_route_module"
] = (
    fixed_route_zmean(
        adata_malignant,
        MS3_ROUTE_IDS
    )
)


adata_malignant.obs[
    "step30_ms6_route_module"
] = (
    fixed_route_zmean(
        adata_malignant,
        MS6_ROUTE_IDS
    )
)


print(
    "MS3 route:",
    MS3_FROZEN_ROUTE
)

print(
    "MS6 route:",
    MS6_FROZEN_ROUTE
)

print(
    "Data-driven weighting:",
    False
)

In [ ]:
# =============================================================================
# Step31H Cell 234
# Within-patient frozen-route validation
# =============================================================================

from scipy.stats import spearmanr


route_patient_rows = []


patient_ids = (
    adata_malignant.obs[
        "patient_id"
    ]
    .astype(str)
    .unique()
    .tolist()
)


for patient in patient_ids:

    sub = (
        adata_malignant.obs.loc[
            adata_malignant.obs[
                "patient_id"
            ].astype(str)
            ==
            patient
        ]
    )


    # MS3 route -> MS3 pole
    rho, p = spearmanr(
        sub[
            "step30_ms3_route_module"
        ],
        sub[
            "ms3_like_pole"
        ]
    )


    route_patient_rows.append({

        "patient_id":
            patient,

        "n_cells":
            len(sub),

        "route_test":
            "MS3_route_to_MS3_pole",

        "spearman_rho":
            rho,

        "p_value":
            p,

        "expected_positive":
            bool(
                rho > 0
            )
    })


    # MS6 route -> MS6 pole
    rho, p = spearmanr(
        sub[
            "step30_ms6_route_module"
        ],
        sub[
            "ms6_like_pole"
        ]
    )


    route_patient_rows.append({

        "patient_id":
            patient,

        "n_cells":
            len(sub),

        "route_test":
            "MS6_route_to_MS6_pole",

        "spearman_rho":
            rho,

        "p_value":
            p,

        "expected_positive":
            bool(
                rho > 0
            )
    })


route_by_patient = pd.DataFrame(
    route_patient_rows
)


route_patient_summary = (
    route_by_patient
    .groupby(
        "route_test",
        observed=True
    )
    .agg(

        n_patients=(
            "patient_id",
            "nunique"
        ),

        n_positive=(
            "expected_positive",
            "sum"
        ),

        median_rho=(
            "spearman_rho",
            "median"
        ),

        min_rho=(
            "spearman_rho",
            "min"
        ),

        max_rho=(
            "spearman_rho",
            "max"
        )
    )
    .reset_index()
)


display(
    route_patient_summary
)

In [ ]:
# =============================================================================
# Step31H Cell 235
# Technical-complexity-adjusted route validation
# =============================================================================

adjusted_route_rows = []


for patient in patient_ids:

    sub = (
        adata_malignant.obs.loc[
            adata_malignant.obs[
                "patient_id"
            ].astype(str)
            ==
            patient
        ]
        .copy()
    )


    if len(sub) < 20:
        continue


    needed = [
        "step30_ms3_route_module",
        "step30_ms6_route_module",
        "ms3_like_pole",
        "ms6_like_pole"
    ] + TECHNICAL_COVARIATES_F


    sub = (
        sub[
            needed
        ]
        .dropna()
    )


    C = (
        sub[
            TECHNICAL_COVARIATES_F
        ]
        .astype(float)
        .to_numpy()
    )


    # ---------------------------------------------------------
    # MS3
    # ---------------------------------------------------------

    route_resid = (
        rank_residualize_f(
            sub[
                "step30_ms3_route_module"
            ],
            C
        )
    )


    pole_resid = (
        rank_residualize_f(
            sub[
                "ms3_like_pole"
            ],
            C
        )
    )


    rho, p = spearmanr(
        route_resid,
        pole_resid
    )


    adjusted_route_rows.append({

        "patient_id":
            patient,

        "n_cells":
            len(sub),

        "route_test":
            "MS3_route_to_MS3_pole",

        "partial_spearman_rho":
            rho,

        "p_value":
            p,

        "expected_positive":
            bool(
                rho > 0
            )
    })


    # ---------------------------------------------------------
    # MS6
    # ---------------------------------------------------------

    route_resid = (
        rank_residualize_f(
            sub[
                "step30_ms6_route_module"
            ],
            C
        )
    )


    pole_resid = (
        rank_residualize_f(
            sub[
                "ms6_like_pole"
            ],
            C
        )
    )


    rho, p = spearmanr(
        route_resid,
        pole_resid
    )


    adjusted_route_rows.append({

        "patient_id":
            patient,

        "n_cells":
            len(sub),

        "route_test":
            "MS6_route_to_MS6_pole",

        "partial_spearman_rho":
            rho,

        "p_value":
            p,

        "expected_positive":
            bool(
                rho > 0
            )
    })


adjusted_route = (
    pd.DataFrame(
        adjusted_route_rows
    )
)


adjusted_route_summary = (
    adjusted_route
    .groupby(
        "route_test",
        observed=True
    )
    .agg(

        n_patients=(
            "patient_id",
            "nunique"
        ),

        n_positive=(
            "expected_positive",
            "sum"
        ),

        median_partial_rho=(
            "partial_spearman_rho",
            "median"
        ),

        min_partial_rho=(
            "partial_spearman_rho",
            "min"
        ),

        max_partial_rho=(
            "partial_spearman_rho",
            "max"
        )
    )
    .reset_index()
)


display(
    adjusted_route_summary
)

In [ ]:
# =============================================================================
# Step31H Cell 236
# Patient-pseudobulk and LOPO validation
# =============================================================================

assert "pseudobulk_logcpm" in globals()
assert "pb_var_index" in globals()
assert "pseudobulk_meta" in globals()


def pseudobulk_route_zmean(
    gene_ids
):

    idx = (
        pb_var_index.get_indexer(
            gene_ids
        )
    )


    assert np.all(
        idx >= 0
    )


    X = (
        pseudobulk_logcpm[
            :,
            idx
        ]
    )


    mu = (
        X.mean(
            axis=0,
            keepdims=True
        )
    )


    sd = (
        X.std(
            axis=0,
            ddof=0,
            keepdims=True
        )
    )


    zero_sd = (
        sd == 0
    )


    safe_sd = sd.copy()

    safe_sd[
        zero_sd
    ] = 1.0


    Z = (
        X - mu
    ) / safe_sd


    Z[
        :,
        zero_sd.ravel()
    ] = 0


    return (
        Z.mean(
            axis=1
        )
    )


pseudobulk_meta[
    "pb_step30_ms3_route"
] = (
    pseudobulk_route_zmean(
        MS3_ROUTE_IDS
    )
)


pseudobulk_meta[
    "pb_step30_ms6_route"
] = (
    pseudobulk_route_zmean(
        MS6_ROUTE_IDS
    )
)


# -------------------------------------------------------------------------
# Primary correlations
# -------------------------------------------------------------------------

pb_ms3_route_rho, pb_ms3_route_p = (
    spearmanr(
        pseudobulk_meta[
            "pb_step30_ms3_route"
        ],
        pseudobulk_meta[
            "pb_ms3_like_pole"
        ]
    )
)


pb_ms6_route_rho, pb_ms6_route_p = (
    spearmanr(
        pseudobulk_meta[
            "pb_step30_ms6_route"
        ],
        pseudobulk_meta[
            "pb_ms6_like_pole"
        ]
    )
)


print(
    "MS3 frozen route -> MS3 pole:"
)

print(
    "rho =",
    pb_ms3_route_rho,
    "p =",
    pb_ms3_route_p
)


print(
    "\nMS6 frozen route -> MS6 pole:"
)

print(
    "rho =",
    pb_ms6_route_rho,
    "p =",
    pb_ms6_route_p
)


# -------------------------------------------------------------------------
# LOPO
# -------------------------------------------------------------------------

route_lopo_rows = []


for left_out in (
    pseudobulk_meta[
        "patient_id"
    ].astype(str)
):

    keep = (
        pseudobulk_meta[
            "patient_id"
        ].astype(str)
        !=
        left_out
    )


    # MS3
    rho, p = spearmanr(
        pseudobulk_meta.loc[
            keep,
            "pb_step30_ms3_route"
        ],
        pseudobulk_meta.loc[
            keep,
            "pb_ms3_like_pole"
        ]
    )


    route_lopo_rows.append({

        "route_test":
            "MS3_route_to_MS3_pole",

        "left_out_patient":
            left_out,

        "spearman_rho":
            rho,

        "p_value":
            p,

        "expected_positive":
            bool(
                rho > 0
            )
    })


    # MS6
    rho, p = spearmanr(
        pseudobulk_meta.loc[
            keep,
            "pb_step30_ms6_route"
        ],
        pseudobulk_meta.loc[
            keep,
            "pb_ms6_like_pole"
        ]
    )


    route_lopo_rows.append({

        "route_test":
            "MS6_route_to_MS6_pole",

        "left_out_patient":
            left_out,

        "spearman_rho":
            rho,

        "p_value":
            p,

        "expected_positive":
            bool(
                rho > 0
            )
    })


route_lopo = pd.DataFrame(
    route_lopo_rows
)


route_lopo_summary = (
    route_lopo
    .groupby(
        "route_test",
        observed=True
    )
    .agg(

        n_lopo=(
            "left_out_patient",
            "size"
        ),

        n_positive=(
            "expected_positive",
            "sum"
        ),

        median_rho=(
            "spearman_rho",
            "median"
        ),

        min_rho=(
            "spearman_rho",
            "min"
        ),

        max_rho=(
            "spearman_rho",
            "max"
        )
    )
    .reset_index()
)


display(
    route_lopo_summary
)

In [ ]:
# =============================================================================
# Step31H Cell 237
# Save targeted Step30 route validation
# =============================================================================

STEP31H_PATH = (
    DATA_DIR /
    "GSE326201_STEP31H_STEP30_TARGETED_ROUTE_VALIDATION.h5ad"
)


adata_malignant.write_h5ad(
    STEP31H_PATH,
    compression="gzip"
)


route_mapping.to_csv(
    AUDIT_DIR /
    "Step31H_FrozenRouteMapping.csv",
    index=False
)


route_expression_audit.to_csv(
    AUDIT_DIR /
    "Step31H_FrozenRouteExpressionAudit.csv",
    index=False
)


route_by_patient.to_csv(
    AUDIT_DIR /
    "Step31H_RouteByPatient.csv",
    index=False
)


route_patient_summary.to_csv(
    AUDIT_DIR /
    "Step31H_RoutePatientSummary.csv",
    index=False
)


adjusted_route_summary.to_csv(
    AUDIT_DIR /
    "Step31H_AdjustedRouteSummary.csv",
    index=False
)


route_lopo.to_csv(
    AUDIT_DIR /
    "Step31H_Route_LOPO.csv",
    index=False
)


route_lopo_summary.to_csv(
    AUDIT_DIR /
    "Step31H_Route_LOPOSummary.csv",
    index=False
)


print(
    "Saved:",
    STEP31H_PATH
)

print(
    "Exists:",
    STEP31H_PATH.exists()
)

In [ ]:
# =============================================================================
# Step31H Cell 238
# REPORT — targeted validation of frozen Step30 signaling architecture
# =============================================================================

print("=" * 120)

print(
    "STEP31H — TARGETED VALIDATION OF FROZEN STEP30 SIGNALING ARCHITECTURE"
)

print("=" * 120)


print(
    "\n[1] FROZEN ROUTES"
)

print(
    "Primary MS3 route:"
)

print(
    "IFNAR1 -> PIK3CA/PIK3R1 -> AKT1 -> BRCA1"
)

print(
    "\nSecondary MS6 route:"
)

print(
    "RXRA -> NR1H4"
)


print(
    "\n[2] GENE MAPPING / EXPRESSION"
)

display(
    route_expression_audit
)


print(
    "\n[3] WITHIN-PATIENT ARCHITECTURE"
)

display(
    route_patient_summary
)


print(
    "\n[4] TECHNICAL-ADJUSTED ARCHITECTURE"
)

display(
    adjusted_route_summary
)


print(
    "\n[5] PATIENT-PSEUDOBULK"
)

print(
    "MS3 route -> MS3 pole:"
)

print(
    "rho:",
    pb_ms3_route_rho,
    "p:",
    pb_ms3_route_p
)


print(
    "\nMS6 route -> MS6 pole:"
)

print(
    "rho:",
    pb_ms6_route_rho,
    "p:",
    pb_ms6_route_p
)


print(
    "\n[6] PSEUDOBULK LOPO"
)

display(
    route_lopo_summary
)


print(
    "\n[7] ROUTES DELIBERATELY NOT RE-DISCOVERED"
)

print(
    "MS5 ITGB1 -> SRC -> FOS:",
    "Not evaluated because no independent MS5-like state was defined."
)

print(
    "MMP9–IFNAR1:",
    "Remains exploratory; not promoted to a primary external-validation claim."
)


print(
    "\n[8] CLAIM BOUNDARY"
)

print(
"""
This analysis validates only the expression architecture of frozen
Step30 receiver-side signaling routes.

No communication-network discovery was rerun.

Expression concordance does not demonstrate ligand-receptor binding,
biochemical pathway activation, physical intercellular signaling,
or causal regulation.

The MS3 IFNAR1-PI3K-AKT-BRCA1 route is the primary article-focused test.

The RXRA-NR1H4 route is secondary.

MS5 and exploratory MMP9-IFNAR1 findings are not expanded in this
independent dataset.
"""
)


print(
    "\n[9] FILE"
)

print(
    STEP31H_PATH
)

print(
    "Exists:",
    STEP31H_PATH.exists()
)


print(
    "\nSTEP31H: COMPLETE"
)

print(
    "NEXT: review and freeze Step31H, then Step31 FINAL LOCK."
)

print("=" * 120)

In [ ]:
# =============================================================================
# Step31H Cell 239
# FINAL CLAIM LOCK
# =============================================================================

import json


ms3_route_lopo = (
    route_lopo_summary.loc[
        route_lopo_summary["route_test"]
        ==
        "MS3_route_to_MS3_pole"
    ]
    .iloc[0]
)


ms6_route_lopo = (
    route_lopo_summary.loc[
        route_lopo_summary["route_test"]
        ==
        "MS6_route_to_MS6_pole"
    ]
    .iloc[0]
)


STEP31H_FINAL_LOCK_PATH = (
    AUDIT_DIR /
    "Step31H_FINAL_STEP30_ROUTE_VALIDATION_LOCK.json"
)


step31h_final_lock = {

    "step":
        "Step31H",

    "status":
        "FROZEN",

    "primary_route":
        "IFNAR1 -> PIK3CA/PIK3R1 -> AKT1 -> BRCA1",

    "secondary_route":
        "RXRA -> NR1H4",

    "primary_article_interpretation":
        (
            "The frozen MS3 receiver-side IFNAR1-PI3K-AKT-BRCA1 "
            "expression architecture showed directionally consistent "
            "association with the independent MS3-like proliferative "
            "program, including preserved positive direction in all "
            "leave-one-patient-out analyses."
        ),

    "secondary_article_interpretation":
        (
            "The frozen RXRA-NR1H4 expression architecture showed "
            "directional concordance with the independent MS6-like "
            "hepatic/metabolic program."
        ),

    "ms3_pseudobulk": {
        "rho":
            float(pb_ms3_route_rho),

        "p_value":
            float(pb_ms3_route_p),

        "lopo_positive":
            (
                f"{int(ms3_route_lopo['n_positive'])}/"
                f"{int(ms3_route_lopo['n_lopo'])}"
            ),

        "lopo_median_rho":
            float(
                ms3_route_lopo[
                    "median_rho"
                ]
            )
    },

    "ms6_pseudobulk": {
        "rho":
            float(pb_ms6_route_rho),

        "p_value":
            float(pb_ms6_route_p),

        "lopo_positive":
            (
                f"{int(ms6_route_lopo['n_positive'])}/"
                f"{int(ms6_route_lopo['n_lopo'])}"
            ),

        "lopo_median_rho":
            float(
                ms6_route_lopo[
                    "median_rho"
                ]
            )
    },

    "claim_boundary": [
        "No communication-network discovery was rerun.",
        "Expression concordance does not establish ligand-receptor binding.",
        "Expression concordance does not establish biochemical activation.",
        "Expression concordance does not establish causality.",
        "MS5 ITGB1-SRC-FOS was not rediscovered.",
        "MMP9-IFNAR1 remains exploratory.",
        "Step31H is supportive mechanistic consistency, not a primary external-validation endpoint."
    ]
}


with open(
    STEP31H_FINAL_LOCK_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        step31h_final_lock,
        f,
        ensure_ascii=False,
        indent=2
    )


print("=" * 110)

print(
    "STEP31H — FINAL LOCK"
)

print("=" * 110)


print(
    "\nMS3 route pseudobulk rho:",
    pb_ms3_route_rho
)

print(
    "MS3 route LOPO positive:",
    f"{int(ms3_route_lopo['n_positive'])}/"
    f"{int(ms3_route_lopo['n_lopo'])}"
)


print(
    "\nMS6 route pseudobulk rho:",
    pb_ms6_route_rho
)

print(
    "MS6 route LOPO positive:",
    f"{int(ms6_route_lopo['n_positive'])}/"
    f"{int(ms6_route_lopo['n_lopo'])}"
)


print(
    "\nARTICLE POSITION:"
)

print(
"""
Step31H provides supportive mechanistic consistency only.

The frozen MS3 IFNAR1-PI3K-AKT-BRCA1 receiver-side architecture is
directionally concordant with the independent proliferative program,
but this result does not independently establish intercellular
signaling or pathway activation.
"""
)


print(
    "\nSTEP31H: FROZEN"
)

# Step31 FINAL LOCK — GSE326201 Independent scRNA Validation

## Main purpose

Independently validate the malignant-cell biology identified in the
original GSE149614 discovery cohort without redefining malignant states,
reselecting Core50 genes, or rediscovering regulatory/signaling networks.

## Article-centered hierarchy

### Primary external validation
Frozen MS3 Core50 and its proliferative / mitotic /
spindle-chromosome-segregation biology.

### Secondary biological validation
Reciprocal MS3-like proliferative versus MS6-like hepatic/metabolic
organization.

### Mechanistic consistency
Frozen Step29 MS3 regulator architecture and selected Step30
receiver-side signaling architecture.

### Boundaries
The external dataset is used for validation, not rediscovery.

In [ ]:
# =============================================================================
# Step31 Cell 241
# Article-centered final evidence summary
# =============================================================================

import pandas as pd


step31_final_summary = pd.DataFrame([

    {
        "component":
            "Independent malignant universe",

        "main_result":
            "4,688 malignant-supported Hepatic cells from 10 patients",

        "article_role":
            "Foundation",

        "interpretation":
            "Malignancy defined independently of Core50"
    },

    {
        "component":
            "Frozen MS3 Core50",

        "main_result":
            (
                "Pseudobulk: G2M rho=0.842; "
                "Mitotic Entry rho=0.976; "
                "Spindle/Chromosome rho=0.721"
            ),

        "article_role":
            "Primary external validation",

        "interpretation":
            (
                "Reproducible proliferative/mitotic "
                "high-risk biology"
            )
    },

    {
        "component":
            "MS3-like vs MS6-like axis",

        "main_result":
            (
                "Pseudobulk rho=-0.564; "
                "LOPO 10/10 negative"
            ),

        "article_role":
            "Secondary biological validation",

        "interpretation":
            (
                "Reciprocal organization supported at "
                "patient level, weaker at raw single-cell level"
            )
    },

    {
        "component":
            "Frozen Step29 regulators",

        "main_result":
            (
                "MS3 regulator module pseudobulk rho=0.758; "
                "LOPO 10/10 positive"
            ),

        "article_role":
            "Mechanistic consistency",

        "interpretation":
            (
                "MS3 regulatory architecture externally supported; "
                "MS6 regulator architecture weaker"
            )
    },

    {
        "component":
            "Frozen Step30 MS3 route",

        "main_result":
            (
                "IFNAR1-PI3K-AKT-BRCA1 pseudobulk rho=0.430; "
                "LOPO 10/10 positive"
            ),

        "article_role":
            "Supportive mechanistic consistency",

        "interpretation":
            (
                "Directionally concordant expression architecture; "
                "not proof of signaling activation"
            )
    }
])


display(
    step31_final_summary
)


STEP31_FINAL_SUMMARY_PATH = (
    AUDIT_DIR /
    "Step31_FINAL_ArticleCenteredSummary.csv"
)


step31_final_summary.to_csv(
    STEP31_FINAL_SUMMARY_PATH,
    index=False
)

In [ ]:
# =============================================================================
# Step31 Cell 242
# FINAL LOCK
# =============================================================================

import json


STEP31_FINAL_LOCK_PATH = (
    AUDIT_DIR /
    "Step31_FINAL_LOCK.json"
)


step31_final_lock = {

    "step":
        "Step31",

    "dataset":
        "GSE326201",

    "status":
        "FROZEN",

    "independent_validation_universe": {
        "malignant_cells":
            4688,

        "patients":
            10,

        "genes":
            32732
    },

    "primary_external_validation": {
        "target":
            "Step25 frozen MS3 Conserved Core50",

        "conclusion":
            (
                "The frozen MS3 Core50 independently reproduced "
                "proliferative/G2M, mitotic-entry and "
                "spindle-chromosome-segregation biology."
            )
    },

    "secondary_biological_validation": {
        "target":
            "MS3-like versus MS6-like reciprocal axis",

        "conclusion":
            (
                "Reciprocal proliferative versus hepatic/metabolic "
                "organization was supported at patient-pseudobulk "
                "resolution and after technical adjustment, but was "
                "not uniformly reproduced at unadjusted single-cell "
                "resolution."
            )
    },

    "regulatory_validation": {
        "primary":
            (
                "The frozen Step29 MS3 regulator architecture was "
                "supported at patient-pseudobulk level."
            ),

        "secondary":
            (
                "The MS6 regulator architecture showed cell-level "
                "concordance but lacked robust patient-level replication."
            )
    },

    "signaling_validation": {
        "primary":
            (
                "The frozen IFNAR1-PI3K-AKT-BRCA1 receiver-side "
                "expression architecture showed directional consistency "
                "with the independent MS3-like program."
            ),

        "interpretation":
            (
                "Supportive mechanistic consistency only; not evidence "
                "of physical communication or causal pathway activation."
            )
    },

    "firewall": [
        "No Core50 genes were reselected.",
        "Original RiskScore was not reconstructed.",
        "MS1-MS8 states were not redefined.",
        "Malignancy was defined independently of Core50.",
        "Step29 regulators were not reselected.",
        "Step30 communication networks were not rediscovered.",
        "No new external-data-driven malignant state was created."
    ],

    "article_main_message":
        (
            "Independent single-cell validation supports the conserved "
            "MS3 proliferative/mitotic high-risk program as the most "
            "reproducible malignant-cell feature, with secondary support "
            "for a reciprocal differentiated/metabolic axis and associated "
            "regulatory/signaling architectures."
        ),

    "next_step":
        "Step32 — spatial validation"
}


with open(
    STEP31_FINAL_LOCK_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        step31_final_lock,
        f,
        ensure_ascii=False,
        indent=2
    )


print("=" * 120)

print(
    "STEP31 — GSE326201 INDEPENDENT scRNA VALIDATION FINAL LOCK"
)

print("=" * 120)


display(
    step31_final_summary
)


print(
    "\nFINAL ARTICLE-CENTERED CONCLUSION:"
)

print(
"""
Independent GSE326201 validation supports the conserved MS3
proliferative/mitotic high-risk program as the most reproducible
malignant-cell feature.

The reciprocal MS3-like versus MS6-like differentiated/metabolic
organization receives secondary patient-level support.

The frozen MS3 regulatory architecture and selected receiver-side
signaling architecture provide mechanistic consistency but do not
establish causal regulation or intercellular signaling.
"""
)


print(
    "\nFIREWALL:"
)

print(
    "Core50 reselection: False"
)

print(
    "Original RiskScore reconstruction: False"
)

print(
    "MS1-MS8 redefinition: False"
)

print(
    "External GRN rediscovery: False"
)

print(
    "External communication rediscovery: False"
)


print(
    "\nFINAL LOCK:"
)

print(
    STEP31_FINAL_LOCK_PATH
)


print(
    "\nSTEP31: FROZEN"
)

print(
    "NEXT: Step32 — spatial validation"
)

print("=" * 120)